
Purpose
-------
This notebook evaluates large language models on structured information
extraction tasks over medieval Iberian charters.

The evaluation framework supports three independent extraction tasks:
1. Entity extraction
2. Fact extraction
3. Event extraction

For each task, the notebook:
- loads the textual corpus and gold-standard annotations,
- builds task-specific prompts,
- queries one or more LLM backends,
- validates generated JSON outputs against controlled schemas,
- normalizes predictions and gold annotations,
- computes document-level and aggregate evaluation metrics,
- and stores prompts, raw outputs, parsed predictions, error analyses,
  and evaluation tables.

The notebook is designed for controlled experimental comparison across:
- different LLM providers and models,
- prompting strategies (baseline, one-shot, few-shot),
- extraction tasks,
- and entity inclusion settings.

The evaluation pipeline intentionally separates prompting, schema validation,
normalization, alignment, and scoring in order to make experimental behavior
fully inspectable and reproducible.

Important design decisions
--------------------------
- Extraction is performed directly from the original charter text.
- Gold annotations are projected into task-specific representations before
  evaluation.
- Event evaluation distinguishes between several types of event matching
- All normalization is evaluation-oriented only; it does not modify the
  original corpus annotations.
- Outputs are stored document-by-document to facilitate error inspection and
  qualitative analysis.

## Install dependencies

This section installs the external libraries required by the evaluation
pipeline, including:
- LLM provider SDKs,
- schema validation utilities,
- dataframe and progress utilities.

The notebook relies on these dependencies for model interaction, structured
JSON validation, experiment execution, and metric computation.

In [ ]:
!pip -q install -U google-genai openai jsonschema pandas tqdm #mistralai

In [ ]:
!pip install git+https://github.com/mistralai/client-python.git

## Configuration

This section defines the default experimental configuration used throughout the
evaluation pipeline.

It specifies:
- corpus and annotation directories,
- output locations,
- active LLM backend and model,
- evaluation task,
- prompting strategy,
- generation parameters,
- few-shot example selections,
- and entity inclusion settings.

These parameters determine the complete default experimental setup for a run, which may be altered in the corresponding calls.

In [ ]:
import json
import os
import re
import unicodedata
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import time
import random

import pandas as pd
from jsonschema import Draft202012Validator
from tqdm.auto import tqdm

from openai import OpenAI
from google import genai
from mistralai.client import Mistral

from dotenv import load_dotenv

In [ ]:
BASE_DIR = Path("..")

TEXT_DIR = BASE_DIR / "data/corpus"
GOLD_DIR = BASE_DIR / "data/references"
SPLIT_PATH = BASE_DIR / "splits" / "split.json"

OUT_DIR = BASE_DIR / "llm_eval_outputs"
OUT_DIR.mkdir(exist_ok=True)

load_dotenv(BASE_DIR / ".env")

# Backends disponibles:
# - "groq_openai"
# - "gemini_native"
BACKEND_NAME = "groq_openai"

BACKEND_CONFIGS = {
    "groq_openai": {
        "api_env": "GROQ_API_KEY",
        "model": "meta-llama/llama-4-scout-17b-16e-instruct",
        "base_url": "https://api.groq.com/openai/v1",
    },
    "gemini_native": {
        "api_env": "GEMINI_API_KEY",
        "model": "gemini-3.1-flash-lite-preview",
    },
    "mistral_openai": {
        "api_env": "MISTRAL_API_KEY",
        "model": "mistral-large-2411",
    },
    "openai": {
        "api_env": "OPENAI_API_KEY",
        "model": "gpt-5.4-nano",
    }
}

GROQ_MODELS = {
    "llama4_scout_17b": "meta-llama/llama-4-scout-17b-16e-instruct",
    "groq_compound": "groq/compound-mini",
    "llama33_70b": "llama-3.3-70b-versatile",
    "llama31_8b": "llama-3.1-8b-instant",
}

DEFAULT_BACKEND = BACKEND_CONFIGS[BACKEND_NAME]
MODEL_NAME = DEFAULT_BACKEND["model"]

# Tareas disponibles:
# - "entityonly"
# - "factsonly"
# - "eventsonly"
TASK_NAME = "eventsonly"

# Variantes de prompt:
# - "baseline"
# - "oneshot"
# - "fewshot"
PROMPT_VARIANT = "fewshot"

FIXED_ONESHOT_EXAMPLE_IDS = [
    "D30",
]

FIXED_FEWSHOT_EXAMPLE_IDS = [
    "D30",
    "D44",
]

DEFAULT_MAX_OUTPUT_TOKENS = 4096
DEFAULT_TEMPERATURE = 0.0

# Gemini native: "minimal" o "low" en flash-lite
DEFAULT_THINKING_LEVEL = "minimal"

# Few-shot
DEFAULT_MAX_CHARS_PER_EXAMPLE = 15000

# Entity extraction modes:
# - "annotated": solo entidades del gold anotado manualmente; excluye provenance="ner"
# - "all": todas las entidades del gold, incluyendo provenance="ner"
ENTITY_MODE = "annotated"

print("BASE_DIR:", BASE_DIR)
print("TEXT_DIR:", TEXT_DIR)
print("GOLD_DIR:", GOLD_DIR)
print("SPLIT_PATH:", SPLIT_PATH)
print("OUT_DIR:", OUT_DIR)
print("BACKEND_NAME:", BACKEND_NAME)
print("MODEL_NAME:", MODEL_NAME)
print("TASK_NAME:", TASK_NAME)
print("PROMPT_VARIANT:", PROMPT_VARIANT)


## I/O Helpers

This section defines utility functions for loading and saving JSON files,
plain-text documents, and experiment outputs.

The helpers standardize file handling across the pipeline and ensure that all
generated artifacts are serialized in a JSON-safe format.

In [ ]:
def load_json(path: Path) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_text(path: Path) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

def make_json_safe(x: Any) -> Any:
    if x is None or isinstance(x, (str, int, float, bool)):
        return x
    if isinstance(x, dict):
        return {str(k): make_json_safe(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [make_json_safe(v) for v in x]
    return str(x)

def save_json(path: Path, data: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(make_json_safe(data), f, ensure_ascii=False, indent=2)

def save_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)

def ensure_list(x: Any) -> List[Any]:
    if x is None:
        return []
    if isinstance(x, list):
        return x
    return [x]


def load_split() -> Dict[str, List[str]]:
    return load_json(SPLIT_PATH)

## Normalization

This section defines lightweight normalization procedures used exclusively
during evaluation and matching.

The normalization routines reduce superficial textual variation by:
- lowercasing,
- standardizing punctuation,
- removing accent variation,
- normalizing whitespace,
- and canonicalizing nested literal structures.

These transformations are applied only for comparison and scoring purposes;
they do not modify the original corpus annotations or model outputs.

In [ ]:
def strip_accents(text: Optional[str]) -> Optional[str]:
    if text is None:
        return None
    return "".join(
        c for c in unicodedata.normalize("NFD", text)
        if unicodedata.category(c) != "Mn"
    )

def normalize_punctuation(text: str) -> str:
    return (
        text.replace("’", "'")
            .replace("‘", "'")
            .replace("´", "'")
            .replace("`", "'")
            .replace("“", '"')
            .replace("”", '"')
            .replace("–", "-")
            .replace("—", "-")
    )

def norm_text(text: Any) -> Optional[str]:
    if text is None:
        return None
    text = str(text).strip().lower()
    text = normalize_punctuation(text)
    text = re.sub(r"\s+", " ", text)
    return text or None


def norm_text_loose(text: Any) -> Optional[str]:
    value = norm_text(text)
    return strip_accents(value) if value is not None else None


def norm_literal(value: Any) -> Any:
    if isinstance(value, str):
        return norm_text_loose(value)
    if isinstance(value, list):
        return tuple(norm_literal(v) for v in value)
    if isinstance(value, dict):
        return tuple(sorted((k, norm_literal(v)) for k, v in value.items()))
    return value


def dedupe_keep_order(values: Iterable[Any]) -> List[Any]:
    seen = set()
    out = []
    for v in values:
        key = json.dumps(v, ensure_ascii=False, sort_keys=True) if isinstance(v, (dict, list)) else str(v)
        if key in seen:
            continue
        seen.add(key)
        out.append(v)
    return out

## Ontology

This section defines the controlled ontology used by the evaluation framework.

It specifies:
- allowed entity types,
- fact predicates,
- event types,
- event subtypes,
- event argument roles,
- and task-specific gold projections.

The ontology constrains both prompt generation and schema validation, ensuring
that model outputs follow a consistent structured representation.

In [ ]:
ENTITY_TYPES = ["Person", "Place"]

FACT_ATTRIBUTE_PREDICATES = [
    "given_name",
    "family_name",
    "nickname",
    "occupation",
    "position_held",
]

FACT_RELATION_PREDICATES = [
    "occupation_location",
    "position_jurisdiction",
    "residence",
    "employer",
    "has_spouse",
    "has_father",
    "has_mother",
    "has_grandfather",
    "has_grandmother",
    "has_grandchild",
    "has_sibling_in_law",
    "has_father_in_law",
    "has_son_in_law",
    "has_child",
    "has_sibling",
    "has_cousin",
    "has_uncle",
    "has_nephew",
]

ALL_FACT_PREDICATES = FACT_ATTRIBUTE_PREDICATES + FACT_RELATION_PREDICATES

EVENT_TYPES = [
    "Purchase",
    "Acquisition",
    "DocumentProduction",
    "GenericActivity",
]

EVENT_SUBTYPES = [
    "Sale",
    "Disembargo",
    "Retrosale",
    "Lease",
    "Donation",
    "Compensation",
    "Agreement",
    "Testimony",
    "Reproduction",
]

EVENT_ROLES = [
    "transfer_from",
    "transfer_to",
    "witness",
    "notary",
    "judge",
    "participant",
]


def entity_name_index(gold_doc: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:
    return {
        e["id"]: {
            "name": e.get("canonical_name", e["id"]),
            "type": e.get("type"),
        }
        for e in gold_doc.get("entities", [])
        if e.get("id")
    }


def project_gold_entityonly(
    gold_doc: Dict[str, Any],
    entity_mode: str = ENTITY_MODE,
) -> Dict[str, Any]:
    entities = gold_doc.get("entities", [])

    if entity_mode == "annotated":
        entities = [
            e for e in entities
            if e.get("provenance") != "ner"
        ]
    else:
        raise ValueError(f"entity_mode no soportado: {entity_mode}")

    return {
        "document_id": gold_doc.get("document_id"),
        "entities": [
            {
                "id": e.get("id"),
                "type": e.get("type"),
                "canonical_name": e.get("canonical_name"),
                "aliases": e.get("aliases", []),
            }
            for e in entities
        ],
    }


def project_gold_factsonly(gold_doc: Dict[str, Any]) -> Dict[str, Any]:
    ent_index = entity_name_index(gold_doc)
    out_facts = []

    for fact in gold_doc.get("facts", []):
        subj_id = fact.get("subject")
        if subj_id not in ent_index:
            continue

        subject_info = ent_index[subj_id]
        object_type = fact.get("object_type")
        obj = fact.get("object")

        if object_type == "entity":
            if obj not in ent_index:
                continue
            object_value = ent_index[obj]["name"]
        else:
            object_value = obj

        out_facts.append({
            "id": fact.get("id"),
            "subject_type": subject_info["type"],
            "subject": subject_info["name"],
            "predicate": fact.get("predicate"),
            "fact_type": fact.get("fact_type"),
            "object_type": object_type,
            "object": object_value,
        })

    return {
        "document_id": gold_doc.get("document_id"),
        "facts": out_facts,
    }


def project_gold_eventsonly(gold_doc: Dict[str, Any]) -> Dict[str, Any]:
    ent_index = entity_name_index(gold_doc)
    out_events = []

    for ev in gold_doc.get("events", []):
        projected_args = []
        for arg in ev.get("arguments", []):
            ent_id = arg.get("entity")
            if ent_id in ent_index:
                projected_args.append({
                    "role": arg.get("role"),
                    "value": ent_index[ent_id]["name"],
                })

        out_event = {
            "id": ev.get("id"),
            "type": ev.get("type"),
            "attributes": ev.get("attributes", {}),
            "arguments": projected_args,
        }
        if ev.get("subtype") is not None:
            out_event["subtype"] = ev["subtype"]
        out_events.append(out_event)

    return {
        "document_id": gold_doc.get("document_id"),
        "events": out_events,
    }

def project_gold(
    task_name: str,
    gold_doc: Dict[str, Any],
    entity_mode: str = ENTITY_MODE,
) -> Dict[str, Any]:
    if task_name == "entityonly":
        return project_gold_entityonly(gold_doc, entity_mode=entity_mode)
    if task_name == "factsonly":
        return project_gold_factsonly(gold_doc)
    if task_name == "eventsonly":
        return project_gold_eventsonly(gold_doc)
    raise ValueError(f"Tarea no soportada: {task_name}")

def extract_gold_event_labels(gold_doc: Dict[str, Any]) -> Dict[str, Any]:
    """
    Extrae metadatos de tipos/subtipos de evento del gold para poder agrupar resultados.
    Un documento puede tener hasta dos tipos/subtipos.
    """

    events = gold_doc.get("events", [])

    types = sorted({
        ev.get("type") for ev in events
        if ev.get("type") is not None
    })

    subtypes = sorted({
        ev.get("subtype") for ev in events
        if ev.get("subtype") is not None
    })

    type_combo = "/".join(types) if types else ""
    subtype_combo = "/".join(subtypes) if subtypes else ""

    return {
        "n_gold_events": len(events),

        "gold_event_types": types,
        "gold_event_subtypes": subtypes,

        "gold_event_type": types[0] if len(types) == 1 else None,
        "gold_event_subtype": subtypes[0] if len(subtypes) == 1 else None,

        "gold_event_type_combo": type_combo,
        "gold_event_subtype_combo": subtype_combo,

        "gold_event_types_str": " | ".join(types) if types else "",
        "gold_event_subtypes_str": " | ".join(subtypes) if subtypes else "",
    }

## JSON schemas per task

This section defines the JSON schemas used to validate model outputs for each
extraction task.

Separate schemas are provided for:
- entity extraction,
- fact extraction,
- and event extraction.

The schemas enforce structural consistency, restrict allowed ontology values,
and support automatic validation of generated outputs before evaluation.

In [ ]:

ENTITYONLY_SCHEMA: Dict[str, Any] = {
    "type": "object",
    "additionalProperties": False,
    "required": ["document_id", "entities"],
    "properties": {
        "document_id": {"type": "string"},
        "entities": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "required": ["id", "type", "canonical_name", "aliases"],
                "properties": {
                    "id": {"type": "string"},
                    "type": {"type": "string", "enum": ENTITY_TYPES},
                    "canonical_name": {"type": "string"},
                    "aliases": {"type": "array", "items": {"type": "string"}},
                },
            },
        },
    },
}

FACTSONLY_SCHEMA: Dict[str, Any] = {
    "type": "object",
    "additionalProperties": False,
    "required": ["document_id", "facts"],
    "properties": {
        "document_id": {"type": "string"},
        "facts": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "required": [
                    "id", "subject_type", "subject", "predicate",
                    "fact_type", "object_type", "object"
                ],
                "properties": {
                    "id": {"type": "string"},
                    "subject_type": {"type": "string", "enum": ENTITY_TYPES},
                    "subject": {"type": "string"},
                    "predicate": {"type": "string", "enum": ALL_FACT_PREDICATES},
                    "fact_type": {"type": "string", "enum": ["attribute", "relation"]},
                    "object_type": {"type": "string", "enum": ["literal", "entity"]},
                    "object": {
                        "oneOf": [
                            {"type": "string"},
                            {"type": "number"},
                            {"type": "integer"},
                            {"type": "object"},
                            {"type": "array"},
                        ]
                    },
                },
            },
        },
    },
}

EVENTSONLY_SCHEMA: Dict[str, Any] = {
    "type": "object",
    "additionalProperties": False,
    "required": ["document_id", "events"],
    "properties": {
        "document_id": {"type": "string"},
        "events": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "required": ["id", "type", "attributes", "arguments"],
                "properties": {
                    "id": {"type": "string"},
                    "type": {"type": "string", "enum": EVENT_TYPES},
                    "subtype": {"type": ["string", "null"], "enum": EVENT_SUBTYPES + [None]},
                    "attributes": {
                        "type": "object",
                        "additionalProperties": False,
                        "properties": {
                            "time": {"type": "array", "items": {"type": "string"}},
                            "price": {
                                "type": "array",
                                "items": {
                                    "type": "object",
                                    "additionalProperties": False,
                                    "required": ["value", "currency"],
                                    "properties": {
                                        "value": {"type": ["number", "integer", "string"]},
                                        "currency": {"type": "string"},
                                    },
                                },
                            },
                        },
                    },
                    "arguments": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "additionalProperties": False,
                            "required": ["role", "value"],
                            "properties": {
                                "role": {"type": "string", "enum": EVENT_ROLES},
                                "value": {"type": "string"},
                            },
                        },
                    },
                },
            },
        },
    },
}


def task_schema(task_name: str) -> Dict[str, Any]:
    if task_name == "entityonly":
        return ENTITYONLY_SCHEMA
    if task_name == "factsonly":
        return FACTSONLY_SCHEMA
    if task_name == "eventsonly":
        return EVENTSONLY_SCHEMA
    raise ValueError(f"Tarea no soportada: {task_name}")

## Parsing and validation

This section parses raw model outputs and validates them against the task
schemas.

The pipeline:
- removes formatting artifacts,
- extracts JSON content,
- parses generated structures,
- validates schema compliance,
- and records parsing or validation errors.

Validation results are stored alongside raw outputs to facilitate debugging
and reproducibility.

In [ ]:
def clean_model_json(text: str) -> str:
    text = text.strip()
    text = re.sub(r"^```json\s*", "", text)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    first = text.find("{")
    last = text.rfind("}")
    if first != -1 and last != -1 and last > first:
        text = text[first:last + 1]
    return text


def parse_and_validate(raw_text: str, schema: Dict[str, Any]) -> Dict[str, Any]:
    cleaned = clean_model_json(raw_text)

    try:
        parsed = json.loads(cleaned)
    except Exception as e:
        return {
            "parsed": None,
            "cleaned_text": cleaned,
            "parse_error": str(e),
            "schema_valid": False,
            "schema_errors": [],
        }

    validator = Draft202012Validator(schema)
    errors = sorted(validator.iter_errors(parsed), key=lambda e: list(e.path))
    schema_errors = [
        {
            "path": "/" + "/".join(str(x) for x in err.path),
            "message": err.message,
        }
        for err in errors
    ]

    return {
        "parsed": parsed,
        "cleaned_text": cleaned,
        "parse_error": None,
        "schema_valid": len(schema_errors) == 0,
        "schema_errors": schema_errors,
    }

## Backends

This section defines the interfaces and implementations used to interact with
different LLM providers.

The notebook supports multiple backends through a shared abstraction layer,
including:
- OpenAI-compatible APIs,
- Gemini native APIs,
- Groq-hosted models,
- and Mistral models.

It also implements retry logic for transient backend failures and standardizes
generation metadata across providers.

In [ ]:
class BaseBackend:
    name: str

    def generate_json(
        self,
        prompt: str,
        schema: Dict[str, Any],
        *,
        temperature: float,
        max_output_tokens: int,
        thinking_level: str,
        system_instruction: Optional[str] = None,
    ) -> Dict[str, Any]:
        raise NotImplementedError


class GroqOpenAIBackend(BaseBackend):
    name = "groq_openai"

    def __init__(self, api_key: str, model: str, base_url: str):
        if not api_key:
            raise ValueError("Falta GROQ_API_KEY")
        self.model = model
        self.client = OpenAI(api_key=api_key, base_url=base_url)

    def generate_json(
        self,
        prompt: str,
        schema: Dict[str, Any],
        *,
        temperature: float,
        max_output_tokens: int,
        thinking_level: str,
        system_instruction: Optional[str] = None,
    ) -> Dict[str, Any]:
        messages = []
        if system_instruction:
            messages.append({"role": "system", "content": system_instruction})
        messages.append({"role": "user", "content": prompt})

        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=temperature,
            max_tokens=max_output_tokens,
        )
        raw_text = response.choices[0].message.content or ""

        usage = getattr(response, "usage", None)

        return {
            "raw_text": raw_text,
            "provider_meta": {
                "backend": self.name,
                "model": self.model,
                "finish_reason": getattr(response.choices[0], "finish_reason", None),
                "usage": str(usage) if usage is not None else None,
            },
        }


class GeminiNativeBackend(BaseBackend):
    name = "gemini_native"

    def __init__(self, api_key: str, model: str):
        if not api_key:
            raise ValueError("Falta GEMINI_API_KEY")
        self.model = model
        self.client = genai.Client(api_key=api_key)

    def generate_json(
        self,
        prompt: str,
        schema: Dict[str, Any],
        *,
        temperature: float,
        max_output_tokens: int,
        thinking_level: str,
        system_instruction: Optional[str] = None,
    ) -> Dict[str, Any]:
        config = {
            "temperature": temperature,
            "max_output_tokens": max_output_tokens,
            "response_mime_type": "application/json",
            "response_json_schema": schema,
            "thinking_config": {
                "thinking_level": thinking_level,
            },
        }
        if system_instruction:
            config["system_instruction"] = system_instruction

        response = self.client.models.generate_content(
            model=self.model,
            contents=prompt,
            config=config,
        )

        raw_text = response.text or ""

        usage_metadata = getattr(response, "usage_metadata", None)

        return {
            "raw_text": raw_text,
            "provider_meta": {
                "backend": self.name,
                "model": self.model,
                "usage_metadata": str(usage_metadata) if usage_metadata is not None else None,
            },
        }

class MistralOpenAIBackend(BaseBackend):
    name = "mistral_openai"

    def __init__(self, api_key: str, model: str):
        if not api_key:
            raise ValueError("Falta MISTRAL_API_KEY")

        self.model = model
        self.client = OpenAI(
            api_key=api_key,
            base_url="https://api.mistral.ai/v1",
            timeout=600.0,  # ← aquí sí funciona
        )

    def generate_json(
        self,
        prompt: str,
        schema: Dict[str, Any],
        *,
        temperature: float,
        max_output_tokens: int,
        thinking_level: str,
        system_instruction: Optional[str] = None,
    ) -> Dict[str, Any]:

        messages = []
        if system_instruction:
            messages.append({"role": "system", "content": system_instruction})
        messages.append({"role": "user", "content": prompt})

        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=temperature,
            max_tokens=max_output_tokens,
            response_format={"type": "json_object"},
        )

        raw_text = response.choices[0].message.content

        return {
            "raw_text": raw_text,
            "provider_meta": {
                "backend": self.name,
                "model": self.model,
                "finish_reason": response.choices[0].finish_reason,
                "usage": str(response.usage) if response.usage else None,
            },
        }

class OpenAIBackend(BaseBackend):
    name = "openai"

    def __init__(self, api_key: str, model: str):
        if not api_key:
            raise ValueError("Falta OPENAI_API_KEY")

        self.model = model
        self.client = OpenAI(
            api_key=api_key,
            timeout=600.0,
        )

    def generate_json(
        self,
        prompt: str,
        schema: Dict[str, Any],
        *,
        temperature: float,
        max_output_tokens: int,
        thinking_level: str,
        system_instruction: Optional[str] = None,
    ) -> Dict[str, Any]:

        messages = []
        if system_instruction:
            messages.append({"role": "system", "content": system_instruction})
        messages.append({"role": "user", "content": prompt})

        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=temperature,
            max_completion_tokens=max_output_tokens,
            response_format={"type": "json_object"},
        )

        raw_text = response.choices[0].message.content

        return {
            "raw_text": raw_text,
            "provider_meta": {
                "backend": self.name,
                "model": self.model,
                "finish_reason": response.choices[0].finish_reason,
                "usage": str(response.usage) if response.usage else None,
            },
        }


def make_backend(
    backend_name: str = BACKEND_NAME,
    model_name: Optional[str] = None,
) -> BaseBackend:
    cfg = BACKEND_CONFIGS[backend_name]
    model_name = model_name or cfg["model"]
    api_key = os.environ.get(cfg["api_env"], "")
    if backend_name == "groq_openai":
        return GroqOpenAIBackend(
            api_key=api_key,
            model=model_name,
            base_url=cfg["base_url"],
        )
    if backend_name == "gemini_native":
        return GeminiNativeBackend(
            api_key=api_key,
            model=model_name,
        )
    if backend_name == "mistral_openai":
        return MistralOpenAIBackend(
            api_key=api_key,
            model=model_name,
        )
    if backend_name == "openai":
        return OpenAIBackend(
            api_key=api_key,
            model=model_name,
        )
    raise ValueError(f"Backend no soportado: {backend_name}")

def is_retryable_backend_error(exc: Exception) -> bool:
    msg = str(exc).lower()

    retry_markers = [
        "503",
        "unavailable",
        "high demand",
        "rate limit",
        "429",
        "timeout",
        "timed out",
        "read operation timed out",
        "temporarily unavailable",
        "resource exhausted",
        "internal error",
        "500",
        "502",
        "504",
    ]

    return any(marker in msg for marker in retry_markers)


def generate_with_retries(
    backend: BaseBackend,
    *,
    prompt: str,
    schema: Dict[str, Any],
    temperature: float,
    max_output_tokens: int,
    thinking_level: str,
    system_instruction: Optional[str] = None,
    max_attempts: int = 6,
    initial_wait: float = 3.0,
    max_wait: float = 60.0,
) -> Dict[str, Any]:
    last_exc = None

    for attempt in range(1, max_attempts + 1):
        try:
            return backend.generate_json(
                prompt=prompt,
                schema=schema,
                temperature=temperature,
                max_output_tokens=max_output_tokens,
                thinking_level=thinking_level,
                system_instruction=system_instruction,
            )
        except Exception as exc:
            last_exc = exc

            if not is_retryable_backend_error(exc):
                raise

            if attempt == max_attempts:
                raise

            wait = min(max_wait, initial_wait * (2 ** (attempt - 1)))
            # jitter pequeño para no reintentar siempre a la vez
            wait = wait * (0.8 + 0.4 * random.random())

            print(
                f"[retry {attempt}/{max_attempts}] "
                f"backend transient error: {exc}. "
                f"Waiting {wait:.1f}s before retrying..."
            )
            time.sleep(wait)

    raise last_exc

## Prompting per task

This section constructs task-specific prompts for entity, fact, and event
extraction.

The prompting layer:
- injects ontology constraints,
- defines JSON output formats,
- provides domain-specific extraction rules,
- optionally includes one-shot or few-shot examples,
- and builds the final instruction shown to the model.

Prompt generation is fully task-aware and configurable by prompting strategy.

In [ ]:
COMMON_DOMAIN_RULES = """
Domain-specific rules:
- If the document uses the Hispanic Era dating system (often marked by "era"), convert it to Common Era by subtracting 38 years.
- Do NOT merge different places into one entity.
""".strip()


def build_fewshot_block(
    task_name: str,
    example_doc_ids: List[str],
    max_chars_per_text: Optional[int] = None,
    entity_mode: str = ENTITY_MODE,
) -> str:
    blocks = []

    for ex_id in example_doc_ids:
        ex_text = load_text(TEXT_DIR / f"{ex_id}.txt")
        ex_gold = load_json(GOLD_DIR / f"{ex_id}.json")

        if max_chars_per_text is not None:
            ex_text = ex_text[:max_chars_per_text]

        projected = project_gold(task_name, ex_gold, entity_mode=entity_mode)

        if task_name == "entityonly":
            projected["entities"] = [
                {
                    **e,
                    "aliases": ensure_list(e.get("aliases"))[:1]
                }
                for e in projected.get("entities", [])
            ]

        block = f'''Example document ID: {ex_id}

Example document:
"""
{ex_text}
"""

Example output:
{json.dumps(projected, ensure_ascii=False, indent=2)}'''
        blocks.append(block)

    return "\n\n".join(blocks)


def build_entityonly_prompt(
    document_text: str,
    doc_id: str,
    variant: str = "baseline",
    example_doc_ids: Optional[List[str]] = None,
    max_chars_per_example: Optional[int] = DEFAULT_MAX_CHARS_PER_EXAMPLE,
    entity_mode: str = ENTITY_MODE,
) -> str:
    schema_block = """
      Return exactly one JSON object with this structure:

      {
        "document_id": "string",
        "entities": [
          {
            "id": "ent_person_0001 or ent_place_0001",
            "type": "Person or Place",
            "canonical_name": "string",
            "aliases": ["string"]
          }
        ]
      }
      """.strip()

    ontology_block = """
      Allowed entity types:
      - Person
      - Place
      """.strip()
    if entity_mode == "annotated":
      rules_block = f"""
      Rules:
      - Extract ONLY entities explicitly stated in the document.
      - Do NOT extract place names that appear only as part of a personal name or surname unless they are mentioned independently as places in the document.
      - Do NOT extract facts.
      - Do NOT extract events.
      - Use exactly this document_id: "{doc_id}".
      - canonical_name should be a readable name from the document.
      - aliases should contain only explicit surface variants from the document.
      """.strip()

    elif entity_mode == "all":
        rules_block = f"""
      Rules:
      - Extract all explicit Person and Place entities mentioned in the document.
      - Do NOT extract facts.
      - Do NOT extract events.
      - Use exactly this document_id: "{doc_id}".
      - canonical_name should be a readable name from the document.
      - aliases should contain only explicit surface variants from the document.
      """.strip()

    else:
        raise ValueError(f"entity_mode no soportado: {entity_mode}")
    extra = COMMON_DOMAIN_RULES
    if variant == "oneshot":
        example_doc_ids = example_doc_ids or FIXED_ONESHOT_EXAMPLE_IDS
        extra += "\nExamples:\n" + build_fewshot_block(
            "entityonly",
            example_doc_ids,
            max_chars_per_text=max_chars_per_example,
            entity_mode=entity_mode,
        )

    elif variant == "fewshot":
        example_doc_ids = example_doc_ids or FIXED_FEWSHOT_EXAMPLE_IDS
        extra += "\nExamples:\n" + build_fewshot_block(
            "entityonly",
            example_doc_ids,
            max_chars_per_text=max_chars_per_example,
            entity_mode=entity_mode,
        )

    return f"""
      You are an expert in medieval Iberian charters.

      Your task is to extract only the entities explicitly stated in the document.

      {schema_block}

      {ontology_block}

      {rules_block}

      {extra}

      Document:
      {document_text}
      """.strip()


def build_factsonly_prompt(
    document_text: str,
    doc_id: str,
    variant: str = "baseline",
    example_doc_ids: Optional[List[str]] = None,
    max_chars_per_example: Optional[int] = DEFAULT_MAX_CHARS_PER_EXAMPLE,
) -> str:
    schema_block = """
Return exactly one JSON object with this structure:

{
  "document_id": "string",
  "facts": [
    {
      "id": "fact_00001",
      "subject_type": "Person or Place",
      "subject": "string",
      "predicate": "allowed predicate",
      "fact_type": "attribute or relation",
      "object_type": "literal or entity",
      "object": "string"
    }
  ]
}
""".strip()

    ontology_block = f"""
Allowed subject types:
- Person
- Place

Allowed fact predicates with fact_type="attribute" and object_type="literal":
{chr(10).join("- " + x for x in FACT_ATTRIBUTE_PREDICATES)}

Allowed fact predicates with fact_type="relation" and object_type="entity":
{chr(10).join("- " + x for x in FACT_RELATION_PREDICATES)}
""".strip()

    rules_block = f"""
Rules:
- Extract ONLY facts explicitly stated in the document.
- Do NOT extract standalone entities.
- Do NOT extract events.
- Use exactly this document_id: "{doc_id}".
- subject must be a person or place name as it appears in the document.
- For relation facts, object must be the target person or place name as a string, not an id.
- For family relations, always extract both directions when an inverse predicate exists (e.g., has_father ↔ has_child, has_spouse ↔ has_spouse).
- For attribute facts, object must be the literal value.
- For every Person, always attempt to extract given_name and family_name, if available.
- Preserve original spelling exactly; do not normalize or correct occupation strings.
""".strip()

    extra = COMMON_DOMAIN_RULES
    if variant == "oneshot":
        example_doc_ids = example_doc_ids or FIXED_ONESHOT_EXAMPLE_IDS
        extra = "Examples:\n" + build_fewshot_block(
            "factsonly",
            example_doc_ids,
            max_chars_per_text=max_chars_per_example,
        )

    elif variant == "fewshot":
        example_doc_ids = example_doc_ids or FIXED_FEWSHOT_EXAMPLE_IDS
        extra = "Examples:\n" + build_fewshot_block(
            "factsonly",
            example_doc_ids,
            max_chars_per_text=max_chars_per_example,
        )

    return f"""
You are an expert in medieval Iberian charters.

Your task is to extract only the facts explicitly stated in the document.

{schema_block}

{ontology_block}

{rules_block}

{extra}

Document:
{document_text}
""".strip()


def build_eventsonly_prompt(
    document_text: str,
    doc_id: str,
    variant: str = "baseline",
    example_doc_ids: Optional[List[str]] = None,
    max_chars_per_example: Optional[int] = DEFAULT_MAX_CHARS_PER_EXAMPLE,
) -> str:
    schema_block = """
Return exactly one JSON object with this structure:

{
  "document_id": "string",
  "events": [
    {
      "id": "evt_0001",
      "type": "allowed type",
      "subtype": "allowed subtype or null",
      "attributes": {
        "time": ["YYYY-MM-DD"],
        "price": [
          {
            "value": number,
            "currency": "string"
          }
        ]
      },
      "arguments": [
        {
          "role": "allowed event role",
          "value": "person or place name as it appears in the document"
        }
      ]
    }
  ]
}
""".strip()

    ontology_block = f"""
Allowed event types:
{chr(10).join("- " + x for x in EVENT_TYPES)}

Allowed event subtypes:
{chr(10).join("- " + x for x in EVENT_SUBTYPES)}

Allowed event argument roles:
{chr(10).join("- " + x for x in EVENT_ROLES)}

Allowed event attributes:
- time
- price
""".strip()

    rules_block = f"""
Rules:
- Extract ONLY event information explicitly stated in the document.
- Do NOT extract entities outside event participation.
- Do NOT extract facts.
- Use exactly this document_id: "{doc_id}".
- In arguments, use names and place names as strings, not ids.
- If subtype is unclear, set subtype to null.
- Do NOT extract a DocumentProduction event for the current document itself; only extract it if the text explicitly describes the production of another document as an event.
- Normalize time to YYYY-MM-DD when possible.
""".strip()

    extra = COMMON_DOMAIN_RULES
    if variant == "oneshot":
        example_doc_ids = example_doc_ids or FIXED_ONESHOT_EXAMPLE_IDS
        extra += "\nExamples:\n" + build_fewshot_block(
            "eventsonly",
            example_doc_ids,
            max_chars_per_text=max_chars_per_example,
        )

    elif variant == "fewshot":
        example_doc_ids = example_doc_ids or FIXED_FEWSHOT_EXAMPLE_IDS
        extra += "\nExamples:\n" + build_fewshot_block(
            "eventsonly",
            example_doc_ids,
            max_chars_per_text=max_chars_per_example,
        )

    return f"""
You are an expert in medieval Iberian charters.

Your task is to extract only the events explicitly stated in the document.

{schema_block}

{ontology_block}

{rules_block}

{extra}

Document:
{document_text}
""".strip()


def build_prompt(
    task_name: str,
    document_text: str,
    doc_id: str,
    variant: str = "baseline",
    example_doc_ids: Optional[List[str]] = None,
    max_chars_per_example: Optional[int] = DEFAULT_MAX_CHARS_PER_EXAMPLE,
    entity_mode: str = ENTITY_MODE,
) -> str:
    if task_name == "entityonly":
        return build_entityonly_prompt(
            document_text=document_text,
            doc_id=doc_id,
            variant=variant,
            example_doc_ids=example_doc_ids,
            max_chars_per_example=max_chars_per_example,
            entity_mode=entity_mode,
        )
    if task_name == "factsonly":
        return build_factsonly_prompt(
            document_text=document_text,
            doc_id=doc_id,
            variant=variant,
            example_doc_ids=example_doc_ids,
            max_chars_per_example=max_chars_per_example,
        )
    if task_name == "eventsonly":
        return build_eventsonly_prompt(
            document_text=document_text,
            doc_id=doc_id,
            variant=variant,
            example_doc_ids=example_doc_ids,
            max_chars_per_example=max_chars_per_example,
        )
    raise ValueError(f"Tarea no soportada: {task_name}")

## Evaluation by task

This section implements task-specific evaluation procedures for entities,
facts, and events.

The evaluation framework includes:
- entity alignment through surface-form matching,
- fact normalization and exact comparison,
- event normalization and structured matching,
- strict and relaxed event metrics,
- role-level and attribute-level scoring,
- and per-category precision, recall, and F1 computation.

The section also includes utilities for collecting structured error analyses.

In [ ]:
@dataclass
class EntityAlignment:
    pred_to_gold: Dict[str, str]
    unmatched_pred_ids: List[str]
    unmatched_gold_ids: List[str]
    ambiguous_pred_ids: List[str]


def entity_surface_forms(entity: Dict[str, Any]) -> List[str]:
    values = []
    canonical = entity.get("canonical_name")
    if canonical:
        values.append(canonical)
    values.extend(ensure_list(entity.get("aliases")))
    cleaned = [norm_text_loose(v) for v in values if norm_text_loose(v)]
    return dedupe_keep_order(cleaned)


def build_gold_entity_label_index(gold_entities: List[Dict[str, Any]]) -> Dict[Tuple[str, str], List[str]]:
    index: Dict[Tuple[str, str], List[str]] = defaultdict(list)
    for ent in gold_entities:
        ent_type = ent.get("type")
        for label in entity_surface_forms(ent):
            index[(ent_type, label)].append(ent["id"])
    return index


def match_pred_entities_to_gold(pred_entities: List[Dict[str, Any]], gold_entities: List[Dict[str, Any]]) -> EntityAlignment:
    gold_index = build_gold_entity_label_index(gold_entities)
    pred_to_gold: Dict[str, str] = {}
    ambiguous: List[str] = []
    used_gold: set[str] = set()

    for pred in pred_entities:
        pred_id = pred.get("id")
        pred_type = pred.get("type")
        if not pred_id or not pred_type:
            continue

        candidate_ids: List[str] = []
        for label in entity_surface_forms(pred):
            candidate_ids.extend(gold_index.get((pred_type, label), []))

        candidate_ids = dedupe_keep_order(candidate_ids)
        candidate_ids = [gid for gid in candidate_ids if gid not in used_gold]

        if len(candidate_ids) == 1:
            pred_to_gold[pred_id] = candidate_ids[0]
            used_gold.add(candidate_ids[0])
        elif len(candidate_ids) > 1:
            ambiguous.append(pred_id)

    pred_ids = {e["id"] for e in pred_entities if e.get("id")}
    gold_ids = {e["id"] for e in gold_entities if e.get("id")}

    return EntityAlignment(
        pred_to_gold=pred_to_gold,
        unmatched_pred_ids=sorted(pred_ids - set(pred_to_gold.keys())),
        unmatched_gold_ids=sorted(gold_ids - set(pred_to_gold.values())),
        ambiguous_pred_ids=sorted(ambiguous),
    )


def prf(tp: int, fp: int, fn: int) -> Tuple[float, float, float]:
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f1


def metric_prefix(counter_tp: Counter, counter_fp: Counter, counter_fn: Counter, prefix: str) -> Dict[str, Any]:
    out: Dict[str, Any] = {}
    all_keys = set(counter_tp) | set(counter_fp) | set(counter_fn)

    def sort_key(x: Any) -> str:
        if x is None:
            return "__NONE__"
        return str(x)

    for key in sorted(all_keys, key=sort_key):
        tp = int(counter_tp.get(key, 0))
        fp = int(counter_fp.get(key, 0))
        fn = int(counter_fn.get(key, 0))
        p, r, f1 = prf(tp, fp, fn)

        safe = re.sub(r"[^a-zA-Z0-9_]+", "_", sort_key(key)).strip("_")
        if not safe:
            safe = "UNKNOWN"

        out[f"{prefix}_{safe}_tp"] = tp
        out[f"{prefix}_{safe}_fp"] = fp
        out[f"{prefix}_{safe}_fn"] = fn
        out[f"{prefix}_{safe}_precision"] = p
        out[f"{prefix}_{safe}_recall"] = r
        out[f"{prefix}_{safe}_f1"] = f1

    return out


def evaluate_entityonly(
    pred_doc: Dict[str, Any],
    gold_doc: Dict[str, Any],
    entity_mode: str = ENTITY_MODE,
) -> Dict[str, Any]:
    pred_entities = pred_doc.get("entities", [])
    gold_entities = project_gold_entityonly(
        gold_doc,
        entity_mode=entity_mode
    ).get("entities", [])
    alignment = match_pred_entities_to_gold(pred_entities, gold_entities)

    type_tp = Counter()
    type_fp = Counter()
    type_fn = Counter()

    gold_by_id = {e["id"]: e for e in gold_entities if e.get("id")}
    pred_by_id = {e["id"]: e for e in pred_entities if e.get("id")}

    for _, gold_id in alignment.pred_to_gold.items():
        ent_type = gold_by_id[gold_id].get("type", "UNKNOWN")
        type_tp[ent_type] += 1

    for pred_id in alignment.unmatched_pred_ids:
        ent_type = pred_by_id[pred_id].get("type", "UNKNOWN")
        type_fp[ent_type] += 1

    for gold_id in alignment.unmatched_gold_ids:
        ent_type = gold_by_id[gold_id].get("type", "UNKNOWN")
        type_fn[ent_type] += 1

    tp = sum(type_tp.values())
    fp = sum(type_fp.values())
    fn = sum(type_fn.values())
    p, r, f1 = prf(tp, fp, fn)

    metrics = {
        "document_id": gold_doc.get("document_id"),
        "entities_tp": tp,
        "entities_fp": fp,
        "entities_fn": fn,
        "entities_precision": p,
        "entities_recall": r,
        "entities_f1": f1,
        "n_pred_entities": len(pred_entities),
        "n_gold_entities": len(gold_entities),
        "n_ambiguous_pred_entities": len(alignment.ambiguous_pred_ids),
    }
    metrics.update(metric_prefix(type_tp, type_fp, type_fn, "entity_type"))
    return metrics


def normalize_fact_only(fact: Dict[str, Any]) -> Tuple[Any, ...]:
    return (
        norm_text_loose(fact.get("subject_type")),
        norm_text_loose(fact.get("subject")),
        norm_text_loose(fact.get("predicate")),
        norm_text_loose(fact.get("fact_type")),
        norm_text_loose(fact.get("object_type")),
        norm_literal(fact.get("object")),
    )


def evaluate_factsonly(pred_doc: Dict[str, Any], gold_doc: Dict[str, Any]) -> Dict[str, Any]:
    pred_facts = pred_doc.get("facts", [])
    gold_facts = project_gold_factsonly(gold_doc).get("facts", [])

    pred_norm = Counter(normalize_fact_only(f) for f in pred_facts)
    gold_norm = Counter(normalize_fact_only(f) for f in gold_facts)

    by_predicate_tp = Counter()
    by_predicate_fp = Counter()
    by_predicate_fn = Counter()

    tp = fp = fn = 0
    for key in set(pred_norm) | set(gold_norm):
        pred_n = pred_norm.get(key, 0)
        gold_n = gold_norm.get(key, 0)
        matched = min(pred_n, gold_n)
        extra_pred = pred_n - matched
        missing = gold_n - matched

        tp += matched
        fp += extra_pred
        fn += missing

        predicate = key[2] if key[2] is not None else "UNKNOWN"
        by_predicate_tp[predicate] += matched
        by_predicate_fp[predicate] += extra_pred
        by_predicate_fn[predicate] += missing

    p, r, f1 = prf(tp, fp, fn)
    metrics = {
        "document_id": gold_doc.get("document_id"),
        "facts_tp": tp,
        "facts_fp": fp,
        "facts_fn": fn,
        "facts_precision": p,
        "facts_recall": r,
        "facts_f1": f1,
        "n_pred_facts": len(pred_facts),
        "n_gold_facts": len(gold_facts),
    }
    metrics.update(metric_prefix(by_predicate_tp, by_predicate_fp, by_predicate_fn, "fact_predicate"))
    return metrics


def normalize_currency(value: Optional[str]) -> Optional[str]:
    v = norm_text_loose(value)
    if v is None:
        return None

    # normalización ligera de moneda
    if v in {"maravedi", "maravedi", "maravedis"}:
        return "maravedi"

    return v


def normalize_event_attribute_value(value: Any) -> Any:
    if isinstance(value, str):
        return norm_text_loose(value)

    if isinstance(value, dict):
        out = {}
        for k, v in value.items():
            nk = norm_text_loose(k)

            if nk == "currency" and isinstance(v, str):
                out[k] = normalize_currency(v)
            elif isinstance(v, str):
                out[k] = norm_text_loose(v)
            else:
                out[k] = v

        return tuple(sorted(out.items()))

    if isinstance(value, list):
        return tuple(normalize_event_attribute_value(v) for v in value)

    return value


def normalize_event_only(event: Dict[str, Any]) -> Dict[str, Any]:
    attrs = event.get("attributes") or {}
    norm_attrs = {
        norm_text_loose(k): tuple(sorted(normalize_event_attribute_value(v) for v in ensure_list(vals)))
        for k, vals in attrs.items()
    }

    args = []
    for arg in event.get("arguments", []):
        role = norm_text_loose(arg.get("role"))
        value = norm_text_loose(arg.get("value"))
        if role is None or value is None:
            continue
        args.append((role, value))

    return {
        "type": norm_text_loose(event.get("type")),
        "subtype": norm_text_loose(event.get("subtype")),
        "attributes": {k: tuple(sorted(v)) for k, v in sorted(norm_attrs.items())},
        "arguments": tuple(sorted(args)),
    }

def event_signature(event_norm: Dict[str, Any]) -> Tuple[Any, ...]:
    return (
        event_norm["type"],
        event_norm["subtype"],
        tuple(sorted(event_norm["attributes"].items())),
        event_norm["arguments"],
    )

def event_header_signature(event_norm: Dict[str, Any]) -> Tuple[Any, ...]:
    return (
        event_norm["type"],
        event_norm["subtype"],
    )


def event_header_score(pred_ev, gold_ev):
    """
    Strict event-header matching.

    A predicted event matches a gold event only if:
    - event type is identical, and
    - event subtype is identical.

    A missing subtype (None) is treated as a valid value:
    - None == None -> match
    - None != explicit subtype -> no match

    Attributes and arguments are not considered.
    """
    if pred_ev["type"] != gold_ev["type"]:
        return -1

    if pred_ev["subtype"] != gold_ev["subtype"]:
        return -1

    return 1

def event_frame_similarity(pred_ev, gold_ev):
    """
    Dice/F1 similarity between normalized frame components.

    Used ONLY to disambiguate events whose headers
    (type, subtype) already match exactly.
    """

    pred_components = []
    gold_components = []

    # Attributes: preserve attribute name + normalized value
    for key, values in pred_ev["attributes"].items():
        for value in values:
            pred_components.append(("attr", key, value))

    for key, values in gold_ev["attributes"].items():
        for value in values:
            gold_components.append(("attr", key, value))

    # Arguments: preserve role + normalized value
    for role, value in pred_ev["arguments"]:
        pred_components.append(("arg", role, value))

    for role, value in gold_ev["arguments"]:
        gold_components.append(("arg", role, value))

    pred_counter = Counter(pred_components)
    gold_counter = Counter(gold_components)

    pred_n = sum(pred_counter.values())
    gold_n = sum(gold_counter.values())

    if pred_n == 0 and gold_n == 0:
        return 1.0

    if pred_n == 0 or gold_n == 0:
        return 0.0

    overlap = sum(
        min(pred_counter[k], gold_counter[k])
        for k in set(pred_counter) | set(gold_counter)
    )

    precision = overlap / pred_n
    recall = overlap / gold_n

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)


def match_events_by_header(
    pred_events_norm: List[Dict[str, Any]],
    gold_events_norm: List[Dict[str, Any]],
) -> List[Tuple[int, int]]:
    """
    One-to-one matching between events.

    Eligibility:
        exact match of event type and subtype.

    Tie-breaking:
        among events sharing the same header, maximize
        normalized frame similarity.

    Frame information never makes an incorrect header eligible.
    """

    pairs = []
    used_pred = set()
    used_gold = set()

    candidates = []

    for i, pe in enumerate(pred_events_norm):
        for j, ge in enumerate(gold_events_norm):

            # Strict header eligibility
            if event_header_score(pe, ge) < 0:
                continue

            frame_sim = event_frame_similarity(pe, ge)

            candidates.append((frame_sim, i, j))

    # Highest frame similarity first
    candidates.sort(reverse=True)

    for frame_sim, i, j in candidates:
        if i in used_pred or j in used_gold:
            continue

        used_pred.add(i)
        used_gold.add(j)
        pairs.append((i, j))

    return pairs


def match_events_greedily(pred_events_norm: List[Dict[str, Any]], gold_events_norm: List[Dict[str, Any]]) -> List[Tuple[int, int]]:
    exact_pairs = []
    used_gold = set()
    gold_signatures = defaultdict(list)

    for j, ge in enumerate(gold_events_norm):
        gold_signatures[event_signature(ge)].append(j)

    for i, pe in enumerate(pred_events_norm):
        candidates = gold_signatures.get(event_signature(pe), [])
        chosen = next((j for j in candidates if j not in used_gold), None)
        if chosen is not None:
            used_gold.add(chosen)
            exact_pairs.append((i, chosen))

    return exact_pairs


def evaluate_eventsonly(pred_doc: Dict[str, Any], gold_doc: Dict[str, Any]) -> Dict[str, Any]:
    gold_event_doc = project_gold_eventsonly(gold_doc)

    pred_events_raw = pred_doc.get("events", [])
    gold_events_raw = gold_event_doc.get("events", [])

    pred_events_norm = [normalize_event_only(e) for e in pred_events_raw]
    gold_events_norm = [normalize_event_only(e) for e in gold_events_raw]

    # 1) métrica dura: exact match total
    exact_pairs = match_events_greedily(pred_events_norm, gold_events_norm)

    exact_tp = len(exact_pairs)
    exact_fp = len(pred_events_norm) - exact_tp
    exact_fn = len(gold_events_norm) - exact_tp
    exact_p, exact_r, exact_f1 = prf(exact_tp, exact_fp, exact_fn)

    # 2) métrica más útil: match por cabecera
    header_pairs = match_events_by_header(pred_events_norm, gold_events_norm)

    header_tp = len(header_pairs)
    header_fp = len(pred_events_norm) - header_tp
    header_fn = len(gold_events_norm) - header_tp
    header_p, header_r, header_f1 = prf(header_tp, header_fp, header_fn)

    # 3) atributos y roles evaluados sobre header_pairs, no sobre exact_pairs
    attr_tp = Counter()
    attr_fp = Counter()
    attr_fn = Counter()

    role_tp = Counter()
    role_fp = Counter()
    role_fn = Counter()

    for i, j in header_pairs:
        pe = pred_events_norm[i]
        ge = gold_events_norm[j]

        # atributos
        all_attr_keys = set(pe["attributes"]) | set(ge["attributes"])
        for key in all_attr_keys:
            metric_key = key if key is not None else "UNKNOWN"
            pred_vals = Counter(pe["attributes"].get(key, ()))
            gold_vals = Counter(ge["attributes"].get(key, ()))

            for val in set(pred_vals) | set(gold_vals):
                matched = min(pred_vals.get(val, 0), gold_vals.get(val, 0))
                attr_tp[metric_key] += matched
                attr_fp[metric_key] += pred_vals.get(val, 0) - matched
                attr_fn[metric_key] += gold_vals.get(val, 0) - matched

        # argumentos por rol
        pred_args = Counter(pe["arguments"])
        gold_args = Counter(ge["arguments"])

        for role, value in set(pred_args) | set(gold_args):
            metric_role = role if role is not None else "UNKNOWN"
            matched = min(pred_args.get((role, value), 0), gold_args.get((role, value), 0))
            role_tp[metric_role] += matched
            role_fp[metric_role] += pred_args.get((role, value), 0) - matched
            role_fn[metric_role] += gold_args.get((role, value), 0) - matched

    metrics = {
        "document_id": gold_doc.get("document_id"),

        # exacto
        "events_exact_tp": exact_tp,
        "events_exact_fp": exact_fp,
        "events_exact_fn": exact_fn,
        "events_exact_precision": exact_p,
        "events_exact_recall": exact_r,
        "events_exact_f1": exact_f1,

        # cabecera
        "events_header_tp": header_tp,
        "events_header_fp": header_fp,
        "events_header_fn": header_fn,
        "events_header_precision": header_p,
        "events_header_recall": header_r,
        "events_header_f1": header_f1,

        "n_pred_events": len(pred_events_raw),
        "n_gold_events": len(gold_events_raw),
        "n_exact_event_pairs": len(exact_pairs),
        "n_header_event_pairs": len(header_pairs),
    }

    metrics.update(metric_prefix(attr_tp, attr_fp, attr_fn, "event_attr"))
    metrics.update(metric_prefix(role_tp, role_fp, role_fn, "event_role"))

    return metrics


def evaluate_task(
    task_name: str,
    pred_doc: Dict[str, Any],
    gold_doc: Dict[str, Any],
    entity_mode: str = ENTITY_MODE,
) -> Dict[str, Any]:
    if task_name == "entityonly":
        return evaluate_entityonly(pred_doc, gold_doc, entity_mode=entity_mode)
    if task_name == "factsonly":
        return evaluate_factsonly(pred_doc, gold_doc)
    if task_name == "eventsonly":
        return evaluate_eventsonly(pred_doc, gold_doc)
    raise ValueError(f"Tarea no soportada: {task_name}")

def collect_fact_errors(pred_doc: Dict[str, Any], gold_doc: Dict[str, Any]) -> Dict[str, Dict[str, List[Dict[str, Any]]]]:
    """
    Devuelve:
    {
      predicate: {
        "fp": [fact_pred, ...],
        "fn": [fact_gold, ...]
      }
    }
    """
    pred_facts = pred_doc.get("facts", [])
    gold_facts = project_gold_factsonly(gold_doc).get("facts", [])

    pred_norm = defaultdict(list)
    gold_norm = defaultdict(list)

    for f in pred_facts:
        key = normalize_fact_only(f)
        pred_norm[key].append(f)

    for f in gold_facts:
        key = normalize_fact_only(f)
        gold_norm[key].append(f)

    errors = defaultdict(lambda: {"fp": [], "fn": []})

    for key in set(pred_norm) | set(gold_norm):
        pred_list = pred_norm.get(key, [])
        gold_list = gold_norm.get(key, [])

        matched = min(len(pred_list), len(gold_list))

        # FP → lo que sobra en pred
        for f in pred_list[matched:]:
            pred_label = f.get("predicate")
            errors[pred_label]["fp"].append(f)

        # FN → lo que falta respecto a gold
        for f in gold_list[matched:]:
            gold_label = f.get("predicate")
            errors[gold_label]["fn"].append(f)

    return errors

## Experiment execution

This section orchestrates end-to-end experiment execution.

For each document, the pipeline:
- builds the prompt,
- queries the selected backend,
- parses and validates model outputs,
- evaluates predictions against gold annotations,
- stores intermediate artifacts,
- and accumulates document-level metrics.

The section also supports:
- batched execution,
- partial runs,
- resumable evaluation,
- and OpenAI batch-processing workflows.

In [ ]:
def get_run_output_dir(
    *,
    task_name: str,
    backend_name: str,
    prompt_variant: str,
    save_prefix: Optional[str] = None,
) -> Path:
    save_prefix = save_prefix or f"{task_name}__{backend_name}__{prompt_variant}"
    prefix_parts = save_prefix.split("__")
    output_dir = OUT_DIR / prefix_parts[0] / prefix_parts[1] / prefix_parts[2] / save_prefix
    output_dir.mkdir(parents=True, exist_ok=True)
    return output_dir

def slice_doc_ids(doc_ids: List[str], start: int = 0, end: Optional[int] = None) -> List[str]:
    if end is None:
        return doc_ids[start:]
    return doc_ids[start:end]

def run_experiment(
    *,
    task_name: str,
    backend_name: str = BACKEND_NAME,
    model_name: Optional[str] = None,
    doc_ids: List[str],
    prompt_variant: str = PROMPT_VARIANT,
    temperature: float = DEFAULT_TEMPERATURE,
    max_output_tokens: int = DEFAULT_MAX_OUTPUT_TOKENS,
    thinking_level: str = DEFAULT_THINKING_LEVEL,
    example_doc_ids: Optional[List[str]] = None,
    max_chars_per_example: Optional[int] = DEFAULT_MAX_CHARS_PER_EXAMPLE,
    save_prefix: Optional[str] = None,
    start_idx: int = 0,
    end_idx: Optional[int] = None,
    entity_mode: str = ENTITY_MODE,
) -> pd.DataFrame:
    backend = make_backend(backend_name=backend_name, model_name=model_name)
    schema = task_schema(task_name)

    output_dir = get_run_output_dir(
        task_name=task_name,
        backend_name=backend_name,
        prompt_variant=prompt_variant,
        save_prefix=save_prefix,
    )

    batch_doc_ids = slice_doc_ids(doc_ids, start=start_idx, end=end_idx)

    batch_manifest = {
    "task_name": task_name,
    "backend_name": backend_name,
    "model_name": model_name or BACKEND_CONFIGS[backend_name]["model"],
    "prompt_variant": prompt_variant,
    "save_prefix": output_dir.name,
    "start_idx": start_idx,
    "end_idx": end_idx,
    "n_docs_in_batch": len(batch_doc_ids),
    "doc_ids": batch_doc_ids,
    "entity_mode": entity_mode,
    }

    manifest_name = f"batch_{start_idx}_{end_idx if end_idx is not None else 'end'}.json"
    save_json(output_dir / manifest_name, batch_manifest)

    rows = []

    batch_label = f"{output_dir.name}[{start_idx}:{end_idx if end_idx is not None else 'end'}]"

    for doc_id in tqdm(batch_doc_ids, desc=batch_label):
        text = load_text(TEXT_DIR / f"{doc_id}.txt")
        gold_doc = load_json(GOLD_DIR / f"{doc_id}.json")
        gold_event_meta = extract_gold_event_labels(gold_doc)

        prompt = build_prompt(
            task_name=task_name,
            document_text=text,
            doc_id=doc_id,
            variant=prompt_variant,
            example_doc_ids=example_doc_ids,
            max_chars_per_example=max_chars_per_example,
            entity_mode=entity_mode,
        )

        save_text(output_dir / f"{doc_id}_prompt.txt", prompt)

        try:
            generation = generate_with_retries(
                backend,
                prompt=prompt,
                schema=schema,
                temperature=temperature,
                max_output_tokens=max_output_tokens,
                thinking_level=thinking_level,
            )
        except Exception as e:
            error_payload = {
                "document_id": doc_id,
                "task_name": task_name,
                "backend_name": backend_name,
                "model_name": model_name or BACKEND_CONFIGS[backend_name]["model"],
                "prompt_variant": prompt_variant,
                "prompt": prompt,
                "raw_response": None,
                "cleaned_text": None,
                "parse_error": f"backend_error: {e}",
                "schema_valid": False,
                "schema_errors": [],
                "provider_meta": {},
                "entity_mode": entity_mode,
            }
            save_json(output_dir / f"{doc_id}.raw.json", error_payload)

            row = {
                "document_id": doc_id,
                "task_name": task_name,
                "backend_name": backend_name,
                "model_name": model_name or BACKEND_CONFIGS[backend_name]["model"],
                "prompt_variant": prompt_variant,
                "schema_valid": False,
                "parse_error": f"backend_error: {e}",
                "n_schema_errors": 0,
                "entity_mode": entity_mode,
                **gold_event_meta,
            }

            projected_gold = project_gold(
                task_name,
                gold_doc,
                entity_mode=entity_mode,
            )
            if task_name == "entityonly":
                row.update({
                    "entities_tp": 0,
                    "entities_fp": 0,
                    "entities_fn": len(projected_gold.get("entities", [])),
                })
            elif task_name == "factsonly":
                row.update({
                    "facts_tp": 0,
                    "facts_fp": 0,
                    "facts_fn": len(projected_gold.get("facts", [])),
                })
            elif task_name == "eventsonly":
                row.update({
                    "events_exact_tp": 0,
                    "events_exact_fp": 0,
                    "events_exact_fn": len(projected_gold.get("events", [])),
                    "events_header_tp": 0,
                    "events_header_fp": 0,
                    "events_header_fn": len(projected_gold.get("events", [])),
                })

            rows.append(row)
            save_json(output_dir / f"{doc_id}.metrics.json", row)
            continue

        parsed = parse_and_validate(generation["raw_text"], schema=schema)

        raw_payload = {
            "document_id": doc_id,
            "task_name": task_name,
            "backend_name": backend_name,
            "model_name": model_name or BACKEND_CONFIGS[backend_name]["model"],
            "prompt_variant": prompt_variant,
            "prompt": prompt,
            "raw_response": generation["raw_text"],
            "cleaned_text": parsed["cleaned_text"],
            "parse_error": parsed["parse_error"],
            "schema_valid": parsed["schema_valid"],
            "schema_errors": parsed["schema_errors"],
            "provider_meta": generation.get("provider_meta", {}),
        }
        save_json(output_dir / f"{doc_id}.raw.json", raw_payload)

        if parsed["parsed"] is None:
            row = {
                "document_id": doc_id,
                "task_name": task_name,
                "backend_name": backend_name,
                "model_name": model_name or BACKEND_CONFIGS[backend_name]["model"],
                "prompt_variant": prompt_variant,
                "schema_valid": False,
                "parse_error": parsed["parse_error"],
                "n_schema_errors": 0,
                "entity_mode": entity_mode,
            }

            projected_gold = project_gold(
                task_name,
                gold_doc,
                entity_mode=entity_mode,
            )
            if task_name == "entityonly":
                row.update({
                    "entities_tp": 0,
                    "entities_fp": 0,
                    "entities_fn": len(projected_gold.get("entities", [])),
                })
            elif task_name == "factsonly":
                row.update({
                    "facts_tp": 0,
                    "facts_fp": 0,
                    "facts_fn": len(projected_gold.get("facts", [])),
                })
            elif task_name == "eventsonly":
                n_gold_events = len(projected_gold.get("events", []))

                row.update({
                    "events_exact_tp": 0,
                    "events_exact_fp": 0,
                    "events_exact_fn": n_gold_events,

                    "events_header_tp": 0,
                    "events_header_fp": 0,
                    "events_header_fn": n_gold_events,
                })

            rows.append(row)
            save_json(output_dir / f"{doc_id}.metrics.json", row)
            continue

        pred_doc = parsed["parsed"]
        metrics = evaluate_task(
            task_name,
            pred_doc,
            gold_doc,
            entity_mode=entity_mode,
        )
        metrics.update(gold_event_meta)
        metrics["task_name"] = task_name
        metrics["backend_name"] = backend_name
        metrics["model_name"] = model_name or BACKEND_CONFIGS[backend_name]["model"]
        metrics["prompt_variant"] = prompt_variant
        metrics["schema_valid"] = parsed["schema_valid"]
        metrics["parse_error"] = parsed["parse_error"]
        metrics["n_schema_errors"] = len(parsed["schema_errors"])
        metrics["entity_mode"] = entity_mode

        rows.append(metrics)

        if task_name == "factsonly":
          fact_errors = collect_fact_errors(pred_doc, gold_doc)
          save_json(output_dir / f"{doc_id}.fact_errors.json", fact_errors)

        save_json(output_dir / f"{doc_id}.parsed.json", pred_doc)
        save_json(output_dir / f"{doc_id}.metrics.json", metrics)

    df = pd.DataFrame(rows)

    batch_csv_name = f"results_{start_idx}_{end_idx if end_idx is not None else 'end'}.csv"
    df.to_csv(output_dir / batch_csv_name, index=False)

    return df

def get_openai_client() -> OpenAI:
    api_key = os.environ.get("OPENAI_API_KEY", "")
    if not api_key:
        raise ValueError("Falta OPENAI_API_KEY")
    return OpenAI(api_key=api_key, timeout=600.0)


def make_openai_batch_request(
    *,
    custom_id: str,
    model: str,
    prompt: str,
    max_output_tokens: int,
    temperature: float = DEFAULT_TEMPERATURE,
) -> Dict[str, Any]:
    return {
        "custom_id": custom_id,
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": model,
            "messages": [
                {"role": "user", "content": prompt}
            ],
            "temperature": temperature,
            "max_completion_tokens": max_output_tokens,
            "response_format": {"type": "json_object"},
        },
    }


def prepare_openai_batch(
    *,
    task_name: str,
    doc_ids: List[str],
    model_name: str,
    prompt_variant: str = PROMPT_VARIANT,
    example_doc_ids: Optional[List[str]] = None,
    max_chars_per_example: Optional[int] = DEFAULT_MAX_CHARS_PER_EXAMPLE,
    max_output_tokens: int = DEFAULT_MAX_OUTPUT_TOKENS,
    temperature: float = DEFAULT_TEMPERATURE,
    save_prefix: str,
    entity_mode: str = ENTITY_MODE,
    start_idx: int = 0,
    end_idx: Optional[int] = None,
) -> Path:
    output_dir = get_run_output_dir(
        task_name=task_name,
        backend_name="openai_batch",
        prompt_variant=prompt_variant,
        save_prefix=save_prefix,
    )

    batch_doc_ids = slice_doc_ids(doc_ids, start=start_idx, end=end_idx)

    input_jsonl = output_dir / f"batch_input_{start_idx}_{end_idx if end_idx is not None else 'end'}.jsonl"

    manifest = {
        "task_name": task_name,
        "backend_name": "openai_batch",
        "model_name": model_name,
        "prompt_variant": prompt_variant,
        "save_prefix": output_dir.name,
        "start_idx": start_idx,
        "end_idx": end_idx,
        "n_docs_in_batch": len(batch_doc_ids),
        "doc_ids": batch_doc_ids,
        "entity_mode": entity_mode,
        "max_output_tokens": max_output_tokens,
        "temperature": temperature,
        "batch_input_file": str(input_jsonl),
    }

    save_json(output_dir / f"batch_manifest_{start_idx}_{end_idx if end_idx is not None else 'end'}.json", manifest)

    with open(input_jsonl, "w", encoding="utf-8") as f:
        for doc_id in batch_doc_ids:
            text = load_text(TEXT_DIR / f"{doc_id}.txt")

            prompt = build_prompt(
                task_name=task_name,
                document_text=text,
                doc_id=doc_id,
                variant=prompt_variant,
                example_doc_ids=example_doc_ids,
                max_chars_per_example=max_chars_per_example,
                entity_mode=entity_mode,
            )

            save_text(output_dir / f"{doc_id}_prompt.txt", prompt)

            custom_id = f"{task_name}::{doc_id}"

            request = make_openai_batch_request(
                custom_id=custom_id,
                model=model_name,
                prompt=prompt,
                max_output_tokens=max_output_tokens,
                temperature=temperature,
            )

            f.write(json.dumps(request, ensure_ascii=False) + "\n")

    print(f"Batch input written to: {input_jsonl}")
    return input_jsonl

def submit_openai_batch(
    input_jsonl_path: Path,
    *,
    metadata: Optional[Dict[str, str]] = None,
) -> Dict[str, Any]:
    client = get_openai_client()

    with open(input_jsonl_path, "rb") as f:
        uploaded = client.files.create(
            file=f,
            purpose="batch",
        )

    batch = client.batches.create(
        input_file_id=uploaded.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
        metadata=metadata or {},
    )

    out = {
        "input_jsonl_path": str(input_jsonl_path),
        "input_file_id": uploaded.id,
        "batch_id": batch.id,
        "status": batch.status,
        "output_file_id": getattr(batch, "output_file_id", None),
        "error_file_id": getattr(batch, "error_file_id", None),
    }

    save_json(input_jsonl_path.parent / f"openai_batch_{batch.id}.json", out)

    print(json.dumps(out, ensure_ascii=False, indent=2))
    return out

def retrieve_openai_batch(batch_id: str) -> Dict[str, Any]:
    client = get_openai_client()
    batch = client.batches.retrieve(batch_id)

    out = {
        "id": batch.id,
        "status": batch.status,
        "endpoint": batch.endpoint,
        "input_file_id": batch.input_file_id,
        "output_file_id": getattr(batch, "output_file_id", None),
        "error_file_id": getattr(batch, "error_file_id", None),
        "request_counts": str(getattr(batch, "request_counts", None)),
        "created_at": getattr(batch, "created_at", None),
        "completed_at": getattr(batch, "completed_at", None),
        "failed_at": getattr(batch, "failed_at", None),
        "expired_at": getattr(batch, "expired_at", None),
    }

    print(json.dumps(out, ensure_ascii=False, indent=2))
    return out


def download_openai_file(file_id: str, out_path: Path) -> Path:
    client = get_openai_client()
    content = client.files.content(file_id)
    text = content.text

    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(text)

    return out_path

def parse_openai_batch_output_line(line: str) -> Tuple[str, Optional[str], Dict[str, Any], Optional[Dict[str, Any]]]:
    obj = json.loads(line)
    custom_id = obj.get("custom_id")
    error = obj.get("error")

    if error is not None:
        return custom_id, None, obj, error

    response = obj.get("response") or {}
    body = response.get("body") or {}
    choices = body.get("choices") or []

    raw_text = ""
    if choices:
        raw_text = choices[0].get("message", {}).get("content") or ""

    return custom_id, raw_text, obj, None


def collect_openai_batch_results(
    *,
    batch_output_path: Path,
    task_name: str,
    model_name: str,
    prompt_variant: str,
    save_prefix: str,
    entity_mode: str = ENTITY_MODE,
    error_output_path: Optional[Path] = None,
) -> pd.DataFrame:
    output_dir = get_run_output_dir(
        task_name=task_name,
        backend_name="openai_batch",
        prompt_variant=prompt_variant,
        save_prefix=save_prefix,
    )

    schema = task_schema(task_name)
    rows = []

    seen_doc_ids = set()

    with open(batch_output_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            custom_id, raw_text, full_obj, error = parse_openai_batch_output_line(line)

            # custom_id = "{task_name}::{doc_id}"
            doc_id = custom_id.split("::", 1)[1] if custom_id and "::" in custom_id else custom_id
            seen_doc_ids.add(doc_id)

            gold_doc = load_json(GOLD_DIR / f"{doc_id}.json")
            gold_event_meta = extract_gold_event_labels(gold_doc)

            if error is not None:
                parsed = {
                    "parsed": None,
                    "cleaned_text": None,
                    "parse_error": f"batch_error: {error}",
                    "schema_valid": False,
                    "schema_errors": [],
                }
            else:
                parsed = parse_and_validate(raw_text or "", schema=schema)

            raw_payload = {
                "document_id": doc_id,
                "task_name": task_name,
                "backend_name": "openai_batch",
                "model_name": model_name,
                "prompt_variant": prompt_variant,
                "raw_response": raw_text,
                "cleaned_text": parsed["cleaned_text"],
                "parse_error": parsed["parse_error"],
                "schema_valid": parsed["schema_valid"],
                "schema_errors": parsed["schema_errors"],
                "provider_meta": full_obj,
                "entity_mode": entity_mode,
            }
            save_json(output_dir / f"{doc_id}.raw.json", raw_payload)

            if parsed["parsed"] is None:
                projected_gold = project_gold(
                    task_name,
                    gold_doc,
                    entity_mode=entity_mode,
                )

                row = {
                    "document_id": doc_id,
                    "task_name": task_name,
                    "backend_name": "openai_batch",
                    "model_name": model_name,
                    "prompt_variant": prompt_variant,
                    "schema_valid": False,
                    "parse_error": parsed["parse_error"],
                    "n_schema_errors": 0,
                    "entity_mode": entity_mode,
                    **gold_event_meta,
                }

                if task_name == "entityonly":
                    row.update({
                        "entities_tp": 0,
                        "entities_fp": 0,
                        "entities_fn": len(projected_gold.get("entities", [])),
                    })
                elif task_name == "factsonly":
                    row.update({
                        "facts_tp": 0,
                        "facts_fp": 0,
                        "facts_fn": len(projected_gold.get("facts", [])),
                    })
                elif task_name == "eventsonly":
                    row.update({
                        "events_exact_tp": 0,
                        "events_exact_fp": 0,
                        "events_exact_fn": len(projected_gold.get("events", [])),
                        "events_header_tp": 0,
                        "events_header_fp": 0,
                        "events_header_fn": len(projected_gold.get("events", [])),
                    })

                rows.append(row)
                save_json(output_dir / f"{doc_id}.metrics.json", row)
                continue

            pred_doc = parsed["parsed"]

            metrics = evaluate_task(
                task_name,
                pred_doc,
                gold_doc,
                entity_mode=entity_mode,
            )
            metrics.update(gold_event_meta)
            metrics["task_name"] = task_name
            metrics["backend_name"] = "openai_batch"
            metrics["model_name"] = model_name
            metrics["prompt_variant"] = prompt_variant
            metrics["schema_valid"] = parsed["schema_valid"]
            metrics["parse_error"] = parsed["parse_error"]
            metrics["n_schema_errors"] = len(parsed["schema_errors"])
            metrics["entity_mode"] = entity_mode

            rows.append(metrics)

            if task_name == "factsonly":
                fact_errors = collect_fact_errors(pred_doc, gold_doc)
                save_json(output_dir / f"{doc_id}.fact_errors.json", fact_errors)

            save_json(output_dir / f"{doc_id}.parsed.json", pred_doc)
            save_json(output_dir / f"{doc_id}.metrics.json", metrics)

    if error_output_path is not None and Path(error_output_path).exists():
        save_json(output_dir / "batch_error_file_pointer.json", {
            "error_output_path": str(error_output_path)
        })

    df = pd.DataFrame(rows)
    df.to_csv(output_dir / "results_batch_collected.csv", index=False)

    return df

def run_experiment_openai_batch(
    *,
    task_name: str,
    model_name: str,
    doc_ids: List[str],
    prompt_variant: str = PROMPT_VARIANT,
    temperature: float = DEFAULT_TEMPERATURE,
    max_output_tokens: int = DEFAULT_MAX_OUTPUT_TOKENS,
    thinking_level: str = DEFAULT_THINKING_LEVEL,
    example_doc_ids: Optional[List[str]] = None,
    max_chars_per_example: Optional[int] = DEFAULT_MAX_CHARS_PER_EXAMPLE,
    save_prefix: Optional[str] = None,
    start_idx: int = 0,
    end_idx: Optional[int] = None,
    entity_mode: str = ENTITY_MODE,
    wait: bool = False,
    poll_seconds: int = 60,
) -> Dict[str, Any]:
    """
    Equivalente batch de run_experiment para OpenAI.

    Si wait=False:
        prepara y lanza el batch, devuelve batch_info.
    Si wait=True:
        además espera a completed, descarga resultados, evalúa y devuelve df.
    """

    save_prefix = save_prefix or f"{task_name}__openai_batch__{prompt_variant}"

    input_path = prepare_openai_batch(
        task_name=task_name,
        doc_ids=doc_ids,
        model_name=model_name,
        prompt_variant=prompt_variant,
        example_doc_ids=example_doc_ids,
        max_chars_per_example=max_chars_per_example,
        max_output_tokens=max_output_tokens,
        temperature=temperature,
        save_prefix=save_prefix,
        entity_mode=entity_mode,
        start_idx=start_idx,
        end_idx=end_idx,
    )

    batch_info = submit_openai_batch(
        input_path,
        metadata={
            "task_name": task_name,
            "model_name": model_name,
            "prompt_variant": prompt_variant,
            "save_prefix": save_prefix,
        },
    )

    result = {
        "batch_info": batch_info,
        "input_path": input_path,
        "df": None,
        "output_path": None,
        "error_path": None,
    }

    if not wait:
        return result

    while True:
        status = retrieve_openai_batch(batch_info["batch_id"])
        batch_status = status["status"]

        if batch_status == "completed":
            break

        if batch_status in {"failed", "expired", "cancelled"}:
            raise RuntimeError(f"Batch ended with status={batch_status}: {status}")

        print(f"Batch status={batch_status}. Waiting {poll_seconds}s...")
        time.sleep(poll_seconds)

    output_dir = get_run_output_dir(
        task_name=task_name,
        backend_name="openai_batch",
        prompt_variant=prompt_variant,
        save_prefix=save_prefix,
    )

    output_path = download_openai_file(
        status["output_file_id"],
        output_dir / "batch_output.jsonl",
    )

    error_path = None
    if status.get("error_file_id"):
        error_path = download_openai_file(
            status["error_file_id"],
            output_dir / "batch_errors.jsonl",
        )

    df = collect_openai_batch_results(
        batch_output_path=output_path,
        task_name=task_name,
        model_name=model_name,
        prompt_variant=prompt_variant,
        save_prefix=save_prefix,
        entity_mode=entity_mode,
        error_output_path=error_path,
    )

    result["df"] = df
    result["output_path"] = output_path
    result["error_path"] = error_path

    return result


## Summary

This section defines utilities for aggregating and exporting experiment results.

It loads saved document-level metrics, combines partial runs belonging to the
same experimental setup, computes task-specific summary tables, and writes
paper-ready CSV files.

The summary utilities are used after model execution to compare:
- tasks,
- backends,
- prompt variants,
- entity modes,
- event types,
- event subtypes,
- event attributes,
- and event roles.

In [ ]:
def load_all_metrics_for_setup(
    *,
    task_name: str,
    backend_name: str,
    prompt_variant: str,
    save_prefix: Optional[str] = None,
) -> pd.DataFrame:
    output_dir = get_run_output_dir(
        task_name=task_name,
        backend_name=backend_name,
        prompt_variant=prompt_variant,
        save_prefix=save_prefix,
    )

    metric_files = sorted(output_dir.glob("*.metrics.json"))

    rows = []
    for path in metric_files:
        try:
            rows.append(load_json(path))
        except Exception as e:
            print(f"Skipping unreadable metrics file {path.name}: {e}")

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)

    # si hubiera duplicados del mismo doc, nos quedamos con la última aparición
    if "document_id" in df.columns:
        df = df.drop_duplicates(subset=["document_id"], keep="last").reset_index(drop=True)

    return df

def summarize_saved_setup(
    *,
    task_name: str,
    backend_name: str,
    prompt_variant: str,
    save_prefix: Optional[str] = None,
) -> Dict[str, pd.DataFrame]:
    df = load_all_metrics_for_setup(
        task_name=task_name,
        backend_name=backend_name,
        prompt_variant=prompt_variant,
        save_prefix=save_prefix,
    )

    out = {
        "documents": df,
        "main": pd.DataFrame(),
        "breakdown": pd.DataFrame(),
        "attributes": pd.DataFrame(),
        "relations": pd.DataFrame(),
        "roles": pd.DataFrame(),
        "all_facts": pd.DataFrame(),
        "by_type": pd.DataFrame(),
        "by_subtype": pd.DataFrame(),
        "per_document": pd.DataFrame(),
    }

    if df.empty:
        return out

    if task_name == "factsonly":
        out["main"] = summarize_facts_main_with_macro(df)
        facts_breakdown = summarize_facts_breakdown(df)
        out["attributes"] = facts_breakdown["attributes"]
        out["relations"] = facts_breakdown["relations"]
        out["all_facts"] = facts_breakdown["all"]
        out["by_type"] = summarize_by_event_type(df, task_name)
        out["by_subtype"] = summarize_by_event_subtype(df, task_name)

    else:
        out["main"] = summarize_main_metrics(df, task_name)

        if task_name == "entityonly":
            out["main"] = summarize_entities_main_with_macro(df)
            out["breakdown"] = summarize_entities_breakdown(df)

        elif task_name == "eventsonly":
            out["main"] = summarize_events_main_with_macro(df)
            events_breakdown = summarize_events_breakdown(df)
            out["attributes"] = events_breakdown["attributes"]
            out["roles"] = events_breakdown["roles"]
            out["by_type"] = summarize_by_event_type(df, task_name)
            out["by_subtype"] = summarize_by_event_subtype(df, task_name)

    return out

def aggregate_prefix_totals(df: pd.DataFrame, prefix: str) -> Tuple[int, int, int]:
    """
    Suma todos los tp/fp/fn de columnas tipo:
      {prefix}_{label}_tp / fp / fn
    """
    tp = 0
    fp = 0
    fn = 0

    for col in df.columns:
        if col.startswith(prefix + "_") and col.endswith("_tp"):
            tp += int(df[col].fillna(0).sum())
        elif col.startswith(prefix + "_") and col.endswith("_fp"):
            fp += int(df[col].fillna(0).sum())
        elif col.startswith(prefix + "_") and col.endswith("_fn"):
            fn += int(df[col].fillna(0).sum())

    return tp, fp, fn

def aggregate_selected_labels(df: pd.DataFrame, prefix: str, labels: Iterable[str]) -> Tuple[int, int, int]:
    """
    Suma tp/fp/fn solo para un subconjunto de labels de columnas tipo:
      {prefix}_{label}_tp / fp / fn
    """
    labels = set(labels)
    tp = 0
    fp = 0
    fn = 0

    pattern = re.compile(rf"^{re.escape(prefix)}_(.+)_(tp|fp|fn)$")

    for col in df.columns:
        m = pattern.match(col)
        if not m:
            continue

        label = m.group(1)
        stat = m.group(2)

        if label not in labels:
            continue

        value = int(df[col].fillna(0).sum())

        if stat == "tp":
            tp += value
        elif stat == "fp":
            fp += value
        elif stat == "fn":
            fn += value

    return tp, fp, fn

def summarize_main_metrics(df: pd.DataFrame, task_name: str) -> pd.DataFrame:
    rows = []

    if task_name == "entityonly":
        specs = [("entities", "entities_tp", "entities_fp", "entities_fn")]

    elif task_name == "factsonly":
        specs = [("facts", "facts_tp", "facts_fp", "facts_fn")]

    elif task_name == "eventsonly":
        specs = [
            ("events_exact", "events_exact_tp", "events_exact_fp", "events_exact_fn"),
            ("events_header", "events_header_tp", "events_header_fp", "events_header_fn"),
        ]

    else:
        raise ValueError(f"Tarea no soportada: {task_name}")

    # métricas base
    for name, tp_col, fp_col, fn_col in specs:
        tp = int(df.get(tp_col, pd.Series(dtype=float)).fillna(0).sum())
        fp = int(df.get(fp_col, pd.Series(dtype=float)).fillna(0).sum())
        fn = int(df.get(fn_col, pd.Series(dtype=float)).fillna(0).sum())
        p, r, f1 = prf(tp, fp, fn)

        rows.append({
            "metric": name,
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "precision": p,
            "recall": r,
            "f1": f1,
        })

    # agregados extra por tarea
    if task_name == "factsonly":
        # todos los facts ya están en "facts", pero aquí añadimos el desglose global
        attr_tp, attr_fp, attr_fn = aggregate_selected_labels(
            df, "fact_predicate", FACT_ATTRIBUTE_PREDICATES
        )
        p, r, f1 = prf(attr_tp, attr_fp, attr_fn)
        rows.append({
            "metric": "fact_attrs",
            "tp": attr_tp,
            "fp": attr_fp,
            "fn": attr_fn,
            "precision": p,
            "recall": r,
            "f1": f1,
        })

        rel_tp, rel_fp, rel_fn = aggregate_selected_labels(
            df, "fact_predicate", FACT_RELATION_PREDICATES
        )
        p, r, f1 = prf(rel_tp, rel_fp, rel_fn)
        rows.append({
            "metric": "fact_rels",
            "tp": rel_tp,
            "fp": rel_fp,
            "fn": rel_fn,
            "precision": p,
            "recall": r,
            "f1": f1,
        })

    elif task_name == "eventsonly":
        attr_tp, attr_fp, attr_fn = aggregate_prefix_totals(df, "event_attr")
        p, r, f1 = prf(attr_tp, attr_fp, attr_fn)
        rows.append({
            "metric": "event_attrs",
            "tp": attr_tp,
            "fp": attr_fp,
            "fn": attr_fn,
            "precision": p,
            "recall": r,
            "f1": f1,
        })

        rel_tp, rel_fp, rel_fn = aggregate_prefix_totals(df, "event_role")
        p, r, f1 = prf(rel_tp, rel_fp, rel_fn)
        rows.append({
            "metric": "event_rels",
            "tp": rel_tp,
            "fp": rel_fp,
            "fn": rel_fn,
            "precision": p,
            "recall": r,
            "f1": f1,
        })

        all_tp = attr_tp + rel_tp
        all_fp = attr_fp + rel_fp
        all_fn = attr_fn + rel_fn

        p, r, f1 = prf(all_tp, all_fp, all_fn)

        rows.append({
            "metric": "event_all",
            "tp": all_tp,
            "fp": all_fp,
            "fn": all_fn,
            "precision": p,
            "recall": r,
            "f1": f1,
        })

    return pd.DataFrame(rows)

def summarize_prefixed_metrics(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    """
    Agrega globalmente métricas del tipo:
      {prefix}_{label}_tp
      {prefix}_{label}_fp
      {prefix}_{label}_fn

    y devuelve una tabla con:
      label, tp, fp, fn, precision, recall, f1
    """
    pattern = re.compile(rf"^{re.escape(prefix)}_(.+)_(tp|fp|fn)$")

    grouped: Dict[str, Dict[str, int]] = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})

    for col in df.columns:
        m = pattern.match(col)
        if not m:
            continue

        label = m.group(1)
        stat = m.group(2)

        value = int(df[col].fillna(0).sum())
        grouped[label][stat] += value

    rows = []
    for label, counts in sorted(grouped.items()):
        tp = counts["tp"]
        fp = counts["fp"]
        fn = counts["fn"]
        p, r, f1 = prf(tp, fp, fn)

        rows.append({
            "label": label,
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "precision": p,
            "recall": r,
            "f1": f1,
        })

    return pd.DataFrame(rows).sort_values(
        by=["f1", "recall", "precision", "label"],
        ascending=[False, False, False, True]
    ).reset_index(drop=True)

def safe_mean(values: Iterable[float]) -> float:
    vals = [float(v) for v in values if pd.notna(v)]
    return sum(vals) / len(vals) if vals else 0.0


def summarize_prefixed_metrics_per_document(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    """
    Para cada documento, agrega métricas del tipo:
      {prefix}_{label}_tp
      {prefix}_{label}_fp
      {prefix}_{label}_fn

    y devuelve una tabla larga con columnas:
      document_id, label, tp, fp, fn, precision, recall, f1
    """
    pattern = re.compile(rf"^{re.escape(prefix)}_(.+)_(tp|fp|fn)$")

    rows = []

    for _, row in df.iterrows():
        grouped: Dict[str, Dict[str, int]] = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})

        for col in df.columns:
            m = pattern.match(col)
            if not m:
                continue

            label = m.group(1)
            stat = m.group(2)

            value = row.get(col, 0)
            if pd.isna(value):
                value = 0

            grouped[label][stat] += int(value)

        for label, counts in grouped.items():
            tp = counts["tp"]
            fp = counts["fp"]
            fn = counts["fn"]
            p, r, f1 = prf(tp, fp, fn)

            rows.append({
                "document_id": row.get("document_id"),
                "label": label,
                "tp": tp,
                "fp": fp,
                "fn": fn,
                "precision": p,
                "recall": r,
                "f1": f1,
            })

    out = pd.DataFrame(rows)
    if out.empty:
        return out

    return out.sort_values(
        by=["document_id", "f1", "recall", "precision", "label"],
        ascending=[True, False, False, False, True]
    ).reset_index(drop=True)


def add_macro_columns_from_breakdown(
    main_df: pd.DataFrame,
    *,
    breakdown_df: pd.DataFrame,
    metric_name: str,
) -> pd.DataFrame:
    """
    Añade columnas macro_precision / macro_recall / macro_f1
    a la fila cuyo metric == metric_name, calculadas como media simple
    sobre el breakdown por label.
    """
    out = main_df.copy()

    macro_p = safe_mean(breakdown_df["precision"]) if not breakdown_df.empty else 0.0
    macro_r = safe_mean(breakdown_df["recall"]) if not breakdown_df.empty else 0.0
    macro_f1 = safe_mean(breakdown_df["f1"]) if not breakdown_df.empty else 0.0

    if "macro_precision" not in out.columns:
        out["macro_precision"] = pd.NA
    if "macro_recall" not in out.columns:
        out["macro_recall"] = pd.NA
    if "macro_f1" not in out.columns:
        out["macro_f1"] = pd.NA

    mask = out["metric"] == metric_name
    out.loc[mask, "macro_precision"] = macro_p
    out.loc[mask, "macro_recall"] = macro_r
    out.loc[mask, "macro_f1"] = macro_f1

    return out


def summarize_facts_main_with_macro(df: pd.DataFrame) -> pd.DataFrame:
    """
    Devuelve la tabla principal de facts con:
    - micro metrics ya existentes
    - macro metrics añadidas para:
      facts, fact_attrs, fact_rels
    """
    main = summarize_main_metrics(df, "factsonly")
    breakdown = summarize_facts_breakdown(df)

    main = add_macro_columns_from_breakdown(
        main,
        breakdown_df=breakdown["all"],
        metric_name="facts",
    )
    main = add_macro_columns_from_breakdown(
        main,
        breakdown_df=breakdown["attributes"],
        metric_name="fact_attrs",
    )
    main = add_macro_columns_from_breakdown(
        main,
        breakdown_df=breakdown["relations"],
        metric_name="fact_rels",
    )

    return main

def summarize_entities_main_with_macro(df: pd.DataFrame) -> pd.DataFrame:
    main = summarize_main_metrics(df, "entityonly")
    breakdown = summarize_entities_breakdown(df)

    main = add_macro_columns_from_breakdown(
        main,
        breakdown_df=breakdown,
        metric_name="entities",
    )

    return main

def summarize_events_main_with_macro(df: pd.DataFrame) -> pd.DataFrame:
    main = summarize_main_metrics(df, "eventsonly")
    breakdown = summarize_events_breakdown(df)

    if "macro_precision" not in main.columns:
        main["macro_precision"] = pd.NA
    if "macro_recall" not in main.columns:
        main["macro_recall"] = pd.NA
    if "macro_f1" not in main.columns:
        main["macro_f1"] = pd.NA

    attrs_df = breakdown["attributes"]
    roles_df = breakdown["roles"]

    if attrs_df.empty and roles_df.empty:
        all_df = pd.DataFrame()
    else:
        all_df = pd.concat([attrs_df, roles_df], ignore_index=True)

    main = add_macro_columns_from_breakdown(
        main,
        breakdown_df=attrs_df,
        metric_name="event_attrs",
    )

    main = add_macro_columns_from_breakdown(
        main,
        breakdown_df=roles_df,
        metric_name="event_rels",
    )

    main = add_macro_columns_from_breakdown(
        main,
        breakdown_df=all_df,
        metric_name="event_all",
    )

    main = add_header_macro_by_event_labels(main, df)

    return main

def add_header_macro_by_event_labels(main: pd.DataFrame, df: pd.DataFrame) -> pd.DataFrame:
    """
    Añade macro-F1 para events_header:
    - macro_f1_by_type
    - macro_f1_by_subtype

    Evita llamar a summarize_by_event_type/subtype para no crear recursión.
    """
    out = main.copy()

    def macro_header_from_group(group_col: str) -> Any:
        if df.empty or group_col not in df.columns:
            return pd.NA

        f1s = []

        for _, subdf in df.groupby(group_col, dropna=False):
            tp = int(subdf.get("events_header_tp", pd.Series(dtype=float)).fillna(0).sum())
            fp = int(subdf.get("events_header_fp", pd.Series(dtype=float)).fillna(0).sum())
            fn = int(subdf.get("events_header_fn", pd.Series(dtype=float)).fillna(0).sum())

            _, _, f1 = prf(tp, fp, fn)
            f1s.append(f1)

        return sum(f1s) / len(f1s) if f1s else pd.NA

    macro_by_type = macro_header_from_group("gold_event_type_combo")
    macro_by_subtype = macro_header_from_group("gold_event_subtype_combo")

    if "macro_f1_by_type" not in out.columns:
        out["macro_f1_by_type"] = pd.NA

    if "macro_f1_by_subtype" not in out.columns:
        out["macro_f1_by_subtype"] = pd.NA

    mask = out["metric"] == "events_header"
    out.loc[mask, "macro_f1_by_type"] = macro_by_type
    out.loc[mask, "macro_f1_by_subtype"] = macro_by_subtype

    return out

def summarize_prefixed_metrics_with_document_frequency(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    """
    Igual que summarize_prefixed_metrics, pero añade:
    - n_docs_with_gold: nº docs donde el label tiene soporte gold (tp+fn > 0)
    - n_docs_predicted: nº docs donde el label fue predicho (tp+fp > 0)
    """
    full = summarize_prefixed_metrics(df, prefix)
    per_doc = summarize_prefixed_metrics_per_document(df, prefix)

    if full.empty or per_doc.empty:
        if "n_docs_with_gold" not in full.columns:
            full["n_docs_with_gold"] = 0
        if "n_docs_predicted" not in full.columns:
            full["n_docs_predicted"] = 0
        return full

    gold_counts = (
        per_doc.assign(has_gold=lambda x: (x["tp"] + x["fn"]) > 0)
        .groupby("label")["has_gold"]
        .sum()
        .rename("n_docs_with_gold")
        .reset_index()
    )

    pred_counts = (
        per_doc.assign(has_pred=lambda x: (x["tp"] + x["fp"]) > 0)
        .groupby("label")["has_pred"]
        .sum()
        .rename("n_docs_predicted")
        .reset_index()
    )

    out = full.merge(gold_counts, on="label", how="left").merge(pred_counts, on="label", how="left")
    out["n_docs_with_gold"] = out["n_docs_with_gold"].fillna(0).astype(int)
    out["n_docs_predicted"] = out["n_docs_predicted"].fillna(0).astype(int)

    return out

def summarize_facts_breakdown(df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    """
    Devuelve tablas de desglose para facts:
    - attributes
    - relations
    - all

    Añade también frecuencia documental para facilitar lectura
    de predicados raros.
    """
    full = summarize_prefixed_metrics_with_document_frequency(df, "fact_predicate")

    attribute_labels = set(FACT_ATTRIBUTE_PREDICATES)
    relation_labels = set(FACT_RELATION_PREDICATES)

    attributes_df = full[full["label"].isin(attribute_labels)].reset_index(drop=True)
    relations_df = full[full["label"].isin(relation_labels)].reset_index(drop=True)

    return {
        "attributes": attributes_df,
        "relations": relations_df,
        "all": full,
    }

def summarize_events_breakdown(df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    """
    Devuelve:
    - attributes: métricas globales por atributo de evento
    - roles: métricas globales por rol de evento
    """
    return {
        "attributes": summarize_prefixed_metrics(df, "event_attr"),
        "roles": summarize_prefixed_metrics(df, "event_role"),
    }

def summarize_entities_breakdown(df: pd.DataFrame) -> pd.DataFrame:
    return summarize_prefixed_metrics(df, "entity_type")

def summarize_eventsonly_by_column(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    """
    Resume métricas de eventsonly agrupadas por tipo o subtipo del gold.
    """
    if group_col not in df.columns:
        raise ValueError(f"No existe la columna {group_col}")

    rows = []
    grouped = df.dropna(subset=[group_col]).groupby(group_col, dropna=True)

    for group_value, g in grouped:
        exact_tp = int(g.get("events_exact_tp", pd.Series(dtype=float)).fillna(0).sum())
        exact_fp = int(g.get("events_exact_fp", pd.Series(dtype=float)).fillna(0).sum())
        exact_fn = int(g.get("events_exact_fn", pd.Series(dtype=float)).fillna(0).sum())
        exact_p, exact_r, exact_f1 = prf(exact_tp, exact_fp, exact_fn)

        header_tp = int(g.get("events_header_tp", pd.Series(dtype=float)).fillna(0).sum())
        header_fp = int(g.get("events_header_fp", pd.Series(dtype=float)).fillna(0).sum())
        header_fn = int(g.get("events_header_fn", pd.Series(dtype=float)).fillna(0).sum())
        header_p, header_r, header_f1 = prf(header_tp, header_fp, header_fn)

        attr_tp, attr_fp, attr_fn = aggregate_prefix_totals(g, "event_attr")
        attr_p, attr_r, attr_f1 = prf(attr_tp, attr_fp, attr_fn)

        rel_tp, rel_fp, rel_fn = aggregate_prefix_totals(g, "event_role")
        rel_p, rel_r, rel_f1 = prf(rel_tp, rel_fp, rel_fn)

        rows.append({
            group_col: group_value,
            "n_docs": len(g),

            "events_exact_tp": exact_tp,
            "events_exact_fp": exact_fp,
            "events_exact_fn": exact_fn,
            "events_exact_precision": exact_p,
            "events_exact_recall": exact_r,
            "events_exact_f1": exact_f1,

            "events_header_tp": header_tp,
            "events_header_fp": header_fp,
            "events_header_fn": header_fn,
            "events_header_precision": header_p,
            "events_header_recall": header_r,
            "events_header_f1": header_f1,

            "event_attrs_tp": attr_tp,
            "event_attrs_fp": attr_fp,
            "event_attrs_fn": attr_fn,
            "event_attrs_precision": attr_p,
            "event_attrs_recall": attr_r,
            "event_attrs_f1": attr_f1,

            "event_rels_tp": rel_tp,
            "event_rels_fp": rel_fp,
            "event_rels_fn": rel_fn,
            "event_rels_precision": rel_p,
            "event_rels_recall": rel_r,
            "event_rels_f1": rel_f1,
        })

    return pd.DataFrame(rows).sort_values(by=[group_col]).reset_index(drop=True)

def summarize_task_by_group(df: pd.DataFrame, task_name: str, group_col: str) -> pd.DataFrame:
    """
    Resume métricas principales de una tarea agrupadas por una columna,
    por ejemplo:
    - gold_event_type_combo
    - gold_event_subtype_combo
    """
    if group_col not in df.columns:
        raise ValueError(f"No existe la columna {group_col}")

    rows = []
    grouped = df[df[group_col].fillna("") != ""].groupby(group_col, dropna=True)

    for group_value, g in grouped:
        if task_name == "entityonly":
            tp = int(g.get("entities_tp", pd.Series(dtype=float)).fillna(0).sum())
            fp = int(g.get("entities_fp", pd.Series(dtype=float)).fillna(0).sum())
            fn = int(g.get("entities_fn", pd.Series(dtype=float)).fillna(0).sum())
            p, r, f1 = prf(tp, fp, fn)

            rows.append({
                group_col: group_value,
                "n_docs": len(g),
                "entities_tp": tp,
                "entities_fp": fp,
                "entities_fn": fn,
                "entities_precision": p,
                "entities_recall": r,
                "entities_f1": f1,
            })

        elif task_name == "factsonly":
            facts_tp = int(g.get("facts_tp", pd.Series(dtype=float)).fillna(0).sum())
            facts_fp = int(g.get("facts_fp", pd.Series(dtype=float)).fillna(0).sum())
            facts_fn = int(g.get("facts_fn", pd.Series(dtype=float)).fillna(0).sum())
            facts_p, facts_r, facts_f1 = prf(facts_tp, facts_fp, facts_fn)

            attr_tp, attr_fp, attr_fn = aggregate_selected_labels(
                g, "fact_predicate", FACT_ATTRIBUTE_PREDICATES
            )
            attr_p, attr_r, attr_f1 = prf(attr_tp, attr_fp, attr_fn)

            rel_tp, rel_fp, rel_fn = aggregate_selected_labels(
                g, "fact_predicate", FACT_RELATION_PREDICATES
            )
            rel_p, rel_r, rel_f1 = prf(rel_tp, rel_fp, rel_fn)

            rows.append({
                group_col: group_value,
                "n_docs": len(g),

                "facts_tp": facts_tp,
                "facts_fp": facts_fp,
                "facts_fn": facts_fn,
                "facts_precision": facts_p,
                "facts_recall": facts_r,
                "facts_f1": facts_f1,

                "fact_attrs_tp": attr_tp,
                "fact_attrs_fp": attr_fp,
                "fact_attrs_fn": attr_fn,
                "fact_attrs_precision": attr_p,
                "fact_attrs_recall": attr_r,
                "fact_attrs_f1": attr_f1,

                "fact_rels_tp": rel_tp,
                "fact_rels_fp": rel_fp,
                "fact_rels_fn": rel_fn,
                "fact_rels_precision": rel_p,
                "fact_rels_recall": rel_r,
                "fact_rels_f1": rel_f1,
            })

        elif task_name == "eventsonly":
            exact_tp = int(g.get("events_exact_tp", pd.Series(dtype=float)).fillna(0).sum())
            exact_fp = int(g.get("events_exact_fp", pd.Series(dtype=float)).fillna(0).sum())
            exact_fn = int(g.get("events_exact_fn", pd.Series(dtype=float)).fillna(0).sum())
            exact_p, exact_r, exact_f1 = prf(exact_tp, exact_fp, exact_fn)

            header_tp = int(g.get("events_header_tp", pd.Series(dtype=float)).fillna(0).sum())
            header_fp = int(g.get("events_header_fp", pd.Series(dtype=float)).fillna(0).sum())
            header_fn = int(g.get("events_header_fn", pd.Series(dtype=float)).fillna(0).sum())
            header_p, header_r, header_f1 = prf(header_tp, header_fp, header_fn)

            attr_tp, attr_fp, attr_fn = aggregate_prefix_totals(g, "event_attr")
            attr_p, attr_r, attr_f1 = prf(attr_tp, attr_fp, attr_fn)

            rel_tp, rel_fp, rel_fn = aggregate_prefix_totals(g, "event_role")
            rel_p, rel_r, rel_f1 = prf(rel_tp, rel_fp, rel_fn)

            rows.append({
                group_col: group_value,
                "n_docs": len(g),

                "events_exact_tp": exact_tp,
                "events_exact_fp": exact_fp,
                "events_exact_fn": exact_fn,
                "events_exact_precision": exact_p,
                "events_exact_recall": exact_r,
                "events_exact_f1": exact_f1,

                "events_header_tp": header_tp,
                "events_header_fp": header_fp,
                "events_header_fn": header_fn,
                "events_header_precision": header_p,
                "events_header_recall": header_r,
                "events_header_f1": header_f1,

                "event_attrs_tp": attr_tp,
                "event_attrs_fp": attr_fp,
                "event_attrs_fn": attr_fn,
                "event_attrs_precision": attr_p,
                "event_attrs_recall": attr_r,
                "event_attrs_f1": attr_f1,

                "event_rels_tp": rel_tp,
                "event_rels_fp": rel_fp,
                "event_rels_fn": rel_fn,
                "event_rels_precision": rel_p,
                "event_rels_recall": rel_r,
                "event_rels_f1": rel_f1,
            })

        else:
            raise ValueError(f"Tarea no soportada: {task_name}")

    return pd.DataFrame(rows).sort_values(by=[group_col]).reset_index(drop=True)

def summarize_facts_grouped_with_macro(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    """
    Resume factsonly por una columna de agrupación (por ejemplo:
    gold_event_type_combo o gold_event_subtype_combo), incluyendo:

    - métricas micro ya agregadas
    - métricas macro calculadas sobre predicados dentro de cada grupo
    """
    rows = []

    if df.empty or group_col not in df.columns:
        return pd.DataFrame()

    for group_value, subdf in df.groupby(group_col, dropna=False):
        main = summarize_facts_main_with_macro(subdf)

        # helper para leer la fila correspondiente
        def get_metric_row(metric_name: str) -> Dict[str, Any]:
            row = main[main["metric"] == metric_name]
            if row.empty:
                return {}
            return row.iloc[0].to_dict()

        facts_row = get_metric_row("facts")
        attrs_row = get_metric_row("fact_attrs")
        rels_row = get_metric_row("fact_rels")

        rows.append({
            group_col: group_value,
            "n_docs": int(subdf["document_id"].nunique()),

            # facts micro
            "facts_tp": facts_row.get("tp", 0),
            "facts_fp": facts_row.get("fp", 0),
            "facts_fn": facts_row.get("fn", 0),
            "facts_precision": facts_row.get("precision", 0.0),
            "facts_recall": facts_row.get("recall", 0.0),
            "facts_f1": facts_row.get("f1", 0.0),

            # facts macro
            "facts_macro_precision": facts_row.get("macro_precision", 0.0),
            "facts_macro_recall": facts_row.get("macro_recall", 0.0),
            "facts_macro_f1": facts_row.get("macro_f1", 0.0),
            "facts_delta_f1_micro_minus_macro": (
                facts_row.get("f1", 0.0) - facts_row.get("macro_f1", 0.0)
            ),

            # attrs micro
            "fact_attrs_tp": attrs_row.get("tp", 0),
            "fact_attrs_fp": attrs_row.get("fp", 0),
            "fact_attrs_fn": attrs_row.get("fn", 0),
            "fact_attrs_precision": attrs_row.get("precision", 0.0),
            "fact_attrs_recall": attrs_row.get("recall", 0.0),
            "fact_attrs_f1": attrs_row.get("f1", 0.0),

            # attrs macro
            "fact_attrs_macro_precision": attrs_row.get("macro_precision", 0.0),
            "fact_attrs_macro_recall": attrs_row.get("macro_recall", 0.0),
            "fact_attrs_macro_f1": attrs_row.get("macro_f1", 0.0),
            "fact_attrs_delta_f1_micro_minus_macro": (
                attrs_row.get("f1", 0.0) - attrs_row.get("macro_f1", 0.0)
            ),

            # rels micro
            "fact_rels_tp": rels_row.get("tp", 0),
            "fact_rels_fp": rels_row.get("fp", 0),
            "fact_rels_fn": rels_row.get("fn", 0),
            "fact_rels_precision": rels_row.get("precision", 0.0),
            "fact_rels_recall": rels_row.get("recall", 0.0),
            "fact_rels_f1": rels_row.get("f1", 0.0),

            # rels macro
            "fact_rels_macro_precision": rels_row.get("macro_precision", 0.0),
            "fact_rels_macro_recall": rels_row.get("macro_recall", 0.0),
            "fact_rels_macro_f1": rels_row.get("macro_f1", 0.0),
            "fact_rels_delta_f1_micro_minus_macro": (
                rels_row.get("f1", 0.0) - rels_row.get("macro_f1", 0.0)
            ),
        })

    out = pd.DataFrame(rows)

    if out.empty:
        return out

    return out.sort_values(
        by=["n_docs", group_col],
        ascending=[False, True]
    ).reset_index(drop=True)

def summarize_events_grouped_with_macro(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    """
    Resume eventsonly por una columna de agrupación
    (por ejemplo gold_event_type_combo o gold_event_subtype_combo), incluyendo:

    - métricas micro ya agregadas
    - métricas macro para atributos y roles
    """
    rows = []

    if df.empty or group_col not in df.columns:
        return pd.DataFrame()

    for group_value, subdf in df.groupby(group_col, dropna=False):
        main = summarize_events_main_with_macro(subdf)

        def get_metric_row(metric_name: str) -> Dict[str, Any]:
            row = main[main["metric"] == metric_name]
            if row.empty:
                return {}
            return row.iloc[0].to_dict()

        exact_row = get_metric_row("events_exact")
        header_row = get_metric_row("events_header")
        attrs_row = get_metric_row("event_attrs")
        roles_row = get_metric_row("event_rels")

        rows.append({
            group_col: group_value,
            "n_docs": int(subdf["document_id"].nunique()),

            # exact micro
            "events_exact_tp": exact_row.get("tp", 0),
            "events_exact_fp": exact_row.get("fp", 0),
            "events_exact_fn": exact_row.get("fn", 0),
            "events_exact_precision": exact_row.get("precision", 0.0),
            "events_exact_recall": exact_row.get("recall", 0.0),
            "events_exact_f1": exact_row.get("f1", 0.0),

            # header micro
            "events_header_tp": header_row.get("tp", 0),
            "events_header_fp": header_row.get("fp", 0),
            "events_header_fn": header_row.get("fn", 0),
            "events_header_precision": header_row.get("precision", 0.0),
            "events_header_recall": header_row.get("recall", 0.0),
            "events_header_f1": header_row.get("f1", 0.0),

            # attrs micro
            "event_attrs_tp": attrs_row.get("tp", 0),
            "event_attrs_fp": attrs_row.get("fp", 0),
            "event_attrs_fn": attrs_row.get("fn", 0),
            "event_attrs_precision": attrs_row.get("precision", 0.0),
            "event_attrs_recall": attrs_row.get("recall", 0.0),
            "event_attrs_f1": attrs_row.get("f1", 0.0),

            # attrs macro
            "event_attrs_macro_precision": attrs_row.get("macro_precision", 0.0),
            "event_attrs_macro_recall": attrs_row.get("macro_recall", 0.0),
            "event_attrs_macro_f1": attrs_row.get("macro_f1", 0.0),
            "event_attrs_delta_f1_micro_minus_macro": (
                attrs_row.get("f1", 0.0) - attrs_row.get("macro_f1", 0.0)
            ),

            # roles micro
            "event_rels_tp": roles_row.get("tp", 0),
            "event_rels_fp": roles_row.get("fp", 0),
            "event_rels_fn": roles_row.get("fn", 0),
            "event_rels_precision": roles_row.get("precision", 0.0),
            "event_rels_recall": roles_row.get("recall", 0.0),
            "event_rels_f1": roles_row.get("f1", 0.0),

            # roles macro
            "event_rels_macro_precision": roles_row.get("macro_precision", 0.0),
            "event_rels_macro_recall": roles_row.get("macro_recall", 0.0),
            "event_rels_macro_f1": roles_row.get("macro_f1", 0.0),
            "event_rels_delta_f1_micro_minus_macro": (
                roles_row.get("f1", 0.0) - roles_row.get("macro_f1", 0.0)
            ),
        })

    out = pd.DataFrame(rows)
    if out.empty:
        return out

    return out.sort_values(
        by=["n_docs", group_col],
        ascending=[False, True]
    ).reset_index(drop=True)

def summarize_by_event_type(df: pd.DataFrame, task_name: str) -> pd.DataFrame:
    if df.empty or "gold_event_type_combo" not in df.columns:
        return pd.DataFrame()

    # Caso especial: factsonly con micro + macro
    if task_name == "factsonly":
        return summarize_facts_grouped_with_macro(df, "gold_event_type_combo")

    if task_name == "eventsonly":
        return summarize_events_grouped_with_macro(df, "gold_event_type_combo")

    # comportamiento original para el resto
    rows = []
    for value, subdf in df.groupby("gold_event_type_combo", dropna=False):
        main = summarize_main_metrics(subdf, task_name)

        row = {
            "gold_event_type_combo": value,
            "n_docs": int(subdf["document_id"].nunique()),
        }

        for _, metric_row in main.iterrows():
            metric_name = metric_row["metric"]
            row[f"{metric_name}_tp"] = metric_row["tp"]
            row[f"{metric_name}_fp"] = metric_row["fp"]
            row[f"{metric_name}_fn"] = metric_row["fn"]
            row[f"{metric_name}_precision"] = metric_row["precision"]
            row[f"{metric_name}_recall"] = metric_row["recall"]
            row[f"{metric_name}_f1"] = metric_row["f1"]

        rows.append(row)

    out = pd.DataFrame(rows)
    if out.empty:
        return out

    return out.sort_values(
        by=["n_docs", "gold_event_type_combo"],
        ascending=[False, True]
    ).reset_index(drop=True)


def summarize_by_event_subtype(df: pd.DataFrame, task_name: str) -> pd.DataFrame:
    if df.empty or "gold_event_subtype_combo" not in df.columns:
        return pd.DataFrame()

    # Caso especial: factsonly con micro + macro
    if task_name == "factsonly":
        return summarize_facts_grouped_with_macro(df, "gold_event_subtype_combo")

    if task_name == "eventsonly":
        return summarize_events_grouped_with_macro(df, "gold_event_subtype_combo")


    # comportamiento original para el resto
    rows = []
    for value, subdf in df.groupby("gold_event_subtype_combo", dropna=False):
        main = summarize_main_metrics(subdf, task_name)

        row = {
            "gold_event_subtype_combo": value,
            "n_docs": int(subdf["document_id"].nunique()),
        }

        for _, metric_row in main.iterrows():
            metric_name = metric_row["metric"]
            row[f"{metric_name}_tp"] = metric_row["tp"]
            row[f"{metric_name}_fp"] = metric_row["fp"]
            row[f"{metric_name}_fn"] = metric_row["fn"]
            row[f"{metric_name}_precision"] = metric_row["precision"]
            row[f"{metric_name}_recall"] = metric_row["recall"]
            row[f"{metric_name}_f1"] = metric_row["f1"]

        rows.append(row)

    out = pd.DataFrame(rows)
    if out.empty:
        return out

    return out.sort_values(
        by=["n_docs", "gold_event_subtype_combo"],
        ascending=[False, True]
    ).reset_index(drop=True)


def display_and_save(df, name: str, prefix: str, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)

    # Mostrar
    display(df)

    # Guardar
    df.to_csv(out_dir / f"{prefix}__{name}.csv", index=False)
    df.to_excel(out_dir / f"{prefix}__{name}.xlsx", index=False)

    with open(out_dir / f"{prefix}__{name}.md", "w", encoding="utf-8") as f:
        f.write(df.to_markdown(index=False))

def display_and_save_summary(summary: dict, prefix: str, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)

    for key, value in summary.items():
        if isinstance(value, pd.DataFrame):
            display_and_save(value, key, prefix, out_dir)

def export_entityonly_summary(
    summary: Dict[str, pd.DataFrame],
    prefix: str,
    out_dir: Path = OUT_DIR / "paper_tables",
):
    prefix_parts = prefix.split("__")
    org_out_dir = out_dir / prefix_parts[0] / prefix_parts[1] / prefix_parts[2]
    display_and_save(summary["main"], "main", prefix, org_out_dir)
    display_and_save(summary["breakdown"], "breakdown", prefix, org_out_dir)


def export_factsonly_summary(
    summary: Dict[str, pd.DataFrame],
    prefix: str,
    out_dir: Path = OUT_DIR / "paper_tables",
):
    prefix_parts = prefix.split("__")
    org_out_dir = out_dir / prefix_parts[0] / prefix_parts[1] / prefix_parts[2]
    display_and_save(summary["main"], "main", prefix, org_out_dir)
    display_and_save(summary["by_type"], "by_type", prefix, org_out_dir)
    display_and_save(summary["by_subtype"], "by_subtype", prefix, org_out_dir)
    display_and_save(summary["attributes"], "attributes", prefix, org_out_dir)
    display_and_save(summary["relations"], "relations", prefix, org_out_dir)
    display_and_save(summary["all_facts"], "all_facts", prefix, org_out_dir)


def export_eventsonly_summary(
    summary: Dict[str, pd.DataFrame],
    prefix: str,
    out_dir: Path = OUT_DIR / "paper_tables",
):
    prefix_parts = prefix.split("__")
    org_out_dir = out_dir / prefix_parts[0] / prefix_parts[1] / prefix_parts[2]
    display_and_save(summary["main"], "main", prefix, org_out_dir)
    display_and_save(summary["by_type"], "by_type", prefix, org_out_dir)
    display_and_save(summary["by_subtype"], "by_subtype", prefix, org_out_dir)
    display_and_save(summary["attributes"], "attributes", prefix, org_out_dir)
    display_and_save(summary["roles"], "roles", prefix, org_out_dir)


# Dev_tune execution

This section runs the development-tuning experiments.

The dev_tune split is used to compare prompting strategies, model backends, and
task configurations before running the final test evaluation. Experiments are
organized by provider, task, prompt variant, and entity mode.

For each run, the notebook displays document-level results and immediate
aggregate summaries, allowing prompt and configuration choices to be inspected
before moving to the test split.

## Gemini

### Entities (annotated)

In [ ]:
split = load_split()

df = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["dev_tune"],
    prompt_variant="baseline",
    entity_mode="annotated",
    save_prefix="entityonly__gemini_native__baseline__annotated__dev_tune",
)

display(df)
display(summarize_main_metrics(df, "entityonly"))
display(summarize_entities_breakdown(df))

In [ ]:
split = load_split()

df = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["dev_tune"],
    prompt_variant="oneshot",
    entity_mode="annotated",
    save_prefix="entityonly__gemini_native__oneshot__annotated__dev_tune",
)

display(df)
display(summarize_main_metrics(df, "entityonly"))
display(summarize_entities_breakdown(df))

In [ ]:
split = load_split()

df = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["dev_tune"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    save_prefix="entityonly__gemini_native__fewshot__annotated__dev_tune",
)

display(df)
display(summarize_main_metrics(df, "entityonly"))
display(summarize_entities_breakdown(df))

### Entities (all)

In [ ]:
df = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["dev_tune"],
    prompt_variant="baseline",
    entity_mode="all",
    save_prefix="entityonly__gemini_native__baseline__all__dev_tune",
)

display(df)
display(summarize_main_metrics(df, "entityonly"))
display(summarize_entities_breakdown(df))

In [ ]:
df = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["dev_tune"],
    prompt_variant="oneshot",
    entity_mode="all",
    save_prefix="entityonly__gemini_native__oneshot__all__dev_tune",
)

display(df)
display(summarize_main_metrics(df, "entityonly"))
display(summarize_entities_breakdown(df))

In [ ]:
df = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["dev_tune"],
    prompt_variant="fewshot",
    entity_mode="all",
    save_prefix="entityonly__gemini_native__fewshot__all__dev_tune",
)

display(df)
display(summarize_main_metrics(df, "entityonly"))
display(summarize_entities_breakdown(df))

### Facts

#### Baseline

In [ ]:
split = load_split()

df = run_experiment(
    task_name="factsonly",
    backend_name="gemini_native",
    doc_ids=split["dev_tune"],
    prompt_variant="baseline",
    example_doc_ids=None,
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gemini_native__baseline__dev_tune",
)

display(df)
display(summarize_main_metrics(df, "factsonly"))
display(summarize_by_event_type(df, "factsonly"))
display(summarize_by_event_subtype(df, "factsonly"))
display(summarize_facts_breakdown(df)["attributes"])
display(summarize_facts_breakdown(df)["relations"])

#### One-shot

In [ ]:
split = load_split()

df = run_experiment(
    task_name="factsonly",
    backend_name="gemini_native",
    doc_ids=split["dev_tune"],
    prompt_variant="oneshot",
    example_doc_ids=None,
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gemini_native__oneshot__dev_tune",
)

display(df)
display(summarize_main_metrics(df, "factsonly"))
display(summarize_by_event_type(df, "factsonly"))
display(summarize_by_event_subtype(df, "factsonly"))
display(summarize_facts_breakdown(df)["attributes"])
display(summarize_facts_breakdown(df)["relations"])

#### Few-shot

In [ ]:
split = load_split()

df = run_experiment(
    task_name="factsonly",
    backend_name="gemini_native",
    doc_ids=split["dev_tune"],
    prompt_variant="fewshot",
    example_doc_ids=None,
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gemini_native__fewshot__dev_tune",
)

display(df)
display(summarize_main_metrics(df, "factsonly"))
display(summarize_by_event_type(df, "factsonly"))
display(summarize_by_event_subtype(df, "factsonly"))
display(summarize_facts_breakdown(df)["attributes"])
display(summarize_facts_breakdown(df)["relations"])

### Events

#### Baseline

In [ ]:
split = load_split()

df = run_experiment(
    task_name="eventsonly",
    backend_name="gemini_native",
    doc_ids=split["dev_tune"],
    prompt_variant="baseline",
    example_doc_ids=None,
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gemini_native__baseline__dev_tune",
)

display(df)
display(summarize_main_metrics(df, "eventsonly"))
display(summarize_by_event_type(df, "eventsonly"))
display(summarize_by_event_subtype(df, "eventsonly"))
display(summarize_events_breakdown(df)["attributes"])
display(summarize_events_breakdown(df)["roles"])

#### One-shot

In [ ]:
split = load_split()

df = run_experiment(
    task_name="eventsonly",
    backend_name="gemini_native",
    doc_ids=split["dev_tune"],
    prompt_variant="oneshot",
    example_doc_ids=None,
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gemini_native__oneshot__dev_tune",
)

display(df)
display(summarize_main_metrics(df, "eventsonly"))
display(summarize_by_event_type(df, "eventsonly"))
display(summarize_by_event_subtype(df, "eventsonly"))
display(summarize_events_breakdown(df)["attributes"])
display(summarize_events_breakdown(df)["roles"])

#### Few-shot

In [ ]:
split = load_split()

df = run_experiment(
    task_name="eventsonly",
    backend_name="gemini_native",
    doc_ids=split["dev_tune"],
    prompt_variant="fewshot",
    example_doc_ids=None,
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gemini_native__fewshot__dev_tune",
)

display(df)
display(summarize_main_metrics(df, "eventsonly"))
display(summarize_by_event_type(df, "eventsonly"))
display(summarize_by_event_subtype(df, "eventsonly"))
display(summarize_events_breakdown(df)["attributes"])
display(summarize_events_breakdown(df)["roles"])

## Llama 17B

### Entities (annotated)

In [ ]:
split = load_split()

df = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["dev_tune"],
    prompt_variant="baseline",
    entity_mode="annotated",
    save_prefix="entityonly__llama17__baseline__annotated__dev_tune",
)

display(df)
display(summarize_main_metrics(df, "entityonly"))
display(summarize_entities_breakdown(df))

In [ ]:
split = load_split()

df = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["dev_tune"],
    prompt_variant="oneshot",
    entity_mode="annotated",
    save_prefix="entityonly__llama17__oneshot__annotated__dev_tune",
)

display(df)
display(summarize_main_metrics(df, "entityonly"))
display(summarize_entities_breakdown(df))

In [ ]:
split = load_split()

df = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["dev_tune"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    save_prefix="entityonly__llama17__fewshot__annotated__dev_tune",
)

display(df)
display(summarize_main_metrics(df, "entityonly"))
display(summarize_entities_breakdown(df))

### Entities (all)

In [ ]:
df = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["dev_tune"],
    prompt_variant="baseline",
    entity_mode="all",
    save_prefix="entityonly__llama17__baseline__all__dev_tune",
)

display(df)
display(summarize_main_metrics(df, "entityonly"))
display(summarize_entities_breakdown(df))

In [ ]:
df = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["dev_tune"],
    prompt_variant="oneshot",
    entity_mode="all",
    save_prefix="entityonly__llama17__oneshot__all__dev_tune",
)

display(df)
display(summarize_main_metrics(df, "entityonly"))
display(summarize_entities_breakdown(df))

In [ ]:
df = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["dev_tune"],
    prompt_variant="fewshot",
    entity_mode="all",
    save_prefix="entityonly__llama17__fewshot__all__dev_tune",
)

display(df)
display(summarize_main_metrics(df, "entityonly"))
display(summarize_entities_breakdown(df))

### Facts

#### Baseline

In [ ]:
split = load_split()

df = run_experiment(
    task_name="factsonly",
    backend_name="groq_openai",
    doc_ids=split["dev_tune"],
    prompt_variant="baseline",
    example_doc_ids=None,
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__llama17__baseline__dev_tune",
)

display(df)
display(summarize_main_metrics(df, "factsonly"))
display(summarize_by_event_type(df, "factsonly"))
display(summarize_by_event_subtype(df, "factsonly"))
display(summarize_facts_breakdown(df)["attributes"])
display(summarize_facts_breakdown(df)["relations"])

#### One-shot

In [ ]:
split = load_split()

df = run_experiment(
    task_name="factsonly",
    backend_name="groq_openai",
    doc_ids=split["dev_tune"],
    prompt_variant="oneshot",
    example_doc_ids=None,
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__llama17__oneshot__dev_tune",
)

display(df)
display(summarize_main_metrics(df, "factsonly"))
display(summarize_by_event_type(df, "factsonly"))
display(summarize_by_event_subtype(df, "factsonly"))
display(summarize_facts_breakdown(df)["attributes"])
display(summarize_facts_breakdown(df)["relations"])

#### Few-shot

In [ ]:
split = load_split()

df = run_experiment(
    task_name="factsonly",
    backend_name="groq_openai",
    doc_ids=split["dev_tune"],
    prompt_variant="fewshot",
    example_doc_ids=None,
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__llama17__fewshot__dev_tune",
)

display(df)
display(summarize_main_metrics(df, "factsonly"))
display(summarize_by_event_type(df, "factsonly"))
display(summarize_by_event_subtype(df, "factsonly"))
display(summarize_facts_breakdown(df)["attributes"])
display(summarize_facts_breakdown(df)["relations"])

### Events

#### Baseline

In [ ]:
split = load_split()

df = run_experiment(
    task_name="eventsonly",
    backend_name="groq_openai",
    doc_ids=split["dev_tune"],
    prompt_variant="baseline",
    example_doc_ids=None,
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__llama17__baseline__dev_tune",
)

display(df)
display(summarize_main_metrics(df, "eventsonly"))
display(summarize_by_event_type(df, "eventsonly"))
display(summarize_by_event_subtype(df, "eventsonly"))
display(summarize_events_breakdown(df)["attributes"])
display(summarize_events_breakdown(df)["roles"])

#### One-shot

In [ ]:
split = load_split()

df = run_experiment(
    task_name="eventsonly",
    backend_name="groq_openai",
    doc_ids=split["dev_tune"],
    prompt_variant="oneshot",
    example_doc_ids=None,
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__llama17__oneshot__dev_tune",
)

display(df)
display(summarize_main_metrics(df, "eventsonly"))
display(summarize_by_event_type(df, "eventsonly"))
display(summarize_by_event_subtype(df, "eventsonly"))
display(summarize_events_breakdown(df)["attributes"])
display(summarize_events_breakdown(df)["roles"])

#### Few-shot

In [ ]:
split = load_split()

df = run_experiment(
    task_name="eventsonly",
    backend_name="groq_openai",
    doc_ids=split["dev_tune"],
    prompt_variant="fewshot",
    example_doc_ids=None,
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__llama17__fewshot__dev_tune",
)

display(df)
display(summarize_main_metrics(df, "eventsonly"))
display(summarize_by_event_type(df, "eventsonly"))
display(summarize_by_event_subtype(df, "eventsonly"))
display(summarize_events_breakdown(df)["attributes"])
display(summarize_events_breakdown(df)["roles"])

## Mistral Large 2411

### Entities (annotated)

In [ ]:
split = load_split()

df = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["dev_tune"],
    prompt_variant="baseline",
    entity_mode="annotated",
    save_prefix="entityonly__mistrallarge__baseline__annotated__dev_tune",
)

display(df)
display(summarize_main_metrics(df, "entityonly"))
display(summarize_entities_breakdown(df))

In [ ]:
split = load_split()

df = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["dev_tune"],
    prompt_variant="oneshot",
    entity_mode="annotated",
    save_prefix="entityonly__mistrallarge__oneshot__annotated__dev_tune",
)

display(df)
display(summarize_main_metrics(df, "entityonly"))
display(summarize_entities_breakdown(df))

In [ ]:
split = load_split()

df = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["dev_tune"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    save_prefix="entityonly__mistrallarge__fewshot__annotated__dev_tune",
)

display(df)
display(summarize_main_metrics(df, "entityonly"))
display(summarize_entities_breakdown(df))

### Entities (all)

In [ ]:
split = load_split()

df = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["dev_tune"],
    prompt_variant="baseline",
    entity_mode="all",
    save_prefix="entityonly__mistrallarge__baseline__all__dev_tune",
)

display(df)
display(summarize_main_metrics(df, "entityonly"))
display(summarize_entities_breakdown(df))

In [ ]:
split = load_split()

df = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["dev_tune"],
    prompt_variant="oneshot",
    entity_mode="all",
    save_prefix="entityonly__mistrallarge__oneshot__all__dev_tune",
)

display(df)
display(summarize_main_metrics(df, "entityonly"))
display(summarize_entities_breakdown(df))

In [ ]:
split = load_split()

df = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["dev_tune"],
    prompt_variant="fewshot",
    entity_mode="all",
    save_prefix="entityonly__mistrallarge__fewshot__all__dev_tune",
)

display(df)
display(summarize_main_metrics(df, "entityonly"))
display(summarize_entities_breakdown(df))

### Facts

#### Baseline

In [ ]:
split = load_split()

df = run_experiment(
    task_name="factsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["dev_tune"],
    prompt_variant="baseline",
    example_doc_ids=None,
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__mistrallarge__baseline__dev_tune",
)

display(df)
display(summarize_main_metrics(df, "factsonly"))
display(summarize_by_event_type(df, "factsonly"))
display(summarize_by_event_subtype(df, "factsonly"))
display(summarize_facts_breakdown(df)["attributes"])
display(summarize_facts_breakdown(df)["relations"])

#### One-shot

In [ ]:
split = load_split()

df = run_experiment(
    task_name="factsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["dev_tune"],
    prompt_variant="oneshot",
    example_doc_ids=None,
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__mistrallarge__oneshot__dev_tune",
)

display(df)
display(summarize_main_metrics(df, "factsonly"))
display(summarize_by_event_type(df, "factsonly"))
display(summarize_by_event_subtype(df, "factsonly"))
display(summarize_facts_breakdown(df)["attributes"])
display(summarize_facts_breakdown(df)["relations"])

#### Few-shot

In [ ]:
split = load_split()

df = run_experiment(
    task_name="factsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["dev_tune"],
    prompt_variant="fewshot",
    example_doc_ids=None,
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__mistrallarge__fewshot__dev_tune",
)

display(df)
display(summarize_main_metrics(df, "factsonly"))
display(summarize_by_event_type(df, "factsonly"))
display(summarize_by_event_subtype(df, "factsonly"))
display(summarize_facts_breakdown(df)["attributes"])
display(summarize_facts_breakdown(df)["relations"])

### Events

#### Baseline

In [ ]:
split = load_split()

df = run_experiment(
    task_name="eventsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["dev_tune"],
    prompt_variant="baseline",
    example_doc_ids=None,
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__mistrallarge__baseline__dev_tune",
)

display(df)
display(summarize_main_metrics(df, "eventsonly"))
display(summarize_by_event_type(df, "eventsonly"))
display(summarize_by_event_subtype(df, "eventsonly"))
display(summarize_events_breakdown(df)["attributes"])
display(summarize_events_breakdown(df)["roles"])

#### One-shot

In [ ]:
split = load_split()

df = run_experiment(
    task_name="eventsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["dev_tune"],
    prompt_variant="oneshot",
    example_doc_ids=None,
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__mistrallarge__oneshot__dev_tune",
)

display(df)
display(summarize_main_metrics(df, "eventsonly"))
display(summarize_by_event_type(df, "eventsonly"))
display(summarize_by_event_subtype(df, "eventsonly"))
display(summarize_events_breakdown(df)["attributes"])
display(summarize_events_breakdown(df)["roles"])

#### Few-shot

In [ ]:
split = load_split()

df = run_experiment(
    task_name="eventsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["dev_tune"],
    prompt_variant="fewshot",
    example_doc_ids=None,
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__mistrallarge__fewshot__dev_tune",
)

display(df)
display(summarize_main_metrics(df, "eventsonly"))
display(summarize_by_event_type(df, "eventsonly"))
display(summarize_by_event_subtype(df, "eventsonly"))
display(summarize_events_breakdown(df)["attributes"])
display(summarize_events_breakdown(df)["roles"])

## GPT 54 Nano

### Entities (annotated)

In [ ]:
split = load_split()

result = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["dev_tune"],
    prompt_variant="baseline",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__baseline__annotated__dev_tune",
    wait=True,
    poll_seconds=60,
)

df = result["df"]
display(df)
display(summarize_main_metrics(df, "entityonly"))
display(summarize_entities_breakdown(df))



In [ ]:
split = load_split()

result = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["dev_tune"],
    prompt_variant="oneshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__oneshot__annotated__dev_tune",
    wait=True,
    poll_seconds=60,
)

df = result["df"]
display(df)
display(summarize_main_metrics(df, "entityonly"))
display(summarize_entities_breakdown(df))



In [ ]:
split = load_split()

result = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["dev_tune"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__fewshot__annotated__dev_tune",
    wait=True,
    poll_seconds=60,
)

df = result["df"]
display(df)
display(summarize_main_metrics(df, "entityonly"))
display(summarize_entities_breakdown(df))



### Entities (all)

In [ ]:
split = load_split()

result = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["dev_tune"],
    prompt_variant="baseline",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__baseline__all__dev_tune",
    wait=True,
    poll_seconds=60,
)

df = result["df"]
display(df)
display(summarize_main_metrics(df, "entityonly"))
display(summarize_entities_breakdown(df))



In [ ]:
split = load_split()

result = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["dev_tune"],
    prompt_variant="oneshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__oneshot__all__dev_tune",
    wait=True,
    poll_seconds=60,
)

df = result["df"]
display(df)
display(summarize_main_metrics(df, "entityonly"))
display(summarize_entities_breakdown(df))



In [ ]:
split = load_split()

result = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["dev_tune"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__fewshot__all__dev_tune",
    wait=True,
    poll_seconds=60,
)

df = result["df"]
display(df)
display(summarize_main_metrics(df, "entityonly"))
display(summarize_entities_breakdown(df))



### Facts

#### Baseline

In [ ]:
split = load_split()

result = run_experiment_openai_batch(
    task_name="factsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["dev_tune"],
    prompt_variant="baseline",
    max_output_tokens=8192,
    save_prefix="factsonly__gpt54nano__baseline__dev_tune",
    wait=True,
    poll_seconds=60,
)

df = result["df"]
display(df)
display(summarize_main_metrics(df, "factsonly"))
display(summarize_by_event_type(df, "factsonly"))
display(summarize_by_event_subtype(df, "factsonly"))
display(summarize_facts_breakdown(df)["attributes"])
display(summarize_facts_breakdown(df)["relations"])


#### One-shot

In [ ]:
split = load_split()

result = run_experiment_openai_batch(
    task_name="factsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["dev_tune"],
    prompt_variant="oneshot",
    max_output_tokens=8192,
    save_prefix="factsonly__gpt54nano__oneshot__dev_tune",
    wait=True,
    poll_seconds=60,
)

df = result["df"]
display(df)
display(summarize_main_metrics(df, "factsonly"))
display(summarize_by_event_type(df, "factsonly"))
display(summarize_by_event_subtype(df, "factsonly"))
display(summarize_facts_breakdown(df)["attributes"])
display(summarize_facts_breakdown(df)["relations"])


#### Few-shot

In [ ]:
split = load_split()

result = run_experiment_openai_batch(
    task_name="factsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["dev_tune"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    save_prefix="factsonly__gpt54nano__fewshot__dev_tune",
    wait=True,
    poll_seconds=60,
)

df = result["df"]
display(df)
display(summarize_main_metrics(df, "factsonly"))
display(summarize_by_event_type(df, "factsonly"))
display(summarize_by_event_subtype(df, "factsonly"))
display(summarize_facts_breakdown(df)["attributes"])
display(summarize_facts_breakdown(df)["relations"])


### Events

#### Baseline

In [ ]:
split = load_split()

result = run_experiment_openai_batch(
    task_name="eventsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["dev_tune"],
    prompt_variant="baseline",
    max_output_tokens=8192,
    save_prefix="eventsonly__gpt54nano__baseline__dev_tune",
    wait=True,
    poll_seconds=60,
)

df = result["df"]
display(df)
display(summarize_main_metrics(df, "eventsonly"))
display(summarize_by_event_type(df, "eventsonly"))
display(summarize_by_event_subtype(df, "eventsonly"))
display(summarize_events_breakdown(df)["attributes"])
display(summarize_events_breakdown(df)["roles"])

#### One-shot

In [ ]:
split = load_split()

result = run_experiment_openai_batch(
    task_name="eventsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["dev_tune"],
    prompt_variant="oneshot",
    max_output_tokens=8192,
    save_prefix="eventsonly__gpt54nano__oneshot__dev_tune",
    wait=True,
    poll_seconds=60,
)

df = result["df"]
display(df)
display(summarize_main_metrics(df, "eventsonly"))
display(summarize_by_event_type(df, "eventsonly"))
display(summarize_by_event_subtype(df, "eventsonly"))
display(summarize_events_breakdown(df)["attributes"])
display(summarize_events_breakdown(df)["roles"])

#### Few-shot

In [ ]:
split = load_split()

result = run_experiment_openai_batch(
    task_name="eventsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["dev_tune"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    save_prefix="eventsonly__gpt54nano__fewshot__dev_tune",
    wait=True,
    poll_seconds=60,
)

df = result["df"]
display(df)
display(summarize_main_metrics(df, "eventsonly"))
display(summarize_by_event_type(df, "eventsonly"))
display(summarize_by_event_subtype(df, "eventsonly"))
display(summarize_events_breakdown(df)["attributes"])
display(summarize_events_breakdown(df)["roles"])

# Test execution

This section runs the final evaluation on the held-out test split.

The test experiments apply the selected task, backend, prompt variant, and model
configurations to the documents reserved for final evaluation. OpenAI runs
are executed in batches as the costs are lower.

Each batch stores prompts, raw responses, parsed outputs, validation reports,
and document-level metrics.

## Gemini

### Entities

#### Baseline (annotated)

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="baseline",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__baseline__annotated__test",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="baseline",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__baseline__annotated__test",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="baseline",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__baseline__annotated__test",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="baseline",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__baseline__annotated__test",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

#### Baseline (all)

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="baseline",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__baseline__all__test",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="baseline",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__baseline__all__test",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="baseline",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__baseline__all__test",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="baseline",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__baseline__all__test",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

#### One-shot (annotated)

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__oneshot__annotated__test",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__oneshot__annotated__test",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__oneshot__annotated__test",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__oneshot__annotated__test",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

#### One-shot (all)

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__oneshot__all__test",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__oneshot__all__test",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__oneshot__all__test",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__oneshot__all__test",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

#### Few-shot (annotated)

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__fewshot__annotated__test",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__fewshot__annotated__test",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__fewshot__annotated__test",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__fewshot__annotated__test",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

#### Few-shot (all)

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__fewshot__all__test",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__fewshot__all__test",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__fewshot__all__test",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__fewshot__all__test",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

#### Few-shot II (annotated)

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__fewshot__annotated__test_2",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__fewshot__annotated__test_2",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__fewshot__annotated__test_2",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__fewshot__annotated__test_2",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

#### Few-shot II (all)

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__fewshot__all__test_2",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__fewshot__all__test_2",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__fewshot__all__test_2",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__fewshot__all__test_2",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

#### Few-shot III (annotated)

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__fewshot__annotated__test_3",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__fewshot__annotated__test_3",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__fewshot__annotated__test_3",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__fewshot__annotated__test_3",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

#### Few-shot III (all)

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__fewshot__all__test_3",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__fewshot__all__test_3",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__fewshot__all__test_3",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="entityonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gemini_native__fewshot__all__test_3",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

### Facts

#### Baseline

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="factsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="baseline",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gemini_native__baseline__test",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "factsonly"))
display(summarize_by_event_type(df_0_30, "factsonly"))
display(summarize_by_event_subtype(df_0_30, "factsonly"))
display(summarize_facts_breakdown(df_0_30)["attributes"])
display(summarize_facts_breakdown(df_0_30)["relations"])

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="factsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="baseline",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gemini_native__baseline__test",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "factsonly"))
display(summarize_by_event_type(df_30_60, "factsonly"))
display(summarize_by_event_subtype(df_30_60, "factsonly"))
display(summarize_facts_breakdown(df_30_60)["attributes"])
display(summarize_facts_breakdown(df_30_60)["relations"])

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="factsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="baseline",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gemini_native__baseline__test",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "factsonly"))
display(summarize_by_event_type(df_60_90, "factsonly"))
display(summarize_by_event_subtype(df_60_90, "factsonly"))
display(summarize_facts_breakdown(df_60_90)["attributes"])
display(summarize_facts_breakdown(df_60_90)["relations"])

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="factsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="baseline",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gemini_native__baseline__test",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "factsonly"))
display(summarize_by_event_type(df_90_end, "factsonly"))
display(summarize_by_event_subtype(df_90_end, "factsonly"))
display(summarize_facts_breakdown(df_90_end)["attributes"])
display(summarize_facts_breakdown(df_90_end)["relations"])

#### One-shot

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="factsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gemini_native__oneshot__test",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "factsonly"))
display(summarize_by_event_type(df_0_30, "factsonly"))
display(summarize_by_event_subtype(df_0_30, "factsonly"))
display(summarize_facts_breakdown(df_0_30)["attributes"])
display(summarize_facts_breakdown(df_0_30)["relations"])

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="factsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gemini_native__oneshot__test",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "factsonly"))
display(summarize_by_event_type(df_30_60, "factsonly"))
display(summarize_by_event_subtype(df_30_60, "factsonly"))
display(summarize_facts_breakdown(df_30_60)["attributes"])
display(summarize_facts_breakdown(df_30_60)["relations"])

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="factsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gemini_native__oneshot__test",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "factsonly"))
display(summarize_by_event_type(df_60_90, "factsonly"))
display(summarize_by_event_subtype(df_60_90, "factsonly"))
display(summarize_facts_breakdown(df_60_90)["attributes"])
display(summarize_facts_breakdown(df_60_90)["relations"])

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="factsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gemini_native__oneshot__test",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "factsonly"))
display(summarize_by_event_type(df_90_end, "factsonly"))
display(summarize_by_event_subtype(df_90_end, "factsonly"))
display(summarize_facts_breakdown(df_90_end)["attributes"])
display(summarize_facts_breakdown(df_90_end)["relations"])

#### Few-shot

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="factsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gemini_native__fewshot__test",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "factsonly"))
display(summarize_by_event_type(df_0_30, "factsonly"))
display(summarize_by_event_subtype(df_0_30, "factsonly"))
display(summarize_facts_breakdown(df_0_30)["attributes"])
display(summarize_facts_breakdown(df_0_30)["relations"])

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="factsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gemini_native__fewshot__test",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "factsonly"))
display(summarize_by_event_type(df_30_60, "factsonly"))
display(summarize_by_event_subtype(df_30_60, "factsonly"))
display(summarize_facts_breakdown(df_30_60)["attributes"])
display(summarize_facts_breakdown(df_30_60)["relations"])

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="factsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gemini_native__fewshot__test",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "factsonly"))
display(summarize_by_event_type(df_60_90, "factsonly"))
display(summarize_by_event_subtype(df_60_90, "factsonly"))
display(summarize_facts_breakdown(df_60_90)["attributes"])
display(summarize_facts_breakdown(df_60_90)["relations"])

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="factsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gemini_native__fewshot__test",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "factsonly"))
display(summarize_by_event_type(df_90_end, "factsonly"))
display(summarize_by_event_subtype(df_90_end, "factsonly"))
display(summarize_facts_breakdown(df_90_end)["attributes"])
display(summarize_facts_breakdown(df_90_end)["relations"])

#### Few-shot II

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="factsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gemini_native__fewshot__test_2",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "factsonly"))
display(summarize_by_event_type(df_0_30, "factsonly"))
display(summarize_by_event_subtype(df_0_30, "factsonly"))
display(summarize_facts_breakdown(df_0_30)["attributes"])
display(summarize_facts_breakdown(df_0_30)["relations"])

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="factsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gemini_native__fewshot__test_2",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "factsonly"))
display(summarize_by_event_type(df_30_60, "factsonly"))
display(summarize_by_event_subtype(df_30_60, "factsonly"))
display(summarize_facts_breakdown(df_30_60)["attributes"])
display(summarize_facts_breakdown(df_30_60)["relations"])

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="factsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gemini_native__fewshot__test_2",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "factsonly"))
display(summarize_by_event_type(df_60_90, "factsonly"))
display(summarize_by_event_subtype(df_60_90, "factsonly"))
display(summarize_facts_breakdown(df_60_90)["attributes"])
display(summarize_facts_breakdown(df_60_90)["relations"])

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="factsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gemini_native__fewshot__test_2",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "factsonly"))
display(summarize_by_event_type(df_90_end, "factsonly"))
display(summarize_by_event_subtype(df_90_end, "factsonly"))
display(summarize_facts_breakdown(df_90_end)["attributes"])
display(summarize_facts_breakdown(df_90_end)["relations"])

#### Few-shot III

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="factsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gemini_native__fewshot__test_3",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "factsonly"))
display(summarize_by_event_type(df_0_30, "factsonly"))
display(summarize_by_event_subtype(df_0_30, "factsonly"))
display(summarize_facts_breakdown(df_0_30)["attributes"])
display(summarize_facts_breakdown(df_0_30)["relations"])

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="factsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gemini_native__fewshot__test_3",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "factsonly"))
display(summarize_by_event_type(df_30_60, "factsonly"))
display(summarize_by_event_subtype(df_30_60, "factsonly"))
display(summarize_facts_breakdown(df_30_60)["attributes"])
display(summarize_facts_breakdown(df_30_60)["relations"])

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="factsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gemini_native__fewshot__test_3",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "factsonly"))
display(summarize_by_event_type(df_60_90, "factsonly"))
display(summarize_by_event_subtype(df_60_90, "factsonly"))
display(summarize_facts_breakdown(df_60_90)["attributes"])
display(summarize_facts_breakdown(df_60_90)["relations"])

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="factsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gemini_native__fewshot__test_3",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "factsonly"))
display(summarize_by_event_type(df_90_end, "factsonly"))
display(summarize_by_event_subtype(df_90_end, "factsonly"))
display(summarize_facts_breakdown(df_90_end)["attributes"])
display(summarize_facts_breakdown(df_90_end)["relations"])

### Events

#### Baseline

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="eventsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="baseline",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gemini_native__baseline__test",
    start_idx=0,
    end_idx=30,
)

display(summarize_main_metrics(df_0_30, "eventsonly"))
display(summarize_by_event_type(df_0_30, "eventsonly"))
display(summarize_by_event_subtype(df_0_30, "eventsonly"))
display(summarize_events_breakdown(df_0_30)["attributes"])
display(summarize_events_breakdown(df_0_30)["roles"])

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="eventsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="baseline",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gemini_native__baseline__test",
    start_idx=30,
    end_idx=60,
)

display(summarize_main_metrics(df_30_60, "eventsonly"))
display(summarize_by_event_type(df_30_60, "eventsonly"))
display(summarize_by_event_subtype(df_30_60, "eventsonly"))
display(summarize_events_breakdown(df_30_60)["attributes"])
display(summarize_events_breakdown(df_30_60)["roles"])

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="eventsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="baseline",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gemini_native__baseline__test",
    start_idx=60,
    end_idx=90,
)

display(summarize_main_metrics(df_60_90, "eventsonly"))
display(summarize_by_event_type(df_60_90, "eventsonly"))
display(summarize_by_event_subtype(df_60_90, "eventsonly"))
display(summarize_events_breakdown(df_60_90)["attributes"])
display(summarize_events_breakdown(df_60_90)["roles"])

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="eventsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="baseline",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gemini_native__baseline__test",
    start_idx=90,
)

display(summarize_main_metrics(df_90_end, "eventsonly"))
display(summarize_by_event_type(df_90_end, "eventsonly"))
display(summarize_by_event_subtype(df_90_end, "eventsonly"))
display(summarize_events_breakdown(df_90_end)["attributes"])
display(summarize_events_breakdown(df_90_end)["roles"])

#### One-shot

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="eventsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gemini_native__oneshot__test",
    start_idx=0,
    end_idx=30,
)

display(summarize_main_metrics(df_0_30, "eventsonly"))
display(summarize_by_event_type(df_0_30, "eventsonly"))
display(summarize_by_event_subtype(df_0_30, "eventsonly"))
display(summarize_events_breakdown(df_0_30)["attributes"])
display(summarize_events_breakdown(df_0_30)["roles"])

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="eventsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gemini_native__oneshot__test",
    start_idx=30,
    end_idx=60,
)

display(summarize_main_metrics(df_30_60, "eventsonly"))
display(summarize_by_event_type(df_30_60, "eventsonly"))
display(summarize_by_event_subtype(df_30_60, "eventsonly"))
display(summarize_events_breakdown(df_30_60)["attributes"])
display(summarize_events_breakdown(df_30_60)["roles"])

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="eventsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gemini_native__oneshot__test",
    start_idx=60,
    end_idx=90,
)

display(summarize_main_metrics(df_60_90, "eventsonly"))
display(summarize_by_event_type(df_60_90, "eventsonly"))
display(summarize_by_event_subtype(df_60_90, "eventsonly"))
display(summarize_events_breakdown(df_60_90)["attributes"])
display(summarize_events_breakdown(df_60_90)["roles"])

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="eventsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gemini_native__oneshot__test",
    start_idx=90,
)

display(summarize_main_metrics(df_90_end, "eventsonly"))
display(summarize_by_event_type(df_90_end, "eventsonly"))
display(summarize_by_event_subtype(df_90_end, "eventsonly"))
display(summarize_events_breakdown(df_90_end)["attributes"])
display(summarize_events_breakdown(df_90_end)["roles"])

#### Few-shot

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="eventsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gemini_native__fewshot__test",
    start_idx=0,
    end_idx=30,
)

display(summarize_main_metrics(df_0_30, "eventsonly"))
display(summarize_by_event_type(df_0_30, "eventsonly"))
display(summarize_by_event_subtype(df_0_30, "eventsonly"))
display(summarize_events_breakdown(df_0_30)["attributes"])
display(summarize_events_breakdown(df_0_30)["roles"])

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="eventsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gemini_native__fewshot__test",
    start_idx=30,
    end_idx=60,
)

display(summarize_main_metrics(df_30_60, "eventsonly"))
display(summarize_by_event_type(df_30_60, "eventsonly"))
display(summarize_by_event_subtype(df_30_60, "eventsonly"))
display(summarize_events_breakdown(df_30_60)["attributes"])
display(summarize_events_breakdown(df_30_60)["roles"])

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="eventsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gemini_native__fewshot__test",
    start_idx=60,
    end_idx=90,
)

display(summarize_main_metrics(df_60_90, "eventsonly"))
display(summarize_by_event_type(df_60_90, "eventsonly"))
display(summarize_by_event_subtype(df_60_90, "eventsonly"))
display(summarize_events_breakdown(df_60_90)["attributes"])
display(summarize_events_breakdown(df_60_90)["roles"])

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="eventsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gemini_native__fewshot__test",
    start_idx=90,
)

display(summarize_main_metrics(df_90_end, "eventsonly"))
display(summarize_by_event_type(df_90_end, "eventsonly"))
display(summarize_by_event_subtype(df_90_end, "eventsonly"))
display(summarize_events_breakdown(df_90_end)["attributes"])
display(summarize_events_breakdown(df_90_end)["roles"])

#### Few-shot II

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="eventsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gemini_native__fewshot__test_2",
    start_idx=0,
    end_idx=30,
)

display(summarize_main_metrics(df_0_30, "eventsonly"))
display(summarize_by_event_type(df_0_30, "eventsonly"))
display(summarize_by_event_subtype(df_0_30, "eventsonly"))
display(summarize_events_breakdown(df_0_30)["attributes"])
display(summarize_events_breakdown(df_0_30)["roles"])

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="eventsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gemini_native__fewshot__test_2",
    start_idx=30,
    end_idx=60,
)

display(summarize_main_metrics(df_30_60, "eventsonly"))
display(summarize_by_event_type(df_30_60, "eventsonly"))
display(summarize_by_event_subtype(df_30_60, "eventsonly"))
display(summarize_events_breakdown(df_30_60)["attributes"])
display(summarize_events_breakdown(df_30_60)["roles"])

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="eventsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gemini_native__fewshot__test_2",
    start_idx=60,
    end_idx=90,
)

display(summarize_main_metrics(df_60_90, "eventsonly"))
display(summarize_by_event_type(df_60_90, "eventsonly"))
display(summarize_by_event_subtype(df_60_90, "eventsonly"))
display(summarize_events_breakdown(df_60_90)["attributes"])
display(summarize_events_breakdown(df_60_90)["roles"])

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="eventsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gemini_native__fewshot__test_2",
    start_idx=90,
)

display(summarize_main_metrics(df_90_end, "eventsonly"))
display(summarize_by_event_type(df_90_end, "eventsonly"))
display(summarize_by_event_subtype(df_90_end, "eventsonly"))
display(summarize_events_breakdown(df_90_end)["attributes"])
display(summarize_events_breakdown(df_90_end)["roles"])

#### Few-shot III

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="eventsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gemini_native__fewshot__test_3",
    start_idx=0,
    end_idx=30,
)

display(summarize_main_metrics(df_0_30, "eventsonly"))
display(summarize_by_event_type(df_0_30, "eventsonly"))
display(summarize_by_event_subtype(df_0_30, "eventsonly"))
display(summarize_events_breakdown(df_0_30)["attributes"])
display(summarize_events_breakdown(df_0_30)["roles"])

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="eventsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gemini_native__fewshot__test_3",
    start_idx=30,
    end_idx=60,
)

display(summarize_main_metrics(df_30_60, "eventsonly"))
display(summarize_by_event_type(df_30_60, "eventsonly"))
display(summarize_by_event_subtype(df_30_60, "eventsonly"))
display(summarize_events_breakdown(df_30_60)["attributes"])
display(summarize_events_breakdown(df_30_60)["roles"])

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="eventsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gemini_native__fewshot__test_3",
    start_idx=60,
    end_idx=90,
)

display(summarize_main_metrics(df_60_90, "eventsonly"))
display(summarize_by_event_type(df_60_90, "eventsonly"))
display(summarize_by_event_subtype(df_60_90, "eventsonly"))
display(summarize_events_breakdown(df_60_90)["attributes"])
display(summarize_events_breakdown(df_60_90)["roles"])

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="eventsonly",
    backend_name="gemini_native",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gemini_native__fewshot__test_3",
    start_idx=90,
)

display(summarize_main_metrics(df_90_end, "eventsonly"))
display(summarize_by_event_type(df_90_end, "eventsonly"))
display(summarize_by_event_subtype(df_90_end, "eventsonly"))
display(summarize_events_breakdown(df_90_end)["attributes"])
display(summarize_events_breakdown(df_90_end)["roles"])

## Llama 17B

### Entities

#### Baseline (annotated)

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="baseline",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__baseline__annotated__test",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="baseline",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__baseline__annotated__test",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="baseline",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__baseline__annotated__test",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="baseline",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__baseline__annotated__test",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

#### Baseline (all)

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="baseline",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__baseline__all__test",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="baseline",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__baseline__all__test",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="baseline",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__baseline__all__test",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="baseline",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__baseline__all__test",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

#### One-shot (annotated)

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__oneshot__annotated__test",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__oneshot__annotated__test",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__oneshot__annotated__test",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__oneshot__annotated__test",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

#### One-shot (all)

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__oneshot__all__test",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__oneshot__all__test",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__oneshot__all__test",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__oneshot__all__test",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

#### Few-shot (annotated)

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__fewshot__annotated__test",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__fewshot__annotated__test",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__fewshot__annotated__test",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__fewshot__annotated__test",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

#### Few-shot (all)

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__fewshot__all__test",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__fewshot__all__test",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__fewshot__all__test",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__fewshot__all__test",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

#### Few-shot II (annotated)

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__fewshot__annotated__test_2",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__fewshot__annotated__test_2",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__fewshot__annotated__test_2",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__fewshot__annotated__test_2",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

#### Few-shot II (all)

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__fewshot__all__test_2",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__fewshot__all__test_2",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__fewshot__all__test_2",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__fewshot__all__test_2",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

#### Few-shot III (annotated)

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__fewshot__annotated__test_3",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__fewshot__annotated__test_3",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__fewshot__annotated__test_3",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__fewshot__annotated__test_3",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

#### Few-shot III (all)

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__fewshot__all__test_3",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__fewshot__all__test_3",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__fewshot__all__test_3",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="entityonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__llama17__fewshot__all__test_3",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

### Facts

#### Baseline

In [ ]:
split = load_split()

df_0_25 = run_experiment(
    task_name="factsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="baseline",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__llama17__baseline__test",
    start_idx=0,
    end_idx=25,
)

display(df_0_25)
display(summarize_main_metrics(df_0_25, "factsonly"))
display(summarize_by_event_type(df_0_25, "factsonly"))
display(summarize_by_event_subtype(df_0_25, "factsonly"))
display(summarize_facts_breakdown(df_0_25)["attributes"])
display(summarize_facts_breakdown(df_0_25)["relations"])

In [ ]:
split = load_split()

df_25_60 = run_experiment(
    task_name="factsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="baseline",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__llama17__baseline__test",
    start_idx=25,
    end_idx=60,
)

display(df_25_60)
display(summarize_main_metrics(df_25_60, "factsonly"))
display(summarize_by_event_type(df_25_60, "factsonly"))
display(summarize_by_event_subtype(df_25_60, "factsonly"))
display(summarize_facts_breakdown(df_25_60)["attributes"])
display(summarize_facts_breakdown(df_25_60)["relations"])

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="factsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="baseline",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__llama17__baseline__test",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "factsonly"))
display(summarize_by_event_type(df_60_90, "factsonly"))
display(summarize_by_event_subtype(df_60_90, "factsonly"))
display(summarize_facts_breakdown(df_60_90)["attributes"])
display(summarize_facts_breakdown(df_60_90)["relations"])

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="factsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="baseline",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__llama17__baseline__test",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "factsonly"))
display(summarize_by_event_type(df_90_end, "factsonly"))
display(summarize_by_event_subtype(df_90_end, "factsonly"))
display(summarize_facts_breakdown(df_90_end)["attributes"])
display(summarize_facts_breakdown(df_90_end)["relations"])

#### One-shot

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="factsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__llama17__oneshot__test",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "factsonly"))
display(summarize_by_event_type(df_0_30, "factsonly"))
display(summarize_by_event_subtype(df_0_30, "factsonly"))
display(summarize_facts_breakdown(df_0_30)["attributes"])
display(summarize_facts_breakdown(df_0_30)["relations"])

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="factsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__llama17__oneshot__test",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "factsonly"))
display(summarize_by_event_type(df_30_60, "factsonly"))
display(summarize_by_event_subtype(df_30_60, "factsonly"))
display(summarize_facts_breakdown(df_30_60)["attributes"])
display(summarize_facts_breakdown(df_30_60)["relations"])

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="factsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__llama17__oneshot__test",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "factsonly"))
display(summarize_by_event_type(df_60_90, "factsonly"))
display(summarize_by_event_subtype(df_60_90, "factsonly"))
display(summarize_facts_breakdown(df_60_90)["attributes"])
display(summarize_facts_breakdown(df_60_90)["relations"])

In [ ]:
split = load_split()

df_88_end = run_experiment( # Previous execution got rate limit at 88
    task_name="factsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__llama17__oneshot__test",
    start_idx=88,
)

display(df_88_end)
display(summarize_main_metrics(df_88_end, "factsonly"))
display(summarize_by_event_type(df_88_end, "factsonly"))
display(summarize_by_event_subtype(df_88_end, "factsonly"))
display(summarize_facts_breakdown(df_88_end)["attributes"])
display(summarize_facts_breakdown(df_88_end)["relations"])

#### Few-shot

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="factsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__llama17__fewshot__test",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "factsonly"))
display(summarize_by_event_type(df_0_30, "factsonly"))
display(summarize_by_event_subtype(df_0_30, "factsonly"))
display(summarize_facts_breakdown(df_0_30)["attributes"])
display(summarize_facts_breakdown(df_0_30)["relations"])

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="factsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__llama17__fewshot__test",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "factsonly"))
display(summarize_by_event_type(df_30_60, "factsonly"))
display(summarize_by_event_subtype(df_30_60, "factsonly"))
display(summarize_facts_breakdown(df_30_60)["attributes"])
display(summarize_facts_breakdown(df_30_60)["relations"])

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="factsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__llama17__fewshot__test",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "factsonly"))
display(summarize_by_event_type(df_60_90, "factsonly"))
display(summarize_by_event_subtype(df_60_90, "factsonly"))
display(summarize_facts_breakdown(df_60_90)["attributes"])
display(summarize_facts_breakdown(df_60_90)["relations"])

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="factsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__llama17__fewshot__test",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "factsonly"))
display(summarize_by_event_type(df_90_end, "factsonly"))
display(summarize_by_event_subtype(df_90_end, "factsonly"))
display(summarize_facts_breakdown(df_90_end)["attributes"])
display(summarize_facts_breakdown(df_90_end)["relations"])

#### Few-shot II

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="factsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__llama17__fewshot__test_2",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "factsonly"))
display(summarize_by_event_type(df_0_30, "factsonly"))
display(summarize_by_event_subtype(df_0_30, "factsonly"))
display(summarize_facts_breakdown(df_0_30)["attributes"])
display(summarize_facts_breakdown(df_0_30)["relations"])

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="factsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__llama17__fewshot__test_2",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "factsonly"))
display(summarize_by_event_type(df_30_60, "factsonly"))
display(summarize_by_event_subtype(df_30_60, "factsonly"))
display(summarize_facts_breakdown(df_30_60)["attributes"])
display(summarize_facts_breakdown(df_30_60)["relations"])

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="factsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__llama17__fewshot__test_2",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "factsonly"))
display(summarize_by_event_type(df_60_90, "factsonly"))
display(summarize_by_event_subtype(df_60_90, "factsonly"))
display(summarize_facts_breakdown(df_60_90)["attributes"])
display(summarize_facts_breakdown(df_60_90)["relations"])

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="factsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__llama17__fewshot__test_2",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "factsonly"))
display(summarize_by_event_type(df_90_end, "factsonly"))
display(summarize_by_event_subtype(df_90_end, "factsonly"))
display(summarize_facts_breakdown(df_90_end)["attributes"])
display(summarize_facts_breakdown(df_90_end)["relations"])

#### Few-shot III

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="factsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__llama17__fewshot__test_3",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "factsonly"))
display(summarize_by_event_type(df_0_30, "factsonly"))
display(summarize_by_event_subtype(df_0_30, "factsonly"))
display(summarize_facts_breakdown(df_0_30)["attributes"])
display(summarize_facts_breakdown(df_0_30)["relations"])

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="factsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__llama17__fewshot__test_3",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "factsonly"))
display(summarize_by_event_type(df_30_60, "factsonly"))
display(summarize_by_event_subtype(df_30_60, "factsonly"))
display(summarize_facts_breakdown(df_30_60)["attributes"])
display(summarize_facts_breakdown(df_30_60)["relations"])

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="factsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__llama17__fewshot__test_3",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "factsonly"))
display(summarize_by_event_type(df_60_90, "factsonly"))
display(summarize_by_event_subtype(df_60_90, "factsonly"))
display(summarize_facts_breakdown(df_60_90)["attributes"])
display(summarize_facts_breakdown(df_60_90)["relations"])

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="factsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__llama17__fewshot__test_3",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "factsonly"))
display(summarize_by_event_type(df_90_end, "factsonly"))
display(summarize_by_event_subtype(df_90_end, "factsonly"))
display(summarize_facts_breakdown(df_90_end)["attributes"])
display(summarize_facts_breakdown(df_90_end)["relations"])

### Events

#### Baseline

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="eventsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="baseline",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__llama17__baseline__test",
    start_idx=0,
    end_idx=30,
)

display(summarize_main_metrics(df_0_30, "eventsonly"))
display(summarize_by_event_type(df_0_30, "eventsonly"))
display(summarize_by_event_subtype(df_0_30, "eventsonly"))
display(summarize_events_breakdown(df_0_30)["attributes"])
display(summarize_events_breakdown(df_0_30)["roles"])

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="eventsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="baseline",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__llama17__baseline__test",
    start_idx=30,
    end_idx=60,
)

display(summarize_main_metrics(df_30_60, "eventsonly"))
display(summarize_by_event_type(df_30_60, "eventsonly"))
display(summarize_by_event_subtype(df_30_60, "eventsonly"))
display(summarize_events_breakdown(df_30_60)["attributes"])
display(summarize_events_breakdown(df_30_60)["roles"])

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="eventsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="baseline",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__llama17__baseline__test",
    start_idx=60,
    end_idx=90,
)

display(summarize_main_metrics(df_60_90, "eventsonly"))
display(summarize_by_event_type(df_60_90, "eventsonly"))
display(summarize_by_event_subtype(df_60_90, "eventsonly"))
display(summarize_events_breakdown(df_60_90)["attributes"])
display(summarize_events_breakdown(df_60_90)["roles"])

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="eventsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="baseline",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__llama17__baseline__test",
    start_idx=90,
)

display(summarize_main_metrics(df_90_end, "eventsonly"))
display(summarize_by_event_type(df_90_end, "eventsonly"))
display(summarize_by_event_subtype(df_90_end, "eventsonly"))
display(summarize_events_breakdown(df_90_end)["attributes"])
display(summarize_events_breakdown(df_90_end)["roles"])

#### One-shot

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="eventsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__llama17__oneshot__test",
    start_idx=0,
    end_idx=30,
)

display(summarize_main_metrics(df_0_30, "eventsonly"))
display(summarize_by_event_type(df_0_30, "eventsonly"))
display(summarize_by_event_subtype(df_0_30, "eventsonly"))
display(summarize_events_breakdown(df_0_30)["attributes"])
display(summarize_events_breakdown(df_0_30)["roles"])

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="eventsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__llama17__oneshot__test",
    start_idx=30,
    end_idx=60,
)

display(summarize_main_metrics(df_30_60, "eventsonly"))
display(summarize_by_event_type(df_30_60, "eventsonly"))
display(summarize_by_event_subtype(df_30_60, "eventsonly"))
display(summarize_events_breakdown(df_30_60)["attributes"])
display(summarize_events_breakdown(df_30_60)["roles"])

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="eventsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__llama17__oneshot__test",
    start_idx=60,
    end_idx=90,
)

display(summarize_main_metrics(df_60_90, "eventsonly"))
display(summarize_by_event_type(df_60_90, "eventsonly"))
display(summarize_by_event_subtype(df_60_90, "eventsonly"))
display(summarize_events_breakdown(df_60_90)["attributes"])
display(summarize_events_breakdown(df_60_90)["roles"])

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="eventsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__llama17__oneshot__test",
    start_idx=90,
)

display(summarize_main_metrics(df_90_end, "eventsonly"))
display(summarize_by_event_type(df_90_end, "eventsonly"))
display(summarize_by_event_subtype(df_90_end, "eventsonly"))
display(summarize_events_breakdown(df_90_end)["attributes"])
display(summarize_events_breakdown(df_90_end)["roles"])

#### Few-shot

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="eventsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__llama17__fewshot__test",
    start_idx=0,
    end_idx=30,
)

display(summarize_main_metrics(df_0_30, "eventsonly"))
display(summarize_by_event_type(df_0_30, "eventsonly"))
display(summarize_by_event_subtype(df_0_30, "eventsonly"))
display(summarize_events_breakdown(df_0_30)["attributes"])
display(summarize_events_breakdown(df_0_30)["roles"])

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="eventsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__llama17__fewshot__test",
    start_idx=30,
    end_idx=60,
)

display(summarize_main_metrics(df_30_60, "eventsonly"))
display(summarize_by_event_type(df_30_60, "eventsonly"))
display(summarize_by_event_subtype(df_30_60, "eventsonly"))
display(summarize_events_breakdown(df_30_60)["attributes"])
display(summarize_events_breakdown(df_30_60)["roles"])

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="eventsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__llama17__fewshot__test",
    start_idx=60,
    end_idx=90,
)

display(summarize_main_metrics(df_60_90, "eventsonly"))
display(summarize_by_event_type(df_60_90, "eventsonly"))
display(summarize_by_event_subtype(df_60_90, "eventsonly"))
display(summarize_events_breakdown(df_60_90)["attributes"])
display(summarize_events_breakdown(df_60_90)["roles"])

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="eventsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__llama17__fewshot__test",
    start_idx=90,
)

display(summarize_main_metrics(df_90_end, "eventsonly"))
display(summarize_by_event_type(df_90_end, "eventsonly"))
display(summarize_by_event_subtype(df_90_end, "eventsonly"))
display(summarize_events_breakdown(df_90_end)["attributes"])
display(summarize_events_breakdown(df_90_end)["roles"])

#### Few-shot II

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="eventsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__llama17__fewshot__test_2",
    start_idx=0,
    end_idx=30,
)

display(summarize_main_metrics(df_0_30, "eventsonly"))
display(summarize_by_event_type(df_0_30, "eventsonly"))
display(summarize_by_event_subtype(df_0_30, "eventsonly"))
display(summarize_events_breakdown(df_0_30)["attributes"])
display(summarize_events_breakdown(df_0_30)["roles"])

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="eventsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__llama17__fewshot__test_2",
    start_idx=30,
    end_idx=60,
)

display(summarize_main_metrics(df_30_60, "eventsonly"))
display(summarize_by_event_type(df_30_60, "eventsonly"))
display(summarize_by_event_subtype(df_30_60, "eventsonly"))
display(summarize_events_breakdown(df_30_60)["attributes"])
display(summarize_events_breakdown(df_30_60)["roles"])

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="eventsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__llama17__fewshot__test_2",
    start_idx=60,
    end_idx=90,
)

display(summarize_main_metrics(df_60_90, "eventsonly"))
display(summarize_by_event_type(df_60_90, "eventsonly"))
display(summarize_by_event_subtype(df_60_90, "eventsonly"))
display(summarize_events_breakdown(df_60_90)["attributes"])
display(summarize_events_breakdown(df_60_90)["roles"])

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="eventsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__llama17__fewshot__test_2",
    start_idx=90,
)

display(summarize_main_metrics(df_90_end, "eventsonly"))
display(summarize_by_event_type(df_90_end, "eventsonly"))
display(summarize_by_event_subtype(df_90_end, "eventsonly"))
display(summarize_events_breakdown(df_90_end)["attributes"])
display(summarize_events_breakdown(df_90_end)["roles"])

#### Few-shot III

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="eventsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__llama17__fewshot__test_3",
    start_idx=0,
    end_idx=30,
)

display(summarize_main_metrics(df_0_30, "eventsonly"))
display(summarize_by_event_type(df_0_30, "eventsonly"))
display(summarize_by_event_subtype(df_0_30, "eventsonly"))
display(summarize_events_breakdown(df_0_30)["attributes"])
display(summarize_events_breakdown(df_0_30)["roles"])

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="eventsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__llama17__fewshot__test_3",
    start_idx=30,
    end_idx=60,
)

display(summarize_main_metrics(df_30_60, "eventsonly"))
display(summarize_by_event_type(df_30_60, "eventsonly"))
display(summarize_by_event_subtype(df_30_60, "eventsonly"))
display(summarize_events_breakdown(df_30_60)["attributes"])
display(summarize_events_breakdown(df_30_60)["roles"])

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="eventsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__llama17__fewshot__test_3",
    start_idx=60,
    end_idx=90,
)

display(summarize_main_metrics(df_60_90, "eventsonly"))
display(summarize_by_event_type(df_60_90, "eventsonly"))
display(summarize_by_event_subtype(df_60_90, "eventsonly"))
display(summarize_events_breakdown(df_60_90)["attributes"])
display(summarize_events_breakdown(df_60_90)["roles"])

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="eventsonly",
    backend_name="groq_openai",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__llama17__fewshot__test_3",
    start_idx=90,
)

display(summarize_main_metrics(df_90_end, "eventsonly"))
display(summarize_by_event_type(df_90_end, "eventsonly"))
display(summarize_by_event_subtype(df_90_end, "eventsonly"))
display(summarize_events_breakdown(df_90_end)["attributes"])
display(summarize_events_breakdown(df_90_end)["roles"])

## Mistral Large 2411

### Entities

#### Baseline (annotated)

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="baseline",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__baseline__annotated__test",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="baseline",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__baseline__annotated__test",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="baseline",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__baseline__annotated__test",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="baseline",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__baseline__annotated__test",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

#### Baseline (all)

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="baseline",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__baseline__all__test",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="baseline",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__baseline__all__test",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="baseline",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__baseline__all__test",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="baseline",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__baseline__all__test",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

#### One-shot (annotated)

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__oneshot__annotated__test",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__oneshot__annotated__test",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__oneshot__annotated__test",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__oneshot__annotated__test",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

#### One-shot (all)

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__oneshot__all__test",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__oneshot__all__test",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__oneshot__all__test",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__oneshot__all__test",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

#### Few-shot (annotated)

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__fewshot__annotated__test",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__fewshot__annotated__test",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__fewshot__annotated__test",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__fewshot__annotated__test",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

#### Few-shot (all)

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__fewshot__all__test",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__fewshot__all__test",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__fewshot__all__test",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__fewshot__all__test",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

#### Few-shot II (annotated)

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__fewshot__annotated__test_2",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__fewshot__annotated__test_2",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__fewshot__annotated__test_2",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__fewshot__annotated__test_2",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

#### Few-shot II (all)

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__fewshot__all__test_2",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__fewshot__all__test_2",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__fewshot__all__test_2",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__fewshot__all__test_2",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

#### Few-shot III (annotated)

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__fewshot__annotated__test_3",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__fewshot__annotated__test_3",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__fewshot__annotated__test_3",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__fewshot__annotated__test_3",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

#### Few-shot III (all)

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__fewshot__all__test_3",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__fewshot__all__test_3",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__fewshot__all__test_3",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="entityonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__mistrallarge__fewshot__all__test_3",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

### Facts

#### Baseline

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="factsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="baseline",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__mistrallarge__baseline__test",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "factsonly"))
display(summarize_by_event_type(df_0_30, "factsonly"))
display(summarize_by_event_subtype(df_0_30, "factsonly"))
display(summarize_facts_breakdown(df_0_30)["attributes"])
display(summarize_facts_breakdown(df_0_30)["relations"])

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="factsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="baseline",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__mistrallarge__baseline__test",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "factsonly"))
display(summarize_by_event_type(df_30_60, "factsonly"))
display(summarize_by_event_subtype(df_30_60, "factsonly"))
display(summarize_facts_breakdown(df_30_60)["attributes"])
display(summarize_facts_breakdown(df_30_60)["relations"])

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="factsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="baseline",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__mistrallarge__baseline__test",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "factsonly"))
display(summarize_by_event_type(df_60_90, "factsonly"))
display(summarize_by_event_subtype(df_60_90, "factsonly"))
display(summarize_facts_breakdown(df_60_90)["attributes"])
display(summarize_facts_breakdown(df_60_90)["relations"])

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="factsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="baseline",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__mistrallarge__baseline__test",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "factsonly"))
display(summarize_by_event_type(df_90_end, "factsonly"))
display(summarize_by_event_subtype(df_90_end, "factsonly"))
display(summarize_facts_breakdown(df_90_end)["attributes"])
display(summarize_facts_breakdown(df_90_end)["relations"])

#### One-shot

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="factsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__mistrallarge__oneshot__test",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "factsonly"))
display(summarize_by_event_type(df_0_30, "factsonly"))
display(summarize_by_event_subtype(df_0_30, "factsonly"))
display(summarize_facts_breakdown(df_0_30)["attributes"])
display(summarize_facts_breakdown(df_0_30)["relations"])

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="factsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__mistrallarge__oneshot__test",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "factsonly"))
display(summarize_by_event_type(df_30_60, "factsonly"))
display(summarize_by_event_subtype(df_30_60, "factsonly"))
display(summarize_facts_breakdown(df_30_60)["attributes"])
display(summarize_facts_breakdown(df_30_60)["relations"])

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="factsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__mistrallarge__oneshot__test",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "factsonly"))
display(summarize_by_event_type(df_60_90, "factsonly"))
display(summarize_by_event_subtype(df_60_90, "factsonly"))
display(summarize_facts_breakdown(df_60_90)["attributes"])
display(summarize_facts_breakdown(df_60_90)["relations"])

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="factsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__mistrallarge__oneshot__test",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "factsonly"))
display(summarize_by_event_type(df_90_end, "factsonly"))
display(summarize_by_event_subtype(df_90_end, "factsonly"))
display(summarize_facts_breakdown(df_90_end)["attributes"])
display(summarize_facts_breakdown(df_90_end)["relations"])

#### Few-shot

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="factsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__mistrallarge__fewshot__test",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)

In [ ]:
display(summarize_main_metrics(df_0_30, "factsonly"))
display(summarize_by_event_type(df_0_30, "factsonly"))
display(summarize_by_event_subtype(df_0_30, "factsonly"))
display(summarize_facts_breakdown(df_0_30)["attributes"])
display(summarize_facts_breakdown(df_0_30)["relations"])

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="factsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__mistrallarge__fewshot__test",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "factsonly"))
display(summarize_by_event_type(df_30_60, "factsonly"))
display(summarize_by_event_subtype(df_30_60, "factsonly"))
display(summarize_facts_breakdown(df_30_60)["attributes"])
display(summarize_facts_breakdown(df_30_60)["relations"])

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="factsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__mistrallarge__fewshot__test",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "factsonly"))
display(summarize_by_event_type(df_60_90, "factsonly"))
display(summarize_by_event_subtype(df_60_90, "factsonly"))
display(summarize_facts_breakdown(df_60_90)["attributes"])
display(summarize_facts_breakdown(df_60_90)["relations"])

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="factsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__mistrallarge__fewshot__test",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "factsonly"))
display(summarize_by_event_type(df_90_end, "factsonly"))
display(summarize_by_event_subtype(df_90_end, "factsonly"))
display(summarize_facts_breakdown(df_90_end)["attributes"])
display(summarize_facts_breakdown(df_90_end)["relations"])

#### Few-shot II

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="factsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__mistrallarge__fewshot__test_2",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "factsonly"))
display(summarize_by_event_type(df_0_30, "factsonly"))
display(summarize_by_event_subtype(df_0_30, "factsonly"))
display(summarize_facts_breakdown(df_0_30)["attributes"])
display(summarize_facts_breakdown(df_0_30)["relations"])

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="factsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__mistrallarge__fewshot__test_2",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "factsonly"))
display(summarize_by_event_type(df_30_60, "factsonly"))
display(summarize_by_event_subtype(df_30_60, "factsonly"))
display(summarize_facts_breakdown(df_30_60)["attributes"])
display(summarize_facts_breakdown(df_30_60)["relations"])

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="factsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__mistrallarge__fewshot__test_2",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "factsonly"))
display(summarize_by_event_type(df_60_90, "factsonly"))
display(summarize_by_event_subtype(df_60_90, "factsonly"))
display(summarize_facts_breakdown(df_60_90)["attributes"])
display(summarize_facts_breakdown(df_60_90)["relations"])

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="factsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__mistrallarge__fewshot__test_2",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "factsonly"))
display(summarize_by_event_type(df_90_end, "factsonly"))
display(summarize_by_event_subtype(df_90_end, "factsonly"))
display(summarize_facts_breakdown(df_90_end)["attributes"])
display(summarize_facts_breakdown(df_90_end)["relations"])

#### Few-shot III

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="factsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__mistrallarge__fewshot__test_3",
    start_idx=0,
    end_idx=30,
)

display(df_0_30)
display(summarize_main_metrics(df_0_30, "factsonly"))
display(summarize_by_event_type(df_0_30, "factsonly"))
display(summarize_by_event_subtype(df_0_30, "factsonly"))
display(summarize_facts_breakdown(df_0_30)["attributes"])
display(summarize_facts_breakdown(df_0_30)["relations"])

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="factsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__mistrallarge__fewshot__test_3",
    start_idx=30,
    end_idx=60,
)

display(df_30_60)
display(summarize_main_metrics(df_30_60, "factsonly"))
display(summarize_by_event_type(df_30_60, "factsonly"))
display(summarize_by_event_subtype(df_30_60, "factsonly"))
display(summarize_facts_breakdown(df_30_60)["attributes"])
display(summarize_facts_breakdown(df_30_60)["relations"])

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="factsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__mistrallarge__fewshot__test_3",
    start_idx=60,
    end_idx=90,
)

display(df_60_90)
display(summarize_main_metrics(df_60_90, "factsonly"))
display(summarize_by_event_type(df_60_90, "factsonly"))
display(summarize_by_event_subtype(df_60_90, "factsonly"))
display(summarize_facts_breakdown(df_60_90)["attributes"])
display(summarize_facts_breakdown(df_60_90)["relations"])

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="factsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__mistrallarge__fewshot__test_3",
    start_idx=90,
)

display(df_90_end)
display(summarize_main_metrics(df_90_end, "factsonly"))
display(summarize_by_event_type(df_90_end, "factsonly"))
display(summarize_by_event_subtype(df_90_end, "factsonly"))
display(summarize_facts_breakdown(df_90_end)["attributes"])
display(summarize_facts_breakdown(df_90_end)["relations"])

### Events

#### Baseline

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="eventsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="baseline",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__mistrallarge__baseline__test",
    start_idx=0,
    end_idx=30,
)

display(summarize_main_metrics(df_0_30, "eventsonly"))
display(summarize_by_event_type(df_0_30, "eventsonly"))
display(summarize_by_event_subtype(df_0_30, "eventsonly"))
display(summarize_events_breakdown(df_0_30)["attributes"])
display(summarize_events_breakdown(df_0_30)["roles"])

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="eventsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="baseline",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__mistrallarge__baseline__test",
    start_idx=30,
    end_idx=60,
)

display(summarize_main_metrics(df_30_60, "eventsonly"))
display(summarize_by_event_type(df_30_60, "eventsonly"))
display(summarize_by_event_subtype(df_30_60, "eventsonly"))
display(summarize_events_breakdown(df_30_60)["attributes"])
display(summarize_events_breakdown(df_30_60)["roles"])

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="eventsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="baseline",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__mistrallarge__baseline__test",
    start_idx=60,
    end_idx=90,
)

display(summarize_main_metrics(df_60_90, "eventsonly"))
display(summarize_by_event_type(df_60_90, "eventsonly"))
display(summarize_by_event_subtype(df_60_90, "eventsonly"))
display(summarize_events_breakdown(df_60_90)["attributes"])
display(summarize_events_breakdown(df_60_90)["roles"])

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="eventsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="baseline",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__mistrallarge__baseline__test",
    start_idx=90,
)

display(summarize_main_metrics(df_90_end, "eventsonly"))
display(summarize_by_event_type(df_90_end, "eventsonly"))
display(summarize_by_event_subtype(df_90_end, "eventsonly"))
display(summarize_events_breakdown(df_90_end)["attributes"])
display(summarize_events_breakdown(df_90_end)["roles"])

#### One-shot

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="eventsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__mistrallarge__oneshot__test",
    start_idx=0,
    end_idx=30,
)

display(summarize_main_metrics(df_0_30, "eventsonly"))
display(summarize_by_event_type(df_0_30, "eventsonly"))
display(summarize_by_event_subtype(df_0_30, "eventsonly"))
display(summarize_events_breakdown(df_0_30)["attributes"])
display(summarize_events_breakdown(df_0_30)["roles"])

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="eventsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__mistrallarge__oneshot__test",
    start_idx=30,
    end_idx=60,
)

display(summarize_main_metrics(df_30_60, "eventsonly"))
display(summarize_by_event_type(df_30_60, "eventsonly"))
display(summarize_by_event_subtype(df_30_60, "eventsonly"))
display(summarize_events_breakdown(df_30_60)["attributes"])
display(summarize_events_breakdown(df_30_60)["roles"])

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="eventsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__mistrallarge__oneshot__test",
    start_idx=60,
    end_idx=90,
)

display(summarize_main_metrics(df_60_90, "eventsonly"))
display(summarize_by_event_type(df_60_90, "eventsonly"))
display(summarize_by_event_subtype(df_60_90, "eventsonly"))
display(summarize_events_breakdown(df_60_90)["attributes"])
display(summarize_events_breakdown(df_60_90)["roles"])

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="eventsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__mistrallarge__oneshot__test",
    start_idx=90,
)

display(summarize_main_metrics(df_90_end, "eventsonly"))
display(summarize_by_event_type(df_90_end, "eventsonly"))
display(summarize_by_event_subtype(df_90_end, "eventsonly"))
display(summarize_events_breakdown(df_90_end)["attributes"])
display(summarize_events_breakdown(df_90_end)["roles"])

#### Few-shot

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="eventsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__mistrallarge__fewshot__test",
    start_idx=0,
    end_idx=30,
)

display(summarize_main_metrics(df_0_30, "eventsonly"))
display(summarize_by_event_type(df_0_30, "eventsonly"))
display(summarize_by_event_subtype(df_0_30, "eventsonly"))
display(summarize_events_breakdown(df_0_30)["attributes"])
display(summarize_events_breakdown(df_0_30)["roles"])

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="eventsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__mistrallarge__fewshot__test",
    start_idx=30,
    end_idx=60,
)

display(summarize_main_metrics(df_30_60, "eventsonly"))
display(summarize_by_event_type(df_30_60, "eventsonly"))
display(summarize_by_event_subtype(df_30_60, "eventsonly"))
display(summarize_events_breakdown(df_30_60)["attributes"])
display(summarize_events_breakdown(df_30_60)["roles"])

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="eventsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__mistrallarge__fewshot__test",
    start_idx=60,
    end_idx=90,
)

display(summarize_main_metrics(df_60_90, "eventsonly"))
display(summarize_by_event_type(df_60_90, "eventsonly"))
display(summarize_by_event_subtype(df_60_90, "eventsonly"))
display(summarize_events_breakdown(df_60_90)["attributes"])
display(summarize_events_breakdown(df_60_90)["roles"])

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="eventsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__mistrallarge__fewshot__test",
    start_idx=90,
)

display(summarize_main_metrics(df_90_end, "eventsonly"))
display(summarize_by_event_type(df_90_end, "eventsonly"))
display(summarize_by_event_subtype(df_90_end, "eventsonly"))
display(summarize_events_breakdown(df_90_end)["attributes"])
display(summarize_events_breakdown(df_90_end)["roles"])

#### Few-shot II

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="eventsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__mistrallarge__fewshot__test_2",
    start_idx=0,
    end_idx=30,
)

display(summarize_main_metrics(df_0_30, "eventsonly"))
display(summarize_by_event_type(df_0_30, "eventsonly"))
display(summarize_by_event_subtype(df_0_30, "eventsonly"))
display(summarize_events_breakdown(df_0_30)["attributes"])
display(summarize_events_breakdown(df_0_30)["roles"])

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="eventsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__mistrallarge__fewshot__test_2",
    start_idx=30,
    end_idx=60,
)

display(summarize_main_metrics(df_30_60, "eventsonly"))
display(summarize_by_event_type(df_30_60, "eventsonly"))
display(summarize_by_event_subtype(df_30_60, "eventsonly"))
display(summarize_events_breakdown(df_30_60)["attributes"])
display(summarize_events_breakdown(df_30_60)["roles"])

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="eventsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__mistrallarge__fewshot__test_2",
    start_idx=60,
    end_idx=90,
)

display(summarize_main_metrics(df_60_90, "eventsonly"))
display(summarize_by_event_type(df_60_90, "eventsonly"))
display(summarize_by_event_subtype(df_60_90, "eventsonly"))
display(summarize_events_breakdown(df_60_90)["attributes"])
display(summarize_events_breakdown(df_60_90)["roles"])

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="eventsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__mistrallarge__fewshot__test_2",
    start_idx=90,
)

display(summarize_main_metrics(df_90_end, "eventsonly"))
display(summarize_by_event_type(df_90_end, "eventsonly"))
display(summarize_by_event_subtype(df_90_end, "eventsonly"))
display(summarize_events_breakdown(df_90_end)["attributes"])
display(summarize_events_breakdown(df_90_end)["roles"])

#### Few-shot III

In [ ]:
split = load_split()

df_0_30 = run_experiment(
    task_name="eventsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__mistrallarge__fewshot__test_3",
    start_idx=0,
    end_idx=30,
)

display(summarize_main_metrics(df_0_30, "eventsonly"))
display(summarize_by_event_type(df_0_30, "eventsonly"))
display(summarize_by_event_subtype(df_0_30, "eventsonly"))
display(summarize_events_breakdown(df_0_30)["attributes"])
display(summarize_events_breakdown(df_0_30)["roles"])

In [ ]:
split = load_split()

df_30_60 = run_experiment(
    task_name="eventsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__mistrallarge__fewshot__test_3",
    start_idx=30,
    end_idx=60,
)

display(summarize_main_metrics(df_30_60, "eventsonly"))
display(summarize_by_event_type(df_30_60, "eventsonly"))
display(summarize_by_event_subtype(df_30_60, "eventsonly"))
display(summarize_events_breakdown(df_30_60)["attributes"])
display(summarize_events_breakdown(df_30_60)["roles"])

In [ ]:
split = load_split()

df_60_90 = run_experiment(
    task_name="eventsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__mistrallarge__fewshot__test_3",
    start_idx=60,
    end_idx=90,
)

display(summarize_main_metrics(df_60_90, "eventsonly"))
display(summarize_by_event_type(df_60_90, "eventsonly"))
display(summarize_by_event_subtype(df_60_90, "eventsonly"))
display(summarize_events_breakdown(df_60_90)["attributes"])
display(summarize_events_breakdown(df_60_90)["roles"])

In [ ]:
split = load_split()

df_90_end = run_experiment(
    task_name="eventsonly",
    backend_name="mistral_openai",
    model_name="mistral-large-2411",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__mistrallarge__fewshot__test_3",
    start_idx=90,
)

display(summarize_main_metrics(df_90_end, "eventsonly"))
display(summarize_by_event_type(df_90_end, "eventsonly"))
display(summarize_by_event_subtype(df_90_end, "eventsonly"))
display(summarize_events_breakdown(df_90_end)["attributes"])
display(summarize_events_breakdown(df_90_end)["roles"])

## GPT 54 Nano

### Entities

#### Baseline (annotated)

In [ ]:
split = load_split()

result_0_30 = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="baseline",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__baseline__annotated__test",
    start_idx=0,
    end_idx=30,
    wait=True,
    poll_seconds=60,
)

df_0_30 = result_0_30["df"]

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))



In [ ]:
split = load_split()

result_30_60 = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="baseline",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__baseline__annotated__test",
    start_idx=30,
    end_idx=60,
    wait=True,
    poll_seconds=60,
)

df_30_60 = result_30_60["df"]

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

result_60_90 = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="baseline",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__baseline__annotated__test",
    start_idx=60,
    end_idx=90,
    wait=True,
    poll_seconds=60,
)

df_60_90 = result_60_90["df"]

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

result_90_end = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="baseline",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__baseline__annotated__test",
    start_idx=90,
    wait=True,
    poll_seconds=60,
)

df_90_end = result_90_end["df"]

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

#### Baseline (all)

In [ ]:
split = load_split()

result_0_30 = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="baseline",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__baseline__all__test",
    start_idx=0,
    end_idx=30,
    wait=True,
    poll_seconds=60,
)

df_0_30 = result_0_30["df"]

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

result_30_60 = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="baseline",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__baseline__all__test",
    start_idx=30,
    end_idx=60,
    wait=True,
    poll_seconds=60,
)

df_30_60 = result_30_60["df"]

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

result_60_90 = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="baseline",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__baseline__all__test",
    start_idx=60,
    end_idx=90,
    wait=True,
    poll_seconds=60,
)

df_60_90 = result_60_90["df"]

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

result_90_end = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="baseline",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__baseline__all__test",
    start_idx=90,
    wait=True,
    poll_seconds=60,
)

df_90_end = result_90_end["df"]

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

#### One-shot (annotated)

In [ ]:
split = load_split()

result_0_30 = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__oneshot__annotated__test",
    start_idx=0,
    end_idx=30,
    wait=True,
    poll_seconds=60,
)

df_0_30 = result_0_30["df"]

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

result_30_60 = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__oneshot__annotated__test",
    start_idx=30,
    end_idx=60,
    wait=True,
    poll_seconds=60,
)

df_30_60 = result_30_60["df"]

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

result_60_90 = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__oneshot__annotated__test",
    start_idx=60,
    end_idx=90,
    wait=True,
    poll_seconds=60,
)

df_60_90 = result_60_90["df"]

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

result_90_end = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__oneshot__annotated__test",
    start_idx=90,
    wait=True,
    poll_seconds=60,
)

df_90_end = result_90_end["df"]

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

#### One-shot (all)

In [ ]:
split = load_split()

result_0_30 = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__oneshot__all__test",
    start_idx=0,
    end_idx=30,
    wait=True,
    poll_seconds=60,
)

df_0_30 = result_0_30["df"]

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

result_30_60 = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__oneshot__all__test",
    start_idx=30,
    end_idx=60,
    wait=True,
    poll_seconds=60,
)

df_30_60 = result_30_60["df"]

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

result_60_90 = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__oneshot__all__test",
    start_idx=60,
    end_idx=90,
    wait=True,
    poll_seconds=60,
)

df_60_90 = result_60_90["df"]

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

result_90_end = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__oneshot__all__test",
    start_idx=90,
    wait=True,
    poll_seconds=60,
)

df_90_end = result_90_end["df"]

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

#### Few-shot (annotated)

In [ ]:
split = load_split()

result_0_30 = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__fewshot__annotated__test",
    start_idx=0,
    end_idx=30,
    wait=True,
    poll_seconds=60,
)

df_0_30 = result_0_30["df"]

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

result_30_60 = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__fewshot__annotated__test",
    start_idx=30,
    end_idx=60,
    wait=True,
    poll_seconds=60,
)

df_30_60 = result_30_60["df"]

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

result_60_90 = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__fewshot__annotated__test",
    start_idx=60,
    end_idx=90,
    wait=True,
    poll_seconds=60,
)

df_60_90 = result_60_90["df"]

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

result_90_end = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__fewshot__annotated__test",
    start_idx=90,
    wait=True,
    poll_seconds=60,
)

df_90_end = result_90_end["df"]

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

#### Few-shot (all)

In [ ]:
split = load_split()

result_0_30 = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__fewshot__all__test",
    start_idx=0,
    end_idx=30,
    wait=True,
    poll_seconds=60,
)

df_0_30 = result_0_30["df"]

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

result_30_60 = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__fewshot__all__test",
    start_idx=30,
    end_idx=60,
    wait=True,
    poll_seconds=60,
)

df_30_60 = result_30_60["df"]

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

result_60_90 = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__fewshot__all__test",
    start_idx=60,
    end_idx=90,
    wait=True,
    poll_seconds=60,
)

df_60_90 = result_60_90["df"]

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

result_90_end = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__fewshot__all__test",
    start_idx=90,
    wait=True,
    poll_seconds=60,
)

df_90_end = result_90_end["df"]

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

#### Few-shot II (annotated)

In [ ]:
split = load_split()

result_0_30 = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__fewshot__annotated__test_2",
    start_idx=0,
    end_idx=30,
    wait=True,
    poll_seconds=60,
)

df_0_30 = result_0_30["df"]

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

result_30_60 = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__fewshot__annotated__test_2",
    start_idx=30,
    end_idx=60,
    wait=True,
    poll_seconds=60,
)

df_30_60 = result_30_60["df"]

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

result_60_90 = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__fewshot__annotated__test_2",
    start_idx=60,
    end_idx=90,
    wait=True,
    poll_seconds=60,
)

df_60_90 = result_60_90["df"]

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

result_90_end = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__fewshot__annotated__test_2",
    start_idx=90,
    wait=True,
    poll_seconds=60,
)

df_90_end = result_90_end["df"]

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

#### Few-shot II (all)

In [ ]:
split = load_split()

result_0_30 = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__fewshot__all__test_2",
    start_idx=0,
    end_idx=30,
    wait=True,
    poll_seconds=60,
)

df_0_30 = result_0_30["df"]

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

result_30_60 = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__fewshot__all__test_2",
    start_idx=30,
    end_idx=60,
    wait=True,
    poll_seconds=60,
)

df_30_60 = result_30_60["df"]

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

result_60_90 = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__fewshot__all__test_2",
    start_idx=60,
    end_idx=90,
    wait=True,
    poll_seconds=60,
)

df_60_90 = result_60_90["df"]

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

result_90_end = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__fewshot__all__test_2",
    start_idx=90,
    wait=True,
    poll_seconds=60,
)

df_90_end = result_90_end["df"]

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

#### Few-shot III (annotated)

In [ ]:
split = load_split()

result_0_30 = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__fewshot__annotated__test_3",
    start_idx=0,
    end_idx=30,
    wait=True,
    poll_seconds=60,
)

df_0_30 = result_0_30["df"]

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

result_30_60 = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__fewshot__annotated__test_3",
    start_idx=30,
    end_idx=60,
    wait=True,
    poll_seconds=60,
)

df_30_60 = result_30_60["df"]

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

result_60_90 = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__fewshot__annotated__test_3",
    start_idx=60,
    end_idx=90,
    wait=True,
    poll_seconds=60,
)

df_60_90 = result_60_90["df"]

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

result_90_end = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="annotated",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__fewshot__annotated__test_3",
    start_idx=90,
    wait=True,
    poll_seconds=60,
)

df_90_end = result_90_end["df"]

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

#### Few-shot III (all)

In [ ]:
split = load_split()

result_0_30 = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__fewshot__all__test_3",
    start_idx=0,
    end_idx=30,
    wait=True,
    poll_seconds=60,
)

df_0_30 = result_0_30["df"]

display(df_0_30)
display(summarize_main_metrics(df_0_30, "entityonly"))
display(summarize_entities_breakdown(df_0_30))

In [ ]:
split = load_split()

result_30_60 = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__fewshot__all__test_3",
    start_idx=30,
    end_idx=60,
    wait=True,
    poll_seconds=60,
)

df_30_60 = result_30_60["df"]

display(df_30_60)
display(summarize_main_metrics(df_30_60, "entityonly"))
display(summarize_entities_breakdown(df_30_60))

In [ ]:
split = load_split()

result_60_90 = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__fewshot__all__test_3",
    start_idx=60,
    end_idx=90,
    wait=True,
    poll_seconds=60,
)

df_60_90 = result_60_90["df"]

display(df_60_90)
display(summarize_main_metrics(df_60_90, "entityonly"))
display(summarize_entities_breakdown(df_60_90))

In [ ]:
split = load_split()

result_90_end = run_experiment_openai_batch(
    task_name="entityonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    entity_mode="all",
    max_output_tokens=8192,
    save_prefix="entityonly__gpt54nano__fewshot__all__test_3",
    start_idx=90,
    wait=True,
    poll_seconds=60,
)

df_90_end = result_90_end["df"]

display(df_90_end)
display(summarize_main_metrics(df_90_end, "entityonly"))
display(summarize_entities_breakdown(df_90_end))

### Facts

#### Baseline

In [ ]:
split = load_split()

result_0_30 = run_experiment_openai_batch(
    task_name="factsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="baseline",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gpt54nano__baseline__test",
    start_idx=0,
    end_idx=30,
    wait=True,
    poll_seconds=60,
)

df_0_30 = result_0_30["df"]

display(df_0_30)
display(summarize_main_metrics(df_0_30, "factsonly"))
display(summarize_by_event_type(df_0_30, "factsonly"))
display(summarize_by_event_subtype(df_0_30, "factsonly"))
display(summarize_facts_breakdown(df_0_30)["attributes"])
display(summarize_facts_breakdown(df_0_30)["relations"])

In [ ]:
split = load_split()

result_30_60 = run_experiment_openai_batch(
    task_name="factsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="baseline",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gpt54nano__baseline__test",
    start_idx=30,
    end_idx=60,
    wait=True,
    poll_seconds=60,
)

df_30_60 = result_30_60["df"]

display(df_30_60)
display(summarize_main_metrics(df_30_60, "factsonly"))
display(summarize_by_event_type(df_30_60, "factsonly"))
display(summarize_by_event_subtype(df_30_60, "factsonly"))
display(summarize_facts_breakdown(df_30_60)["attributes"])
display(summarize_facts_breakdown(df_30_60)["relations"])

In [ ]:
split = load_split()

result_60_90 = run_experiment_openai_batch(
    task_name="factsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="baseline",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gpt54nano__baseline__test",
    start_idx=60,
    end_idx=90,
    wait=True,
    poll_seconds=60,
)

df_60_90 = result_60_90["df"]

display(df_60_90)
display(summarize_main_metrics(df_60_90, "factsonly"))
display(summarize_by_event_type(df_60_90, "factsonly"))
display(summarize_by_event_subtype(df_60_90, "factsonly"))
display(summarize_facts_breakdown(df_60_90)["attributes"])
display(summarize_facts_breakdown(df_60_90)["relations"])

In [ ]:
split = load_split()

result_90_end = run_experiment_openai_batch(
    task_name="factsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="baseline",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gpt54nano__baseline__test",
    start_idx=90,
    wait=True,
    poll_seconds=60,
)

df_90_end = result_90_end["df"]

display(df_90_end)
display(summarize_main_metrics(df_90_end, "factsonly"))
display(summarize_by_event_type(df_90_end, "factsonly"))
display(summarize_by_event_subtype(df_90_end, "factsonly"))
display(summarize_facts_breakdown(df_90_end)["attributes"])
display(summarize_facts_breakdown(df_90_end)["relations"])

#### One-shot

In [ ]:
split = load_split()

result_0_30 = run_experiment_openai_batch(
    task_name="factsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gpt54nano__oneshot__test",
    start_idx=0,
    end_idx=30,
    wait=True,
    poll_seconds=60,
)

df_0_30 = result_0_30["df"]

display(df_0_30)
display(summarize_main_metrics(df_0_30, "factsonly"))
display(summarize_by_event_type(df_0_30, "factsonly"))
display(summarize_by_event_subtype(df_0_30, "factsonly"))
display(summarize_facts_breakdown(df_0_30)["attributes"])
display(summarize_facts_breakdown(df_0_30)["relations"])

In [ ]:
split = load_split()

result_30_60 = run_experiment_openai_batch(
    task_name="factsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gpt54nano__oneshot__test",
    start_idx=30,
    end_idx=60,
    wait=True,
    poll_seconds=60,
)

df_30_60 = result_30_60["df"]

display(df_30_60)
display(summarize_main_metrics(df_30_60, "factsonly"))
display(summarize_by_event_type(df_30_60, "factsonly"))
display(summarize_by_event_subtype(df_30_60, "factsonly"))
display(summarize_facts_breakdown(df_30_60)["attributes"])
display(summarize_facts_breakdown(df_30_60)["relations"])

In [ ]:
split = load_split()

result_60_90 = run_experiment_openai_batch(
    task_name="factsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gpt54nano__oneshot__test",
    start_idx=60,
    end_idx=90,
    wait=True,
    poll_seconds=60,
)

df_60_90 = result_60_90["df"]

display(df_60_90)
display(summarize_main_metrics(df_60_90, "factsonly"))
display(summarize_by_event_type(df_60_90, "factsonly"))
display(summarize_by_event_subtype(df_60_90, "factsonly"))
display(summarize_facts_breakdown(df_60_90)["attributes"])
display(summarize_facts_breakdown(df_60_90)["relations"])

In [ ]:
split = load_split()

result_90_end = run_experiment_openai_batch(
    task_name="factsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gpt54nano__oneshot__test",
    start_idx=90,
    wait=True,
    poll_seconds=60,
)

df_90_end = result_90_end["df"]

display(df_90_end)
display(summarize_main_metrics(df_90_end, "factsonly"))
display(summarize_by_event_type(df_90_end, "factsonly"))
display(summarize_by_event_subtype(df_90_end, "factsonly"))
display(summarize_facts_breakdown(df_90_end)["attributes"])
display(summarize_facts_breakdown(df_90_end)["relations"])

#### Few-shot

In [ ]:
split = load_split()

result_0_30 = run_experiment_openai_batch(
    task_name="factsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gpt54nano__fewshot__test",
    start_idx=0,
    end_idx=30,
    wait=True,
    poll_seconds=60,
)

df_0_30 = result_0_30["df"]

display(df_0_30)
display(summarize_main_metrics(df_0_30, "factsonly"))
display(summarize_by_event_type(df_0_30, "factsonly"))
display(summarize_by_event_subtype(df_0_30, "factsonly"))
display(summarize_facts_breakdown(df_0_30)["attributes"])
display(summarize_facts_breakdown(df_0_30)["relations"])

In [ ]:
split = load_split()

result_30_60 = run_experiment_openai_batch(
    task_name="factsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gpt54nano__fewshot__test",
    start_idx=30,
    end_idx=60,
    wait=True,
    poll_seconds=60,
)

df_30_60 = result_30_60["df"]

display(df_30_60)
display(summarize_main_metrics(df_30_60, "factsonly"))
display(summarize_by_event_type(df_30_60, "factsonly"))
display(summarize_by_event_subtype(df_30_60, "factsonly"))
display(summarize_facts_breakdown(df_30_60)["attributes"])
display(summarize_facts_breakdown(df_30_60)["relations"])

In [ ]:
split = load_split()

result_60_90 = run_experiment_openai_batch(
    task_name="factsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gpt54nano__fewshot__test",
    start_idx=60,
    end_idx=90,
    wait=True,
    poll_seconds=60,
)

df_60_90 = result_60_90["df"]

display(df_60_90)
display(summarize_main_metrics(df_60_90, "factsonly"))
display(summarize_by_event_type(df_60_90, "factsonly"))
display(summarize_by_event_subtype(df_60_90, "factsonly"))
display(summarize_facts_breakdown(df_60_90)["attributes"])
display(summarize_facts_breakdown(df_60_90)["relations"])

In [ ]:
split = load_split()

result_90_end = run_experiment_openai_batch(
    task_name="factsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gpt54nano__fewshot__test",
    start_idx=90,
    wait=True,
    poll_seconds=60,
)

df_90_end = result_90_end["df"]

display(df_90_end)
display(summarize_main_metrics(df_90_end, "factsonly"))
display(summarize_by_event_type(df_90_end, "factsonly"))
display(summarize_by_event_subtype(df_90_end, "factsonly"))
display(summarize_facts_breakdown(df_90_end)["attributes"])
display(summarize_facts_breakdown(df_90_end)["relations"])

#### Few-shot II

In [ ]:
split = load_split()

result_0_30 = run_experiment_openai_batch(
    task_name="factsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gpt54nano__fewshot__test_2",
    start_idx=0,
    end_idx=30,
    wait=True,
    poll_seconds=60,
)

df_0_30 = result_0_30["df"]

display(df_0_30)
display(summarize_main_metrics(df_0_30, "factsonly"))
display(summarize_by_event_type(df_0_30, "factsonly"))
display(summarize_by_event_subtype(df_0_30, "factsonly"))
display(summarize_facts_breakdown(df_0_30)["attributes"])
display(summarize_facts_breakdown(df_0_30)["relations"])

In [ ]:
split = load_split()

result_30_60 = run_experiment_openai_batch(
    task_name="factsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gpt54nano__fewshot__test_2",
    start_idx=30,
    end_idx=60,
    wait=True,
    poll_seconds=60,
)

df_30_60 = result_30_60["df"]

display(df_30_60)
display(summarize_main_metrics(df_30_60, "factsonly"))
display(summarize_by_event_type(df_30_60, "factsonly"))
display(summarize_by_event_subtype(df_30_60, "factsonly"))
display(summarize_facts_breakdown(df_30_60)["attributes"])
display(summarize_facts_breakdown(df_30_60)["relations"])

In [ ]:
split = load_split()

result_60_90 = run_experiment_openai_batch(
    task_name="factsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gpt54nano__fewshot__test_2",
    start_idx=60,
    end_idx=90,
    wait=True,
    poll_seconds=60,
)

df_60_90 = result_60_90["df"]

display(df_60_90)
display(summarize_main_metrics(df_60_90, "factsonly"))
display(summarize_by_event_type(df_60_90, "factsonly"))
display(summarize_by_event_subtype(df_60_90, "factsonly"))
display(summarize_facts_breakdown(df_60_90)["attributes"])
display(summarize_facts_breakdown(df_60_90)["relations"])

In [ ]:
split = load_split()

result_90_end = run_experiment_openai_batch(
    task_name="factsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gpt54nano__fewshot__test_2",
    start_idx=90,
    wait=True,
    poll_seconds=60,
)

df_90_end = result_90_end["df"]

display(df_90_end)
display(summarize_main_metrics(df_90_end, "factsonly"))
display(summarize_by_event_type(df_90_end, "factsonly"))
display(summarize_by_event_subtype(df_90_end, "factsonly"))
display(summarize_facts_breakdown(df_90_end)["attributes"])
display(summarize_facts_breakdown(df_90_end)["relations"])

#### Few-shot III

In [ ]:
split = load_split()

result_0_30 = run_experiment_openai_batch(
    task_name="factsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gpt54nano__fewshot__test_3",
    start_idx=0,
    end_idx=30,
    wait=True,
    poll_seconds=60,
)

df_0_30 = result_0_30["df"]

display(df_0_30)
display(summarize_main_metrics(df_0_30, "factsonly"))
display(summarize_by_event_type(df_0_30, "factsonly"))
display(summarize_by_event_subtype(df_0_30, "factsonly"))
display(summarize_facts_breakdown(df_0_30)["attributes"])
display(summarize_facts_breakdown(df_0_30)["relations"])

In [ ]:
split = load_split()

result_30_60 = run_experiment_openai_batch(
    task_name="factsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gpt54nano__fewshot__test_3",
    start_idx=30,
    end_idx=60,
    wait=True,
    poll_seconds=60,
)

df_30_60 = result_30_60["df"]

display(df_30_60)
display(summarize_main_metrics(df_30_60, "factsonly"))
display(summarize_by_event_type(df_30_60, "factsonly"))
display(summarize_by_event_subtype(df_30_60, "factsonly"))
display(summarize_facts_breakdown(df_30_60)["attributes"])
display(summarize_facts_breakdown(df_30_60)["relations"])

In [ ]:
split = load_split()

result_60_90 = run_experiment_openai_batch(
    task_name="factsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gpt54nano__fewshot__test_3",
    start_idx=60,
    end_idx=90,
    wait=True,
    poll_seconds=60,
)

df_60_90 = result_60_90["df"]

display(df_60_90)
display(summarize_main_metrics(df_60_90, "factsonly"))
display(summarize_by_event_type(df_60_90, "factsonly"))
display(summarize_by_event_subtype(df_60_90, "factsonly"))
display(summarize_facts_breakdown(df_60_90)["attributes"])
display(summarize_facts_breakdown(df_60_90)["relations"])

In [ ]:
split = load_split()

result_90_end = run_experiment_openai_batch(
    task_name="factsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=8192,
    thinking_level="minimal",
    save_prefix="factsonly__gpt54nano__fewshot__test_3",
    start_idx=90,
    wait=True,
    poll_seconds=60,
)

df_90_end = result_90_end["df"]

display(df_90_end)
display(summarize_main_metrics(df_90_end, "factsonly"))
display(summarize_by_event_type(df_90_end, "factsonly"))
display(summarize_by_event_subtype(df_90_end, "factsonly"))
display(summarize_facts_breakdown(df_90_end)["attributes"])
display(summarize_facts_breakdown(df_90_end)["relations"])

### Events

#### Baseline

In [ ]:
split = load_split()

result_0_30 = run_experiment_openai_batch(
    task_name="eventsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="baseline",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gpt54nano__baseline__test",
    start_idx=0,
    end_idx=30,
    wait=True,
    poll_seconds=60,
)

df_0_30 = result_0_30["df"]

display(summarize_main_metrics(df_0_30, "eventsonly"))
display(summarize_by_event_type(df_0_30, "eventsonly"))
display(summarize_by_event_subtype(df_0_30, "eventsonly"))
display(summarize_events_breakdown(df_0_30)["attributes"])
display(summarize_events_breakdown(df_0_30)["roles"])

In [ ]:
split = load_split()

result_30_60 = run_experiment_openai_batch(
    task_name="eventsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="baseline",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gpt54nano__baseline__test",
    start_idx=30,
    end_idx=60,
    wait=True,
    poll_seconds=60,
)

df_30_60 = result_30_60["df"]

display(summarize_main_metrics(df_30_60, "eventsonly"))
display(summarize_by_event_type(df_30_60, "eventsonly"))
display(summarize_by_event_subtype(df_30_60, "eventsonly"))
display(summarize_events_breakdown(df_30_60)["attributes"])
display(summarize_events_breakdown(df_30_60)["roles"])

In [ ]:
split = load_split()

result_60_90 = run_experiment_openai_batch(
    task_name="eventsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="baseline",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gpt54nano__baseline__test",
    start_idx=60,
    end_idx=90,
    wait=True,
    poll_seconds=60,
)

df_60_90 = result_60_90["df"]

display(summarize_main_metrics(df_60_90, "eventsonly"))
display(summarize_by_event_type(df_60_90, "eventsonly"))
display(summarize_by_event_subtype(df_60_90, "eventsonly"))
display(summarize_events_breakdown(df_60_90)["attributes"])
display(summarize_events_breakdown(df_60_90)["roles"])

In [ ]:
split = load_split()

result_90_end = run_experiment_openai_batch(
    task_name="eventsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="baseline",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gpt54nano__baseline__test",
    start_idx=90,
    wait=True,
    poll_seconds=60,
)

df_90_end = result_90_end["df"]

display(summarize_main_metrics(df_90_end, "eventsonly"))
display(summarize_by_event_type(df_90_end, "eventsonly"))
display(summarize_by_event_subtype(df_90_end, "eventsonly"))
display(summarize_events_breakdown(df_90_end)["attributes"])
display(summarize_events_breakdown(df_90_end)["roles"])

#### One-shot

In [ ]:
split = load_split()

result_0_30 = run_experiment_openai_batch(
    task_name="eventsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gpt54nano__oneshot__test",
    start_idx=0,
    end_idx=30,
    wait=True,
    poll_seconds=60,
)

df_0_30 = result_0_30["df"]

display(summarize_main_metrics(df_0_30, "eventsonly"))
display(summarize_by_event_type(df_0_30, "eventsonly"))
display(summarize_by_event_subtype(df_0_30, "eventsonly"))
display(summarize_events_breakdown(df_0_30)["attributes"])
display(summarize_events_breakdown(df_0_30)["roles"])

In [ ]:
split = load_split()

result_30_60 = run_experiment_openai_batch(
    task_name="eventsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gpt54nano__oneshot__test",
    start_idx=30,
    end_idx=60,
    wait=True,
    poll_seconds=60,
)

df_30_60 = result_30_60["df"]

display(summarize_main_metrics(df_30_60, "eventsonly"))
display(summarize_by_event_type(df_30_60, "eventsonly"))
display(summarize_by_event_subtype(df_30_60, "eventsonly"))
display(summarize_events_breakdown(df_30_60)["attributes"])
display(summarize_events_breakdown(df_30_60)["roles"])

In [ ]:
split = load_split()

result_60_90 = run_experiment_openai_batch(
    task_name="eventsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gpt54nano__oneshot__test",
    start_idx=60,
    end_idx=90,
    wait=True,
    poll_seconds=60,
)

df_60_90 = result_60_90["df"]

display(summarize_main_metrics(df_60_90, "eventsonly"))
display(summarize_by_event_type(df_60_90, "eventsonly"))
display(summarize_by_event_subtype(df_60_90, "eventsonly"))
display(summarize_events_breakdown(df_60_90)["attributes"])
display(summarize_events_breakdown(df_60_90)["roles"])

In [ ]:
split = load_split()

result_90_end = run_experiment_openai_batch(
    task_name="eventsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gpt54nano__oneshot__test",
    start_idx=90,
    wait=True,
    poll_seconds=60,
)

df_90_end = result_90_end["df"]

display(summarize_main_metrics(df_90_end, "eventsonly"))
display(summarize_by_event_type(df_90_end, "eventsonly"))
display(summarize_by_event_subtype(df_90_end, "eventsonly"))
display(summarize_events_breakdown(df_90_end)["attributes"])
display(summarize_events_breakdown(df_90_end)["roles"])

#### One-shot II

In [ ]:
split = load_split()

result_0_30 = run_experiment_openai_batch(
    task_name="eventsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gpt54nano__oneshot__test_2",
    start_idx=0,
    end_idx=30,
    wait=True,
    poll_seconds=60,
)

df_0_30 = result_0_30["df"]

display(summarize_main_metrics(df_0_30, "eventsonly"))
display(summarize_by_event_type(df_0_30, "eventsonly"))
display(summarize_by_event_subtype(df_0_30, "eventsonly"))
display(summarize_events_breakdown(df_0_30)["attributes"])
display(summarize_events_breakdown(df_0_30)["roles"])

In [ ]:
split = load_split()

result_30_60 = run_experiment_openai_batch(
    task_name="eventsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gpt54nano__oneshot__test_2",
    start_idx=30,
    end_idx=60,
    wait=True,
    poll_seconds=60,
)

df_30_60 = result_30_60["df"]

display(summarize_main_metrics(df_30_60, "eventsonly"))
display(summarize_by_event_type(df_30_60, "eventsonly"))
display(summarize_by_event_subtype(df_30_60, "eventsonly"))
display(summarize_events_breakdown(df_30_60)["attributes"])
display(summarize_events_breakdown(df_30_60)["roles"])

In [ ]:
split = load_split()

result_60_90 = run_experiment_openai_batch(
    task_name="eventsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gpt54nano__oneshot__test_2",
    start_idx=60,
    end_idx=90,
    wait=True,
    poll_seconds=60,
)

df_60_90 = result_60_90["df"]

display(summarize_main_metrics(df_60_90, "eventsonly"))
display(summarize_by_event_type(df_60_90, "eventsonly"))
display(summarize_by_event_subtype(df_60_90, "eventsonly"))
display(summarize_events_breakdown(df_60_90)["attributes"])
display(summarize_events_breakdown(df_60_90)["roles"])

In [ ]:
split = load_split()

result_90_end = run_experiment_openai_batch(
    task_name="eventsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gpt54nano__oneshot__test_2",
    start_idx=90,
    wait=True,
    poll_seconds=60,
)

df_90_end = result_90_end["df"]

display(summarize_main_metrics(df_90_end, "eventsonly"))
display(summarize_by_event_type(df_90_end, "eventsonly"))
display(summarize_by_event_subtype(df_90_end, "eventsonly"))
display(summarize_events_breakdown(df_90_end)["attributes"])
display(summarize_events_breakdown(df_90_end)["roles"])

#### One-shot III

In [ ]:
split = load_split()

result_0_30 = run_experiment_openai_batch(
    task_name="eventsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gpt54nano__oneshot__test_3",
    start_idx=0,
    end_idx=30,
    wait=True,
    poll_seconds=60,
)

df_0_30 = result_0_30["df"]

display(summarize_main_metrics(df_0_30, "eventsonly"))
display(summarize_by_event_type(df_0_30, "eventsonly"))
display(summarize_by_event_subtype(df_0_30, "eventsonly"))
display(summarize_events_breakdown(df_0_30)["attributes"])
display(summarize_events_breakdown(df_0_30)["roles"])

In [ ]:
split = load_split()

result_30_60 = run_experiment_openai_batch(
    task_name="eventsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gpt54nano__oneshot__test_3",
    start_idx=30,
    end_idx=60,
    wait=True,
    poll_seconds=60,
)

df_30_60 = result_30_60["df"]

display(summarize_main_metrics(df_30_60, "eventsonly"))
display(summarize_by_event_type(df_30_60, "eventsonly"))
display(summarize_by_event_subtype(df_30_60, "eventsonly"))
display(summarize_events_breakdown(df_30_60)["attributes"])
display(summarize_events_breakdown(df_30_60)["roles"])

In [ ]:
split = load_split()

result_60_90 = run_experiment_openai_batch(
    task_name="eventsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gpt54nano__oneshot__test_3",
    start_idx=60,
    end_idx=90,
    wait=True,
    poll_seconds=60,
)

df_60_90 = result_60_90["df"]

display(summarize_main_metrics(df_60_90, "eventsonly"))
display(summarize_by_event_type(df_60_90, "eventsonly"))
display(summarize_by_event_subtype(df_60_90, "eventsonly"))
display(summarize_events_breakdown(df_60_90)["attributes"])
display(summarize_events_breakdown(df_60_90)["roles"])

In [ ]:
split = load_split()

result_90_end = run_experiment_openai_batch(
    task_name="eventsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="oneshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gpt54nano__oneshot__test_3",
    start_idx=90,
    wait=True,
    poll_seconds=60,
)

df_90_end = result_90_end["df"]

display(summarize_main_metrics(df_90_end, "eventsonly"))
display(summarize_by_event_type(df_90_end, "eventsonly"))
display(summarize_by_event_subtype(df_90_end, "eventsonly"))
display(summarize_events_breakdown(df_90_end)["attributes"])
display(summarize_events_breakdown(df_90_end)["roles"])

#### Few-shot

In [ ]:
split = load_split()

result_0_30 = run_experiment_openai_batch(
    task_name="eventsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gpt54nano__fewshot__test",
    start_idx=0,
    end_idx=30,
    wait=True,
    poll_seconds=60,
)

df_0_30 = result_0_30["df"]

display(summarize_main_metrics(df_0_30, "eventsonly"))
display(summarize_by_event_type(df_0_30, "eventsonly"))
display(summarize_by_event_subtype(df_0_30, "eventsonly"))
display(summarize_events_breakdown(df_0_30)["attributes"])
display(summarize_events_breakdown(df_0_30)["roles"])

In [ ]:
split = load_split()

result_30_60 = run_experiment_openai_batch(
    task_name="eventsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gpt54nano__fewshot__test",
    start_idx=30,
    end_idx=60,
    wait=True,
    poll_seconds=60,
)

df_30_60 = result_30_60["df"]

display(summarize_main_metrics(df_30_60, "eventsonly"))
display(summarize_by_event_type(df_30_60, "eventsonly"))
display(summarize_by_event_subtype(df_30_60, "eventsonly"))
display(summarize_events_breakdown(df_30_60)["attributes"])
display(summarize_events_breakdown(df_30_60)["roles"])

In [ ]:
split = load_split()

result_60_90 = run_experiment_openai_batch(
    task_name="eventsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gpt54nano__fewshot__test",
    start_idx=60,
    end_idx=90,
    wait=True,
    poll_seconds=60,
)

df_60_90 = result_60_90["df"]

display(summarize_main_metrics(df_60_90, "eventsonly"))
display(summarize_by_event_type(df_60_90, "eventsonly"))
display(summarize_by_event_subtype(df_60_90, "eventsonly"))
display(summarize_events_breakdown(df_60_90)["attributes"])
display(summarize_events_breakdown(df_60_90)["roles"])

In [ ]:
split = load_split()

result_90_end = run_experiment_openai_batch(
    task_name="eventsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gpt54nano__fewshot__test",
    start_idx=90,
    wait=True,
    poll_seconds=60,
)

df_90_end = result_90_end["df"]

display(summarize_main_metrics(df_90_end, "eventsonly"))
display(summarize_by_event_type(df_90_end, "eventsonly"))
display(summarize_by_event_subtype(df_90_end, "eventsonly"))
display(summarize_events_breakdown(df_90_end)["attributes"])
display(summarize_events_breakdown(df_90_end)["roles"])

#### Few-shot II

In [ ]:
split = load_split()

result_0_30 = run_experiment_openai_batch(
    task_name="eventsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gpt54nano__fewshot__test_2",
    start_idx=0,
    end_idx=30,
    wait=True,
    poll_seconds=60,
)

df_0_30 = result_0_30["df"]

display(summarize_main_metrics(df_0_30, "eventsonly"))
display(summarize_by_event_type(df_0_30, "eventsonly"))
display(summarize_by_event_subtype(df_0_30, "eventsonly"))
display(summarize_events_breakdown(df_0_30)["attributes"])
display(summarize_events_breakdown(df_0_30)["roles"])

In [ ]:
split = load_split()

result_30_60 = run_experiment_openai_batch(
    task_name="eventsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gpt54nano__fewshot__test_2",
    start_idx=30,
    end_idx=60,
    wait=True,
    poll_seconds=60,
)

df_30_60 = result_30_60["df"]

display(summarize_main_metrics(df_30_60, "eventsonly"))
display(summarize_by_event_type(df_30_60, "eventsonly"))
display(summarize_by_event_subtype(df_30_60, "eventsonly"))
display(summarize_events_breakdown(df_30_60)["attributes"])
display(summarize_events_breakdown(df_30_60)["roles"])

In [ ]:
split = load_split()

result_60_90 = run_experiment_openai_batch(
    task_name="eventsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gpt54nano__fewshot__test_2",
    start_idx=60,
    end_idx=90,
    wait=True,
    poll_seconds=60,
)

df_60_90 = result_60_90["df"]

display(summarize_main_metrics(df_60_90, "eventsonly"))
display(summarize_by_event_type(df_60_90, "eventsonly"))
display(summarize_by_event_subtype(df_60_90, "eventsonly"))
display(summarize_events_breakdown(df_60_90)["attributes"])
display(summarize_events_breakdown(df_60_90)["roles"])

In [ ]:
split = load_split()

result_90_end = run_experiment_openai_batch(
    task_name="eventsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gpt54nano__fewshot__test_2",
    start_idx=90,
    wait=True,
    poll_seconds=60,
)

df_90_end = result_90_end["df"]

display(summarize_main_metrics(df_90_end, "eventsonly"))
display(summarize_by_event_type(df_90_end, "eventsonly"))
display(summarize_by_event_subtype(df_90_end, "eventsonly"))
display(summarize_events_breakdown(df_90_end)["attributes"])
display(summarize_events_breakdown(df_90_end)["roles"])

#### Few-shot III

In [ ]:
split = load_split()

result_0_30 = run_experiment_openai_batch(
    task_name="eventsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gpt54nano__fewshot__test_3",
    start_idx=0,
    end_idx=30,
    wait=True,
    poll_seconds=60,
)

df_0_30 = result_0_30["df"]

display(summarize_main_metrics(df_0_30, "eventsonly"))
display(summarize_by_event_type(df_0_30, "eventsonly"))
display(summarize_by_event_subtype(df_0_30, "eventsonly"))
display(summarize_events_breakdown(df_0_30)["attributes"])
display(summarize_events_breakdown(df_0_30)["roles"])

In [ ]:
split = load_split()

result_30_60 = run_experiment_openai_batch(
    task_name="eventsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gpt54nano__fewshot__test_3",
    start_idx=30,
    end_idx=60,
    wait=True,
    poll_seconds=60,
)

df_30_60 = result_30_60["df"]

display(summarize_main_metrics(df_30_60, "eventsonly"))
display(summarize_by_event_type(df_30_60, "eventsonly"))
display(summarize_by_event_subtype(df_30_60, "eventsonly"))
display(summarize_events_breakdown(df_30_60)["attributes"])
display(summarize_events_breakdown(df_30_60)["roles"])

In [ ]:
split = load_split()

result_60_90 = run_experiment_openai_batch(
    task_name="eventsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gpt54nano__fewshot__test_3",
    start_idx=60,
    end_idx=90,
    wait=True,
    poll_seconds=60,
)

df_60_90 = result_60_90["df"]

display(summarize_main_metrics(df_60_90, "eventsonly"))
display(summarize_by_event_type(df_60_90, "eventsonly"))
display(summarize_by_event_subtype(df_60_90, "eventsonly"))
display(summarize_events_breakdown(df_60_90)["attributes"])
display(summarize_events_breakdown(df_60_90)["roles"])

In [ ]:
split = load_split()

result_90_end = run_experiment_openai_batch(
    task_name="eventsonly",
    model_name="gpt-5.4-nano",
    doc_ids=split["test"],
    prompt_variant="fewshot",
    max_output_tokens=4096,
    thinking_level="minimal",
    save_prefix="eventsonly__gpt54nano__fewshot__test_3",
    start_idx=90,
    wait=True,
    poll_seconds=60,
)

df_90_end = result_90_end["df"]

display(summarize_main_metrics(df_90_end, "eventsonly"))
display(summarize_by_event_type(df_90_end, "eventsonly"))
display(summarize_by_event_subtype(df_90_end, "eventsonly"))
display(summarize_events_breakdown(df_90_end)["attributes"])
display(summarize_events_breakdown(df_90_end)["roles"])

# Summary of test execution

This section aggregates and exports the final test results.

It reloads saved outputs from completed test runs, combines partial batches when
needed, computes the final task-specific metrics, and writes summary tables for
paper reporting.

The exported summaries include main precision/recall/F1 scores and, depending
on the task, breakdowns by entity type, fact predicate, event type, event
subtype, event attribute, and event role.

Configure the following variables to check the results for IE:

*   **task_name** : *[entityonly, factsonly, eventsonly]*
*   **backend_name** : *[gemini_native, llama17, mistrallarge, gpt54nano]*
*   **prompt_variant** : *[baseline, oneshot, fewshot]*

If the task is *entityonly*:
*   **entity_extraction_mode** : *[annotated, all]*



In [ ]:
task_name = "eventsonly"
backend_name = "gemini_native"
prompt_variant = "fewshot"

entity_extraction_mode = "annotated"

rerun = True
rerun_number = 3

In [ ]:
prefix = f"{task_name}__{backend_name}__{prompt_variant}__test"
if task_name == "entityonly":
    prefix = f"{task_name}__{backend_name}__{prompt_variant}__{entity_extraction_mode}__test"
if rerun:
    prefix += f"_{rerun_number}"

summary = summarize_saved_setup(
    task_name=task_name,
    backend_name=backend_name,
    prompt_variant=prompt_variant,
    save_prefix=prefix,
)

if task_name == "entityonly":
    export_entityonly_summary(summary, prefix)
elif task_name == "factsonly":
    export_factsonly_summary(summary, prefix)
elif task_name == "eventsonly":
    export_eventsonly_summary(summary, prefix)